# adventproclaimer_multilingual_translator

Kaggle copy of the Colab pipeline (generated from the git repo `book-translation-pipeline` by
`tools/build_notebook.py`; edit the repo, not this notebook). It runs headless: **Save & Run All**
(or `tools/kaggle_run.py` from the repo) executes the `JOB` chosen below.

* Audio is streamed from Bible.is (Kaggle cannot mount Google Drive), nothing is stored but clips.
* The workspace lives in `/kaggle/working/btp_workspace` and is saved as this notebook's output;
  the next run attaches that output and continues where the previous one stopped.
* Every stage stops cleanly ~1 h before Kaggle's 12 h limit (a run that times out saves nothing).
* Settings -> Accelerator: **GPU**, Internet: **on**. W&B key: Kaggle secret `WANDB_API_KEY`
  (Add-ons -> Secrets) or the private dataset `btp-secrets`.

In [ ]:
# ---- run parameters (tools/kaggle_run.py rewrites these lines) ----
JOB = 'smoke'        # smoke | full | b | book
LANGS = 'a'          # 'a' = the 17 target languages, 'b:text', or 'swh,kik'
SMOKE_LANG = 'saq'
ALIGN_CAP_HOURS = 8  # aligned hours per language (keeps the saved workspace under Kaggle's 20 GB)
DEADLINE_HOURS = 11.0  # stages stop cleanly before this; Kaggle kills runs at 12 h and then saves nothing
WEEKLY_GPU_HOURS = 30.0  # Kaggle's weekly GPU quota: runs shrink their deadline / skip to stay inside it
QUOTA_MARGIN_HOURS = 1.5  # head-room for GPU use this notebook cannot see (other notebooks, smoke tests)
STAGE_HOURS = {'mt': 5.0, 'asr': 2.5}  # per-run time slice, so translation, ASR and voices all advance each run
OUTPUT_LIMIT_GB = 19.3  # Kaggle saves NOTHING if the output exceeds 20 GB: trim to stay below this
B_LANG = 'guz'
BOOK = ''            # path of an English book under /kaggle/input for JOB='book'
BOOK_LANG = 'swh'

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
START = time.time()
WS = '/kaggle/working/btp_workspace'
os.environ.update({
    'BTP_WORKSPACE': WS, 'BTP_AUDIO_SOURCE': 'bibleis', 'BTP_RAW_AUDIO': '/tmp/no_drive',
    'BTP_LOCAL_CACHE': '/tmp/btp_cache', 'BTP_DEADLINE': str(START + float(DEADLINE_HOURS) * 3600),
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'WANDB_PROJECT': 'kenyan-speech-mt',
    'BTP_KEEP_PREV': '0',      # no <checkpoint>.prev copies: tmp+rename already prevents torn files (20 GB limit)
})
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- enable a GPU accelerator')

# W&B key: Kaggle secret first, then the private dataset btp-secrets
key = None
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception:
    pass
for f in glob.glob('/kaggle/input/**/.env', recursive=True) + glob.glob('/kaggle/input/**/btp.env', recursive=True):
    for line in open(f):
        if not key and line.startswith('WANDB_API_KEY='):
            key = line.split('=', 1)[1].strip()
if key:
    os.environ['WANDB_API_KEY'] = key
print('W&B:', 'online' if key else 'offline (no key found)')

# ---- Self-sufficient start (safe for runs started by Kaggle's own scheduler) ----
SKIP = None
if not torch.cuda.is_available():
    SKIP = 'no GPU assigned (weekly quota used up?) -- nothing run, no GPU hours used'


def ws_progress(ws):
    # (#finished trainings, #done units, sum of checkpoint steps) of a saved workspace
    trained = done = steps = 0
    for f in glob.glob(ws + '/status/*.json'):
        try:
            d = json.load(open(f))
        except Exception:
            continue
        if not isinstance(d, dict):          # e.g. kaggle_ledger.json is a list, not a status record
            continue
        if os.path.basename(f).startswith('train_'):
            trained += bool(d.get('finished'))
        elif isinstance(d.get('done'), list):
            done += len(d['done'])
    for f in glob.glob(ws + '/models/*/*/checkpoint.pt') + glob.glob(ws + '/models/mt/checkpoint/trainer_state.pt'):
        try:
            steps += int(torch.load(f, map_location='cpu', weights_only=False)['step'])
        except Exception:
            pass
    return (trained, done, steps)


# Candidate workspaces: the attached previous output, plus (if a Kaggle token secret is available) a fresh
# download of this notebook's latest output -- so a scheduled run never restarts from stale progress.
cands = [p for p in glob.glob('/kaggle/input/*/btp_workspace') + glob.glob('/kaggle/input/*/*/btp_workspace')
         if os.path.isdir(p)]
KREF = 'lutherlunyamwi/adventproclaimer-multilingual-translator'


def summary_score(s):
    # same ordering as ws_progress, from a run's run_summary.json (no need to download its workspace)
    p = (s or {}).get('progress') or {}
    steps = sum(v for v in (p.get('checkpoint_steps') or {}).values() if isinstance(v, int))
    return (len(p.get('trained') or []), sum((p.get('status_done') or {}).values()), steps)


def pick_best(summaries):
    scored = [(summary_score(s), v) for v, s in summaries.items() if s]
    return max(scored)[1] if scored else None


def kaggle_api():
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    return api


def current_version(api):
    from kagglesdk.kernels.types.kernels_api_service import ApiGetKernelRequest
    with api.build_kaggle_client() as k:
        r = ApiGetKernelRequest()
        r.user_name, r.kernel_slug = KREF.split('/')
        return int(k.kernels.kernels_api_client.get_kernel(r).metadata.current_version_number)


tok = API = None
BEST_VERSION = None
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
except Exception:
    pass
if tok:
    os.environ['KAGGLE_API_TOKEN'] = tok
    subprocess.run('pip install -q kaggle six >/dev/null 2>&1', shell=True)
    try:
        API = kaggle_api()
    except (Exception, SystemExit) as e:
        print('Kaggle API unavailable:', e)
else:
    print('WARNING: no KAGGLE_API_TOKEN secret -- restoring only the attached input (may be an older version)')
if API is not None and not SKIP:
    # The newest version is not necessarily the most advanced: Kaggle's daily schedule started version 13
    # while version 12 was training; 13 got no GPU, skipped and saved the OLD workspace. So compare the
    # recent versions' summaries and restore the best one.
    try:
        cur = current_version(API)
    except Exception as e:
        cur = None
        print('current version unknown:', e)
    sums = {}
    for v in (range(cur - 1, max(cur - 9, 0), -1) if cur else []):
        d = f'/tmp/summary_v{v}'
        subprocess.run(f"kaggle kernels output {KREF}/{v} -p {d} --file-pattern 'run_summary.json' -q >/dev/null 2>&1",
                       shell=True)
        try:
            sums[v] = json.load(open(d + '/run_summary.json'))
        except Exception:
            sums[v] = None
        shutil.rmtree(d, ignore_errors=True)
    BEST_VERSION = pick_best(sums)
    print('recent versions (trained, done, steps):', {v: summary_score(s) for v, s in sums.items() if s},
          '-> restoring version', BEST_VERSION)
    ref = f'{KREF}/{BEST_VERSION}' if BEST_VERSION else KREF
    rc = subprocess.run(f"kaggle kernels output {ref} -p /tmp/latest_output --file-pattern 'btp_workspace/.*' -q",
                        shell=True).returncode
    if rc == 0 and os.path.isdir('/tmp/latest_output/btp_workspace'):
        cands.append('/tmp/latest_output/btp_workspace')
RESTORED = None
if cands and not os.path.exists(WS):
    best = max(cands, key=ws_progress)
    print('workspace candidates:', {c: ws_progress(c) for c in cands}, '-> restoring', best)
    shutil.copytree(best, WS)
    RESTORED = best
os.makedirs(WS + '/status', exist_ok=True)
shutil.rmtree('/tmp/latest_output', ignore_errors=True)

# Weekly GPU budget, from a ledger kept inside the workspace (rolling 7 days, conservative)
LEDGER = WS + '/status/kaggle_ledger.json'
ledger = json.load(open(LEDGER)) if os.path.exists(LEDGER) else []
used_7d = sum(e['hours'] for e in ledger if e['start'] >= START - 7 * 86400)
left = WEEKLY_GPU_HOURS - QUOTA_MARGIN_HOURS - used_7d
QUOTA = None
if API is not None:
    # Kaggle's own count (every notebook and editor session, real weekly reset) beats this notebook's ledger
    try:
        g = API.quota_view().gpu_quota
        QUOTA = {'used_h': round(g.time_used.total_seconds() / 3600, 2),
                 'allowed_h': round(g.total_time_allowed.total_seconds() / 3600, 2)}
        used_7d = QUOTA['used_h']
        left = QUOTA['allowed_h'] - 0.5 - used_7d
    except Exception as e:
        print('GPU quota unavailable -- using the ledger:', e)
if not SKIP:
    DEADLINE_HOURS = min(float(DEADLINE_HOURS), left - 0.4)   # 0.4 h: start-up + summary
    if DEADLINE_HOURS < 1.0:
        SKIP = f'weekly GPU budget: {used_7d:.1f} h used in the last 7 days, {left:.1f} h left'
    os.environ['BTP_DEADLINE'] = str(START + max(DEADLINE_HOURS, 0) * 3600)
print(f'GPU budget: {used_7d:.1f} h used (7 days) | deadline this run: {DEADLINE_HOURS:.2f} h | skip: {SKIP}')

In [ ]:
!pip install -q uroman sacrebleu jiwer wandb safetensors python-docx pypdf
!mkdir -p /tmp/btp

## Source code (generated)

In [ ]:
!mkdir -p /tmp/btp /tmp/btp/btp /tmp/btp/config /tmp/btp/third_party /tmp/btp/third_party/finetune_hf_vits

In [ ]:
%%writefile /tmp/btp/btp/__init__.py
"""btp -- book translation pipeline: Bible-seeded ASR, TTS and MT for Kenyan languages."""

__version__ = "1.0.0"

In [ ]:
%%writefile /tmp/btp/btp/align.py
"""Verse-level forced alignment of chapter MP3s with torchaudio's MMS_FA aligner.

Why it is robust to Bible.is recordings:
  * a <star> token at the start and end of every chapter transcript absorbs speech
    that is not in the text (the "Mark, chapter 1" announcement, closing jingles);
  * drama recordings have music beds -- CTC blanks absorb background audio and the
    per-verse alignment score lets us drop verses that aligned badly;
  * emissions are computed in overlapping windows, so a 20-minute chapter never needs
    O(T^2) self-attention memory on a T4.

Output: one parquet shard per (audio set, book) under
    <workspace>/aligned/<lang>/<AUDIO_SET>/<BOOK>.parquet
with one row per verse (FLAC bytes + text + timing + score). Storing shards instead
of one file per verse keeps Google Drive fast (Drive FUSE is slow with many small files).
"""
from __future__ import annotations

import logging
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

import numpy as np

from btp import audio as A
from btp.text import normalize_transcript, romanize

log = logging.getLogger(__name__)

FRAME_RATE = 50          # wav2vec2: one frame per 20 ms (320 samples at 16 kHz)
STAR = "*"


@dataclass
class VerseUnit:
    ref: str
    text: str
    verse_start: int
    verse_end: int


@dataclass
class AlignedVerse:
    ref: str
    text: str
    verse_start: int
    verse_end: int
    start: float
    end: float
    score: float             # mean over the verse's frames of log p(aligned label) - max_c log p(c);
                             # 0 = the transcript is exactly what the acoustic model hears


@lru_cache(maxsize=1)
def load_aligner(device: str = "cpu"):
    import torchaudio
    bundle = torchaudio.pipelines.MMS_FA
    model = bundle.get_model(with_star=True).to(device).eval()
    dictionary = bundle.get_dict(star=STAR)
    return model, dictionary


@lru_cache(maxsize=200_000)
def _romanize_word(word: str) -> str:
    return romanize(word).lower()


def tokenize_for_aligner(text: str, dictionary: dict[str, int]) -> list[str]:
    """Words romanised with uroman and filtered to the aligner's 28-letter alphabet."""
    words = []
    for w in normalize_transcript(text).split():
        r = "".join(c for c in _romanize_word(w) if c in dictionary and c not in ("-", STAR))
        if r:
            words.append(r)
    return words


def compute_emission(model, wav: np.ndarray, device: str = "cpu", window_sec: float = 30.0,
                     context_sec: float = 2.0):
    """Log-prob emission (T, C) for arbitrarily long audio via overlapping windows."""
    import torch
    sr = A.SR
    n = len(wav)
    total_frames = n // 320
    hop = int(window_sec * sr)
    ctx = int(context_sec * sr)
    chunks = []
    with torch.inference_mode():
        for s in range(0, n, hop):
            e = min(n, s + hop)
            ws, we = max(0, s - ctx), min(n, e + ctx)
            x = torch.from_numpy(wav[ws:we]).float().unsqueeze(0).to(device)
            em, _ = model(x)
            em = torch.log_softmax(em[0].float(), dim=-1).cpu()
            f0 = int(round((s - ws) / 320))
            nf = int(round((e - s) / 320))
            chunks.append(em[f0:f0 + nf])
    em = torch.cat(chunks, 0) if chunks else torch.zeros(0, 1)
    if em.shape[0] > total_frames:
        em = em[:total_frames]
    return em


def align_chapter(wav: np.ndarray, units: list[VerseUnit], model, dictionary: dict[str, int],
                  device: str = "cpu") -> list[AlignedVerse]:
    """Align the verses of one chapter recording. Verses with no alignable letters are skipped."""
    import torch
    import torchaudio.functional as F

    verse_words = [tokenize_for_aligner(u.text, dictionary) for u in units]
    tokens: list[int] = [dictionary[STAR]]
    owner: list[int] = [-1]                      # token index -> verse index (-1 = star)
    for vi, words in enumerate(verse_words):
        for w in words:
            for c in w:
                tokens.append(dictionary[c])
                owner.append(vi)
    tokens.append(dictionary[STAR])
    owner.append(-1)
    if len(tokens) <= 2:
        return []

    emission = compute_emission(model, wav, device=device)
    T = emission.shape[0]
    # CTC needs at least one frame per token (+ blanks between repeats).
    repeats = sum(1 for a, b in zip(tokens, tokens[1:]) if a == b)
    if T < len(tokens) + repeats:
        raise ValueError(f"audio too short for transcript ({T} frames < {len(tokens) + repeats} tokens)")

    # forced_align has a CUDA kernel; on CPU a 20k-frame chapter takes minutes.
    fa_dev = device if str(device).startswith("cuda") else "cpu"
    targets = torch.tensor([tokens], dtype=torch.int32, device=fa_dev)
    ali, scores = F.forced_align(emission.unsqueeze(0).to(fa_dev), targets, blank=0)
    ali, scores = ali.cpu(), scores.cpu()
    spans = F.merge_tokens(ali[0], scores[0])
    if len(spans) != len(tokens):
        raise RuntimeError(f"aligner returned {len(spans)} spans for {len(tokens)} tokens")

    # Frame-level confidence: how far the forced path is below the best label per frame.
    # Absolute log-probs do not separate right from wrong transcripts (the aligner can
    # always squeeze tokens somewhere); the gap to the unconstrained best path does.
    gap = (scores[0] - emission.max(dim=-1).values).numpy()

    sec_per_frame = len(wav) / A.SR / T
    by_verse: dict[int, list] = {}
    for tok_i, span in enumerate(spans):
        v = owner[tok_i]
        if v >= 0:
            by_verse.setdefault(v, []).append(span)
    out = []
    for vi, u in enumerate(units):
        sp = by_verse.get(vi)
        if not sp:
            continue
        f0, f1 = sp[0].start, sp[-1].end
        out.append(AlignedVerse(ref=u.ref, text=u.text, verse_start=u.verse_start, verse_end=u.verse_end,
                                start=f0 * sec_per_frame, end=f1 * sec_per_frame,
                                score=float(gap[f0:f1].mean()) if f1 > f0 else -99.0))
    return out


# ---------------------------------------------------------------------------
# Parquet shards
# ---------------------------------------------------------------------------
SHARD_COLUMNS = ["id", "lang", "audio_set", "style", "book", "chapter", "verse_start", "verse_end",
                 "text", "start", "end", "duration", "score", "source", "audio"]


def write_shard(rows: list[dict], path: Path) -> None:
    import pyarrow as pa
    import pyarrow.parquet as pq
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pylist([{k: r.get(k) for k in SHARD_COLUMNS} for r in rows])
    tmp = path.with_suffix(".parquet.tmp")
    pq.write_table(table, tmp, compression="zstd")
    tmp.replace(path)


def read_shards(paths: list[Path], columns: list[str] | None = None) -> list[dict]:
    import pyarrow.parquet as pq
    rows: list[dict] = []
    for p in paths:
        rows += pq.read_table(p, columns=columns).to_pylist()
    return rows


def clip_rows(verses: list[AlignedVerse], wav: np.ndarray, meta: dict, cfg_align: dict) -> tuple[list[dict], dict]:
    """Cut aligned verses into FLAC clips, applying duration/score filters."""
    rows, stats = [], {"aligned": len(verses), "kept": 0, "too_short": 0, "too_long": 0, "low_score": 0}
    for v in verses:
        clip = A.slice_seconds(wav, v.start, v.end, pad=cfg_align["pad_sec"])
        dur = A.duration(clip)
        if dur < cfg_align["min_clip_sec"]:
            stats["too_short"] += 1
            continue
        if dur > cfg_align["max_clip_sec"]:
            stats["too_long"] += 1
            continue
        if v.score < cfg_align["min_score"]:
            stats["low_score"] += 1
            continue
        stats["kept"] += 1
        rows.append({**meta, "id": f"{meta['audio_set']}.{v.ref}", "verse_start": v.verse_start,
                     "verse_end": v.verse_end, "text": v.text, "start": round(v.start, 3),
                     "end": round(v.end, 3), "duration": round(dur, 3), "score": round(v.score, 4),
                     "source": "aligned", "audio": A.encode_flac(clip)})
    return rows, stats


def aligned_hours(paths, aset) -> float:
    """Hours already aligned for one set (from its resumable status file)."""
    from btp.utils import read_json
    st = read_json(paths.status() / f"align_{aset.audio}.json", {}) or {}
    return float(sum((v or {}).get("hours", 0.0) for v in (st.get("info") or {}).values()))


def align_audio_set(lang, aset, cfg: dict, paths, max_chapters: int | None = None,
                    device: str | None = None, books: list[str] | None = None,
                    max_chapters_per_book: int | None = None, hours_budget: float | None = None) -> dict:
    """Align every chapter of one (audio set, text set) pair; resumable per book."""
    from btp.bibleis import read_text_jsonl
    from btp.corpus import chapter_sources, fetch_chapter_audio, ref_id, verses_by_chapter
    from btp.utils import Status
    from btp.utils import device as auto_device

    device = device or auto_device()
    index = (paths.audio_index(aset.audio))
    from btp.utils import read_json
    index = read_json(index, []) or []
    files = chapter_sources(cfg, paths, aset, index)
    verses = verses_by_chapter(read_text_jsonl(paths.text_set(aset.text)))
    status = Status(paths.status() / f"align_{aset.audio}.json")
    model, dictionary = load_aligner(device)
    out_dir = paths.aligned_dir(lang.code, aset.audio)

    by_book: dict[str, list[dict]] = {}
    for r in files:
        if books and r["book"] not in books:
            continue
        if (r["book"], r["chapter"]) in verses:
            by_book.setdefault(r["book"], []).append(r)
    # held-out (dev/test) books first, so a time/hours cap never starves evaluation
    from btp.corpus import split_of
    by_book = dict(sorted(by_book.items(), key=lambda kv: (split_of(kv[0], cfg) == "train",)))
    if max_chapters_per_book:
        # smoke tests: a few chapters of each requested book, written as partial shards
        by_book = {b: chs[:max_chapters_per_book] for b, chs in by_book.items()}
        partial_books = {b for b, chs in by_book.items()
                         if len(chs) < sum(1 for r in files if r["book"] == b and (b, r["chapter"]) in verses)}
    else:
        partial_books = set()

    totals = {"chapters": 0, "verses": 0, "kept": 0, "hours": 0.0, "failed": 0}
    done_chapters = 0
    from btp.utils import deadline_passed
    for book, chapters in by_book.items():
        if status.is_done(book):
            continue
        if deadline_passed(margin_sec=900):
            log.warning("%s: deadline near -- stopping; the next run resumes at %s", aset.audio, book)
            totals["deadline"] = True
            break
        if hours_budget is not None and totals["hours"] >= hours_budget and split_of(book, cfg) == "train":
            log.info("%s: hours budget (%.1f h) reached -- remaining train books skipped", aset.audio, hours_budget)
            totals["budget_reached"] = True
            break
        rows: list[dict] = []
        book_stats = {"chapters": 0, "aligned": 0, "kept": 0, "low_score": 0, "failed": 0}
        for r in chapters:
            if max_chapters is not None and done_chapters >= max_chapters:
                break
            units = [VerseUnit(ref_id(book, r["chapter"], v["verse_start"], v["verse_end"]), v["text"],
                               v["verse_start"], v["verse_end"]) for v in verses[(book, r["chapter"])]]
            try:
                wav = fetch_chapter_audio(r)
                aligned = align_chapter(wav, units, model, dictionary, device=device)
            except Exception as e:  # noqa: BLE001 -- one bad chapter must not stop the set
                log.warning("%s %s %d: alignment failed: %s", aset.audio, book, r["chapter"], e)
                book_stats["failed"] += 1
                continue
            meta = {"lang": lang.code, "audio_set": aset.audio, "style": aset.style, "book": book,
                    "chapter": r["chapter"]}
            crows, st = clip_rows(aligned, wav, meta, cfg["align"])
            rows += crows
            done_chapters += 1
            book_stats["chapters"] += 1
            book_stats["aligned"] += st["aligned"]
            book_stats["kept"] += st["kept"]
            book_stats["low_score"] += st["low_score"]
            totals["verses"] += len(units)
        if book in partial_books:
            if rows:
                write_shard(rows, out_dir / f"{book}.partial.parquet")
            totals["chapters"] += book_stats["chapters"]
            totals["kept"] += len(rows)
            totals["hours"] += sum(x["duration"] for x in rows) / 3600
            continue
        if max_chapters is not None and done_chapters >= max_chapters and book_stats["chapters"] < len(chapters):
            # partial book (test runs): write it under a distinct name, never mark the book done
            if rows:
                write_shard(rows, out_dir / f"{book}.partial.parquet")
            totals["chapters"] += book_stats["chapters"]
            totals["kept"] += len(rows)
            totals["hours"] += sum(x["duration"] for x in rows) / 3600
            break
        if rows:
            write_shard(rows, out_dir / f"{book}.parquet")
        stale = out_dir / f"{book}.partial.parquet"      # left by an earlier --max-chapters run
        if stale.is_file():
            stale.unlink()
        hours = sum(x["duration"] for x in rows) / 3600
        status.mark_done(book, **book_stats, hours=round(hours, 3))
        totals["chapters"] += book_stats["chapters"]
        totals["kept"] += len(rows)
        totals["failed"] += book_stats["failed"]
        totals["hours"] += hours
        log.info("%s %s: %d chapters, kept %d/%d verses (%.2f h), %d low-score, %d failed",
                 aset.audio, book, book_stats["chapters"], book_stats["kept"], book_stats["aligned"], hours,
                 book_stats["low_score"], book_stats["failed"])
    return totals

In [ ]:
%%writefile /tmp/btp/btp/asr.py
"""ASR: MMS-1B adapter fine-tuning, decoding, evaluation and pseudo-labelling.

Recipe (Pratap et al. 2023 / HF "fine-tune MMS adapters" guide): keep the 1B wav2vec2
body frozen and train only the per-language adapter layers + output head (~2.5M
params). That fits a free T4, trains in well under an hour per language, and the
saved adapter is a few MB instead of a 4 GB model per language.

Starting point per language: its own MMS adapter when one exists (all 17 pipeline-A
languages), otherwise the bridge language's adapter (pipeline B). The output vocabulary
is the starting adapter's vocabulary plus any characters that appear in our
transcripts but not in it (new head rows are initialised small; existing rows kept).
"""
from __future__ import annotations

import json
import logging
import math
import time
from pathlib import Path

import numpy as np

from btp import audio as A
from btp.text import normalize_transcript

log = logging.getLogger(__name__)

SPECIAL = ["<pad>", "<s>", "</s>", "<unk>", "|"]


# ---------------------------------------------------------------------------
# Model / vocabulary
# ---------------------------------------------------------------------------
def base_vocab(base_model: str, adapter: str) -> dict[str, int]:
    from transformers import Wav2Vec2CTCTokenizer
    tok = Wav2Vec2CTCTokenizer.from_pretrained(base_model, target_lang=adapter)
    return dict(tok.get_vocab())


def extend_vocab(vocab: dict[str, int], texts: list[str]) -> tuple[dict[str, int], list[str]]:
    """Append characters used in ``texts`` that ``vocab`` lacks (ids of old entries unchanged)."""
    new = dict(vocab)
    added = []
    for t in texts:
        for ch in t:
            if ch == " ":
                continue
            if ch not in new:
                new[ch] = max(new.values()) + 1
                added.append(ch)
    for sp in SPECIAL:
        if sp not in new:
            new[sp] = max(new.values()) + 1
            added.append(sp)
    return new, added


def make_tokenizer(vocab: dict[str, int], lang: str, out_dir: Path):
    from transformers import Wav2Vec2CTCTokenizer
    out_dir.mkdir(parents=True, exist_ok=True)
    vf = out_dir / "vocab.json"
    with open(vf, "w", encoding="utf-8") as f:
        json.dump({lang: vocab}, f, ensure_ascii=False)
    return Wav2Vec2CTCTokenizer(str(vf), unk_token="<unk>", pad_token="<pad>", word_delimiter_token="|",
                                target_lang=lang)


def load_for_training(base_model: str, start_adapter: str | None, vocab: dict[str, int], old_vocab: dict[str, int]):
    """Base model + starting adapter, head resized to ``vocab`` (old rows copied by character)."""
    import torch
    from transformers import Wav2Vec2ForCTC

    kwargs = {"ignore_mismatched_sizes": True}
    if start_adapter:
        kwargs["target_lang"] = start_adapter
    model = Wav2Vec2ForCTC.from_pretrained(base_model, **kwargs)
    old_head = model.lm_head
    new_head = torch.nn.Linear(old_head.in_features, len(vocab))
    torch.nn.init.normal_(new_head.weight, std=0.02)
    torch.nn.init.zeros_(new_head.bias)
    with torch.no_grad():
        for ch, nid in vocab.items():
            oid = old_vocab.get(ch)
            if oid is not None and oid < old_head.out_features:
                new_head.weight[nid] = old_head.weight[oid]
                new_head.bias[nid] = old_head.bias[oid]
    model.lm_head = new_head
    model.config.vocab_size = len(vocab)
    if not start_adapter:
        model.init_adapter_layers()
    model.freeze_base_model()
    for p in model._get_adapters().values():
        p.requires_grad = True
    return model


def adapter_state(model) -> dict:
    return {k: v.detach().cpu().contiguous() for k, v in model._get_adapters().items()}


def save_adapter(model, tokenizer, out_dir: Path, meta: dict) -> None:
    from safetensors.torch import save_file

    from btp.utils import atomic_dir
    with atomic_dir(out_dir) as tmp:
        save_file(adapter_state(model), str(tmp / "adapter.safetensors"))
        tokenizer.save_pretrained(str(tmp))
        with open(tmp / "btp_asr.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=1)


def load_finetuned(model_dir: str | Path, device: str = "cpu"):
    """Load base model + our adapter/head from ``model_dir`` -> (model, tokenizer, feature_extractor)."""
    import torch
    from safetensors.torch import load_file
    from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC

    model_dir = Path(model_dir)
    meta = json.loads((model_dir / "btp_asr.json").read_text(encoding="utf-8"))
    tok = Wav2Vec2CTCTokenizer.from_pretrained(str(model_dir), target_lang=meta["lang"])
    model = Wav2Vec2ForCTC.from_pretrained(meta["base_model"], ignore_mismatched_sizes=True,
                                           vocab_size=meta["vocab_size"],
                                           **({"target_lang": meta["start_adapter"]} if meta.get("start_adapter") else {}))
    if not meta.get("start_adapter"):
        model.init_adapter_layers()
    if model.lm_head.out_features != meta["vocab_size"]:
        # Loading the released adapter resizes the head back to ITS vocabulary; ours may be larger
        # (characters added for this language). The saved state holds the full head.
        model.lm_head = torch.nn.Linear(model.lm_head.in_features, meta["vocab_size"])
        model.config.vocab_size = meta["vocab_size"]
    state = load_file(str(model_dir / "adapter.safetensors"))
    missing = set(state) - set(dict(model.named_parameters()))
    if missing:
        raise RuntimeError(f"adapter keys not in model: {sorted(missing)[:5]}")
    with torch.no_grad():
        params = dict(model.named_parameters())
        for k, v in state.items():
            params[k].copy_(v)
    fe = Wav2Vec2FeatureExtractor.from_pretrained(meta["base_model"])
    return model.to(device).eval(), tok, fe


def load_pretrained_adapter(base_model: str, adapter: str, device: str = "cpu"):
    """Off-the-shelf MMS model for ``adapter`` -> (model, tokenizer, feature_extractor)."""
    from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC
    tok = Wav2Vec2CTCTokenizer.from_pretrained(base_model, target_lang=adapter)
    model = Wav2Vec2ForCTC.from_pretrained(base_model, target_lang=adapter, ignore_mismatched_sizes=True)
    fe = Wav2Vec2FeatureExtractor.from_pretrained(base_model)
    return model.to(device).eval(), tok, fe


# ---------------------------------------------------------------------------
# Decoding
# ---------------------------------------------------------------------------
def transcribe(model, tokenizer, fe, wavs: list[np.ndarray], batch_size: int = 8,
               device: str = "cpu") -> list[tuple[str, float]]:
    """Greedy CTC decoding -> [(text, confidence)], confidence = mean max-prob of emitting frames."""
    import torch
    out: list[tuple[str, float]] = []
    order = sorted(range(len(wavs)), key=lambda i: len(wavs[i]))
    results: dict[int, tuple[str, float]] = {}
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        batch = fe([wavs[i] for i in idx], sampling_rate=A.SR, return_tensors="pt", padding=True,
                   return_attention_mask=True)
        with torch.inference_mode(), torch.autocast(device_type="cuda", enabled=device.startswith("cuda")):
            logits = model(batch.input_values.to(device),
                           attention_mask=batch.attention_mask.to(device)).logits.float()
        probs = logits.softmax(-1)
        maxp, ids = probs.max(-1)
        for j, i in enumerate(idx):
            n = int(math.ceil(len(wavs[i]) / 320))
            seq = ids[j, :n].cpu().numpy()
            mp = maxp[j, :n].cpu().numpy()
            emit = seq != tokenizer.pad_token_id
            conf = float(mp[emit].mean()) if emit.any() else 0.0
            text = tokenizer.decode(seq.tolist())
            results[i] = (normalize_transcript(text), conf)
    for i in range(len(wavs)):
        out.append(results[i])
    return out


def error_rates(refs: list[str], hyps: list[str]) -> dict:
    import jiwer
    refs_n = [normalize_transcript(r) or "<empty>" for r in refs]
    hyps_n = [normalize_transcript(h) for h in hyps]
    return {"wer": float(jiwer.wer(refs_n, hyps_n)), "cer": float(jiwer.cer(refs_n, hyps_n)), "n": len(refs)}


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
class _ClipDataset:
    def __init__(self, rows: list[dict], tokenizer, fe, max_sec: float):
        self.rows, self.tok, self.fe, self.max_sec = rows, tokenizer, fe, max_sec

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, i: int) -> dict:
        r = self.rows[i]
        wav = A.decode_audio_bytes(r["audio"])[: int(self.max_sec * A.SR)]
        x = self.fe(wav, sampling_rate=A.SR).input_values[0]
        labels = self.tok(normalize_transcript(r["text"])).input_ids
        return {"input_values": np.asarray(x, dtype=np.float32), "labels": labels}


def _collate(batch: list[dict]) -> dict:
    import torch
    n = max(len(b["input_values"]) for b in batch)
    L = max(len(b["labels"]) for b in batch)
    x = torch.zeros(len(batch), n)
    am = torch.zeros(len(batch), n, dtype=torch.long)
    y = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        x[i, :len(b["input_values"])] = torch.from_numpy(b["input_values"])
        am[i, :len(b["input_values"])] = 1
        y[i, :len(b["labels"])] = torch.tensor(b["labels"])
    return {"input_values": x, "attention_mask": am, "labels": y}


def evaluate_rows(model, tok, fe, rows: list[dict], device: str, limit: int | None = None,
                  batch_size: int = 8) -> tuple[dict, list[tuple[str, str, str]]]:
    rows = rows[:limit] if limit else rows
    if not rows:
        return {"wer": None, "cer": None, "n": 0}, []
    wavs = [A.decode_audio_bytes(r["audio"]) for r in rows]
    hyps = transcribe(model, tok, fe, wavs, batch_size=batch_size, device=device)
    m = error_rates([r["text"] for r in rows], [h for h, _ in hyps])
    examples = [(r["id"], normalize_transcript(r["text"]), h) for r, (h, _) in zip(rows[:10], hyps[:10])]
    return m, examples


def train_asr(lang, cfg: dict, paths, tracker=None, device: str | None = None, max_steps: int | None = None,
              resume: bool = True) -> dict:
    """Fine-tune the MMS adapter for ``lang`` on its materialised clips. Resumable."""
    import torch
    from transformers import Wav2Vec2FeatureExtractor, get_linear_schedule_with_warmup

    from btp.speechdata import load_metadata, materialize, read_materialized, select_clips
    from btp.utils import atomic_torch_save, deadline_passed, safe_torch_load, seed_everything
    from btp.utils import device as auto_device

    seed_everything()
    device = device or auto_device()
    c = cfg["asr"]
    max_steps = max_steps or c["max_steps"]
    out_dir = paths.asr_dir(lang.code)
    ckpt = out_dir / "checkpoint.pt"
    from btp.utils import mark_training_done, too_late_to_start, training_done
    if resume and (prev := training_done(paths, f"asr_{lang.code}", max_steps)) is not None:
        log.info("%s ASR already trained to %d steps -- skipped", lang.code, max_steps)
        return prev
    if too_late_to_start(c.get("min_minutes_to_start", 20)):
        log.warning("%s ASR: too little time before the deadline -- deferred to the next run", lang.code)
        return {"lang": lang.code, "deferred": "deadline"}

    # data ---------------------------------------------------------------
    data_dir = paths.cache / "asr" / lang.code
    if not (data_dir / "train.parquet").is_file():
        meta = load_metadata(paths, lang.code)
        splits = select_clips(meta, cfg, task="asr", hours_cap=c["hours_per_lang"], max_sec=c["max_audio_sec"])
        if not splits["train"]:
            raise RuntimeError(f"No ASR training clips for {lang.code} -- run the align/pseudo stages first")
        for k, rows in splits.items():
            materialize(paths, rows if k == "train" else rows[: c["eval_samples"]], data_dir / f"{k}.parquet")
    train_rows = read_materialized(data_dir / "train.parquet")
    dev_rows = read_materialized(data_dir / "dev.parquet")
    test_rows = read_materialized(data_dir / "test.parquet")
    if not train_rows:
        raise RuntimeError(f"No ASR training clips for {lang.code} -- run the align/pseudo stages first")
    gold_eval = any(r["source"] in ("aligned", "human") for r in test_rows)
    log.info("%s ASR data: %d train (%.1f h), %d dev, %d test (gold reference: %s)", lang.code, len(train_rows),
             sum(r["duration"] for r in train_rows) / 3600, len(dev_rows), len(test_rows), gold_eval)

    # model ----------------------------------------------------------------
    start = lang.asr_start
    old_vocab = base_vocab(c["base_model"], start) if start else {s: i for i, s in enumerate(SPECIAL)}
    vocab, added = extend_vocab(old_vocab, [normalize_transcript(r["text"]) for r in train_rows])
    if added:
        log.info("%s: added %d characters to the %s vocabulary: %s", lang.code, len(added), start, "".join(added))
    tok = make_tokenizer(vocab, lang.code, out_dir)
    fe = Wav2Vec2FeatureExtractor.from_pretrained(c["base_model"])
    model = load_for_training(c["base_model"], start, vocab, old_vocab).to(device)
    model.gradient_checkpointing_enable()
    params = [p for p in model.parameters() if p.requires_grad]
    log.info("trainable params: %.2fM", sum(p.numel() for p in params) / 1e6)

    meta = {"lang": lang.code, "base_model": c["base_model"], "start_adapter": start, "vocab_size": len(vocab),
            "added_chars": added}

    def finish(step_done: int, best_cer, done: bool, extra: dict | None = None) -> dict:
        from btp.utils import resolve_dir
        fmodel, ftok, ffe = load_finetuned(resolve_dir(out_dir / "final"), device)
        test_m, ex = evaluate_rows(fmodel, ftok, ffe, test_rows, device)
        result = {"lang": lang.code, "steps": step_done, "best_dev_cer": best_cer, "test": test_m,
                  "gold_reference": gold_eval, "train_hours": round(sum(r["duration"] for r in train_rows) / 3600, 2),
                  **(extra or {})}
        log.info("%s ASR test: %s", lang.code, test_m)
        if done:
            # recorded with the step TARGET so a re-run skips; result["steps"] keeps the real count
            mark_training_done(paths, f"asr_{lang.code}", max(step_done, max_steps), result)
        if tracker:
            tracker.summary({"test/wer": test_m["wer"], "test/cer": test_m["cer"], "gold_reference": gold_eval})
            tracker.table("test/examples", ["id", "reference", "hypothesis"], [list(e) for e in ex])
        return result

    # Baseline: the released MMS adapter, measured before any checkpoint is restored. MMS was trained on
    # Bible recordings, so on several languages it is already as good as fine-tuning gets (Kaggle run of
    # 2026-11-19: Swahili dev CER 2.41% zero-shot vs 2.50% after 2000 steps) -- don't spend GPU hours there.
    base_m = None
    if dev_rows:
        model.eval()
        base_m, _ = evaluate_rows(model, tok, fe, dev_rows, device, limit=c["eval_samples"])
        log.info("%s dev before fine-tuning: %s", lang.code, base_m)
        if tracker:
            tracker.log({"dev/wer": base_m["wer"], "dev/cer": base_m["cer"]}, step=0)
    thr = c.get("accept_zero_shot_cer")
    if (thr is not None and start and gold_eval and base_m and base_m["cer"] is not None
            and base_m["cer"] <= float(thr)):
        log.info("%s: released MMS adapter has dev CER %.2f%% (<= %.2f%%) -- adopted as is, no fine-tuning",
                 lang.code, 100 * base_m["cer"], 100 * float(thr))
        save_adapter(model, tok, out_dir / "final", {**meta, "step": 0, "dev": base_m, "zero_shot": True})
        from btp.utils import remove_checkpoint
        remove_checkpoint(ckpt)
        return finish(0, base_m["cer"], True, {"zero_shot": True})

    opt = torch.optim.AdamW(params, lr=c["lr"], weight_decay=0.0)
    sched = get_linear_schedule_with_warmup(opt, c["warmup_steps"], max_steps)
    use_amp = c.get("fp16", True) and device.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    step = 0
    bad_evals = 0
    st, _src = safe_torch_load(ckpt) if resume else (None, None)
    if st is not None:
        if st.get("vocab") == vocab:
            pdict = dict(model.named_parameters())
            with torch.no_grad():
                for k, v in st["adapter"].items():
                    pdict[k].copy_(v)
            opt.load_state_dict(st["opt"])
            sched.load_state_dict(st["sched"])
            scaler.load_state_dict(st["scaler"])
            step = st["step"]
            bad_evals = int(st.get("bad_evals", 0))
            log.info("resumed ASR %s from step %d", lang.code, step)
        else:
            log.warning("checkpoint vocabulary differs from current data -- starting fresh")

    # best so far: the better of the released adapter and an earlier run's final/ (fine-tuning must beat it)
    from btp.utils import read_json, resolve_dir
    best = base_m["cer"] if base_m else None
    prev_final = resolve_dir(out_dir / "final")
    prev_meta = read_json(prev_final / "btp_asr.json", {}) if prev_final is not None else {}
    prev_cer = (prev_meta.get("dev") or {}).get("cer")
    if (prev_cer is not None and prev_meta.get("vocab_size") == len(vocab)
            and (prev_final / "adapter.safetensors").is_file()):
        best = prev_cer if best is None else min(best, prev_cer)
    elif base_m is not None:
        save_adapter(model, tok, out_dir / "final", {**meta, "step": 0, "dev": base_m, "zero_shot": True})
    patience = int(c.get("early_stop_evals", 0) or 0)
    min_rel = float(c.get("early_stop_min_rel_improvement", 0.0))

    ds = _ClipDataset(train_rows, tok, fe, c["max_audio_sec"])
    gen = torch.Generator().manual_seed(step)
    loader = torch.utils.data.DataLoader(ds, batch_size=c["per_device_batch"], shuffle=True, collate_fn=_collate,
                                         num_workers=2, generator=gen, drop_last=True)
    accum = c["grad_accum"]
    stopped_for_deadline = early_stopped = False
    t0 = time.time()
    model.train()
    while step < max_steps and not (stopped_for_deadline or early_stopped):
        for i, batch in enumerate(loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss / accum
            if not torch.isfinite(loss):
                log.warning("non-finite loss at step %d -- batch skipped", step)
                opt.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            if (i + 1) % accum:
                continue
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if tracker and step % 10 == 0:
                tracker.log({"train/loss": loss.item() * accum, "train/grad_norm": float(gn),
                             "train/lr": sched.get_last_lr()[0], "train/sec_per_step": (time.time() - t0) / 10}, step=step)
                t0 = time.time()
            if step % c["eval_steps"] == 0 or step == max_steps:
                model.eval()
                m, ex = evaluate_rows(model, tok, fe, dev_rows, device, limit=c["eval_samples"])
                model.train()
                log.info("%s step %d dev %s", lang.code, step, m)
                if tracker:
                    tracker.log({"dev/wer": m["wer"], "dev/cer": m["cer"]}, step=step)
                    tracker.table("dev/examples", ["id", "reference", "hypothesis"], [list(e) for e in ex], step=step)
                if m["cer"] is not None and (best is None or m["cer"] < best):
                    improved_enough = best is None or m["cer"] < best * (1 - min_rel)
                    best = m["cer"]
                    save_adapter(model, tok, out_dir / "final", {**meta, "step": step, "dev": m})
                else:
                    improved_enough = False
                bad_evals = 0 if improved_enough else bad_evals + 1
                if patience and bad_evals >= patience and step < max_steps:
                    early_stopped = True
                    log.info("%s: no real dev CER gain for %d evaluations -- stopping at step %d (best %.4f)",
                             lang.code, bad_evals, step, best if best is not None else float("nan"))
            out_of_time = deadline_passed(margin_sec=900)
            if step % c["save_steps"] == 0 or step == max_steps or out_of_time or early_stopped:
                atomic_torch_save({"adapter": adapter_state(model), "opt": opt.state_dict(),
                                   "sched": sched.state_dict(), "scaler": scaler.state_dict(), "step": step,
                                   "vocab": vocab, "bad_evals": bad_evals}, ckpt)
            if out_of_time and step < max_steps and not early_stopped:
                log.warning("%s: deadline near -- ASR checkpoint saved at step %d, resume next run", lang.code, step)
                stopped_for_deadline = True
            if step >= max_steps or stopped_for_deadline or early_stopped:
                break
    if not (out_dir / "final" / "adapter.safetensors").is_file():
        save_adapter(model, tok, out_dir / "final", {**meta, "step": step})
    done = (step >= max_steps or early_stopped) and not stopped_for_deadline
    return finish(step, best, done, {"early_stopped": True} if early_stopped else None)


# ---------------------------------------------------------------------------
# Pseudo-labelling (recordings with no matching text)
# ---------------------------------------------------------------------------
def pseudo_label_set(lang, aset, cfg: dict, paths, device: str | None = None, max_chapters: int | None = None) -> dict:
    """Segment chapters at pauses, transcribe with the best available model, keep confident clips."""
    from btp.align import write_shard
    from btp.corpus import chapter_sources, fetch_chapter_audio
    from btp.utils import Status, read_json
    from btp.utils import device as auto_device

    device = device or auto_device()
    pc = cfg["pseudo_label"]
    final = paths.asr_dir(lang.code) / "final"
    if (final / "adapter.safetensors").is_file():
        model, tok, fe = load_finetuned(final, device)
        model_name = f"finetuned:{lang.code}"
    else:
        start = lang.asr_start
        if not start:
            raise RuntimeError(f"{lang.code}: no ASR model to pseudo-label with")
        model, tok, fe = load_pretrained_adapter(cfg["asr"]["base_model"], start, device)
        model_name = f"mms-1b-all:{start}"
    index = read_json(paths.audio_index(aset.audio), []) or []
    files = chapter_sources(cfg, paths, aset, index)
    status = Status(paths.status() / f"pseudo_{aset.audio}.json")
    out_dir = paths.pseudo_dir(lang.code, aset.audio)
    hours_total = float(status.data.get("hours", 0.0))
    by_book: dict[str, list[dict]] = {}
    for r in files:
        by_book.setdefault(r["book"], []).append(r)
    n_ch = 0
    stats = {"segments": 0, "kept": 0}
    from btp.utils import deadline_passed
    for book, chapters in by_book.items():
        if status.is_done(book) or hours_total >= pc["max_hours_per_set"]:
            continue
        if deadline_passed(margin_sec=900):
            log.warning("%s: deadline near -- stopping pseudo-labelling at %s", aset.audio, book)
            break
        rows = []
        for r in chapters:
            if max_chapters is not None and n_ch >= max_chapters:
                break
            try:
                wav = fetch_chapter_audio(r)
            except Exception as e:  # noqa: BLE001
                log.warning("%s: cannot read %s: %s", aset.audio, r["file"], e)
                continue
            segs = A.split_on_silence(wav, top_db=pc["silence_top_db"], min_sec=pc["segment_min_sec"],
                                      max_sec=pc["segment_max_sec"])
            clips = [A.slice_seconds(wav, s, e) for s, e in segs]
            hyps = transcribe(model, tok, fe, clips, device=device) if clips else []
            for k, ((s, e), clip, (text, conf)) in enumerate(zip(segs, clips, hyps)):
                stats["segments"] += 1
                # audio-only languages: bridge drafts are kept regardless of confidence -- they are
                # material for human correction (btp.humanloop), and training filters by confidence
                keep_all = getattr(lang, "tier", "") == "audio_only"
                if (conf < pc["min_confidence"] and not keep_all) or len(text.split()) < 2:
                    continue
                stats["kept"] += 1
                rows.append({"id": f"{aset.audio}.{book}.{r['chapter']}.s{k:03d}", "lang": lang.code,
                             "audio_set": aset.audio, "style": aset.style, "book": book, "chapter": r["chapter"],
                             "verse_start": 0, "verse_end": 0, "text": text, "start": round(s, 3), "end": round(e, 3),
                             "duration": round(A.duration(clip), 3), "score": round(conf, 4), "source": "pseudo",
                             "audio": A.encode_flac(clip)})
            n_ch += 1
        if max_chapters is not None and n_ch >= max_chapters:
            if rows:
                write_shard(rows, out_dir / f"{book}.partial.parquet")
            break
        if rows:
            write_shard(rows, out_dir / f"{book}.parquet")
        h = sum(x["duration"] for x in rows) / 3600
        hours_total += h
        status.mark_done(book, kept=len(rows), hours=round(h, 3), model=model_name)
        status.set("hours", round(hours_total, 3))
        log.info("%s %s: kept %d pseudo-labelled clips (%.2f h, total %.2f h)", aset.audio, book, len(rows), h,
                 hours_total)
    return {**stats, "hours": round(hours_total, 3), "model": model_name}

In [ ]:
%%writefile /tmp/btp/btp/audio.py
"""Audio I/O and processing.

Deliberately avoids ``torchaudio.load`` (torchaudio >= 2.9 needs the separate
torchcodec package for it). ``soundfile`` (libsndfile >= 1.1) decodes MP3, WAV and FLAC
directly, and resampling uses ``torchaudio.functional.resample`` which is pure tensor
math and stable across versions.
"""
from __future__ import annotations

import io
from pathlib import Path

import numpy as np
import soundfile as sf

SR = 16_000  # MMS ASR, MMS forced aligner and MMS-TTS all use 16 kHz mono


def to_mono(x: np.ndarray) -> np.ndarray:
    return x.mean(axis=1) if x.ndim == 2 else x


def resample(x: np.ndarray, sr: int, target_sr: int = SR) -> np.ndarray:
    if sr == target_sr or len(x) == 0:
        return x.astype(np.float32, copy=False)
    import torch
    import torchaudio.functional as AF
    y = AF.resample(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)), sr, target_sr)
    return y.numpy().astype(np.float32, copy=False)


def load_audio(src: str | Path | bytes | io.BytesIO, target_sr: int = SR) -> np.ndarray:
    """Load any MP3/WAV/FLAC (path or bytes) as float32 mono at ``target_sr``."""
    if isinstance(src, (bytes, bytearray)):
        src = io.BytesIO(src)
    data, sr = sf.read(src, dtype="float32", always_2d=False)
    return resample(to_mono(data), sr, target_sr)


def save_audio(path: str | Path, audio: np.ndarray, sr: int = SR) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), audio, sr)


def encode_flac(audio: np.ndarray, sr: int = SR) -> bytes:
    """Lossless, ~50% the size of WAV -- how aligned clips are stored inside parquet shards."""
    buf = io.BytesIO()
    sf.write(buf, np.clip(audio, -1.0, 1.0), sr, format="FLAC", subtype="PCM_16")
    return buf.getvalue()


def decode_audio_bytes(b: bytes, target_sr: int = SR) -> np.ndarray:
    return load_audio(b, target_sr)


def slice_seconds(audio: np.ndarray, start: float, end: float, sr: int = SR, pad: float = 0.0) -> np.ndarray:
    s = max(0, int(round((start - pad) * sr)))
    e = min(len(audio), int(round((end + pad) * sr)))
    return audio[s:e]


def trim_silence(audio: np.ndarray, top_db: float = 35.0) -> np.ndarray:
    import librosa
    if len(audio) == 0:
        return audio
    trimmed, _ = librosa.effects.trim(audio, top_db=top_db)
    return trimmed


def split_on_silence(audio: np.ndarray, sr: int = SR, top_db: float = 35.0,
                     min_sec: float = 3.0, max_sec: float = 15.0) -> list[tuple[float, float]]:
    """Split long audio into [min_sec, max_sec] speech segments at pauses.

    Used for pseudo-labelling recordings that have no matching text: non-silent
    intervals are greedily merged up to ``max_sec``; anything longer is cut evenly.
    Returns (start_sec, end_sec) pairs.
    """
    import librosa
    if len(audio) == 0:
        return []
    intervals = librosa.effects.split(audio, top_db=top_db, frame_length=1024, hop_length=256)
    segs: list[list[float]] = []
    for s, e in intervals:
        s_sec, e_sec = s / sr, e / sr
        if segs and (e_sec - segs[-1][0]) <= max_sec:
            segs[-1][1] = e_sec
        else:
            segs.append([s_sec, e_sec])
    out: list[tuple[float, float]] = []
    for s_sec, e_sec in segs:
        dur = e_sec - s_sec
        if dur > max_sec:
            n = int(np.ceil(dur / max_sec))
            step = dur / n
            out += [(s_sec + i * step, s_sec + (i + 1) * step) for i in range(n)]
        elif dur >= min_sec:
            out.append((s_sec, e_sec))
    return out


def concatenate_with_pauses(clips: list[np.ndarray], sr: int = SR, sentence_pause_sec: float = 0.35,
                            paragraph_pause_sec: float = 0.9, paragraph_breaks: list[int] | None = None
                            ) -> np.ndarray:
    """Stitch sentence clips; longer pause after indices listed in ``paragraph_breaks``."""
    breaks = set(paragraph_breaks or [])
    short = np.zeros(int(sentence_pause_sec * sr), dtype=np.float32)
    long = np.zeros(int(paragraph_pause_sec * sr), dtype=np.float32)
    pieces: list[np.ndarray] = []
    for i, clip in enumerate(clips):
        pieces.append(np.asarray(clip, dtype=np.float32))
        if i < len(clips) - 1:
            pieces.append(long if i in breaks else short)
    return np.concatenate(pieces) if pieces else np.zeros(0, dtype=np.float32)


def normalize_loudness(audio: np.ndarray, target_dbfs: float = -20.0) -> np.ndarray:
    """RMS normalisation with a hard ceiling at full scale."""
    if len(audio) == 0:
        return audio.astype(np.float32)
    rms = float(np.sqrt(np.mean(np.square(audio)))) + 1e-9
    gain = (10 ** (target_dbfs / 20)) / rms
    return np.clip(audio * gain, -1.0, 1.0).astype(np.float32)


def duration(audio: np.ndarray, sr: int = SR) -> float:
    return len(audio) / sr

In [ ]:
%%writefile /tmp/btp/btp/bibleis.py
"""Bible.is client: chapter -> audio file mapping and verse-level text, cached to Drive.

Two kinds of request, both against the public live.bible.is API:
  * /bibles/filesets/<AUDIO_SET>?type=audio|audio_drama  -> one row per chapter with
    book_id, chapter and the MP3 file name (we strip the expiring signature; the MP3s
    themselves are already on Drive).
  * /bibles/filesets/<TEXT_SET>/<BOOK>/<CHAPTER>?type=text_plain -> verse rows.

Everything is cached under <workspace>/corpus so each request is made once, and every
fetch is resumable (a killed Colab session continues where it stopped).
"""
from __future__ import annotations

import json
import logging
import os
import time
from pathlib import Path
from urllib.parse import urlparse

import requests

from btp.text import clean_bible_text
from btp.utils import read_json, retry, write_json

log = logging.getLogger(__name__)

# Canonical book order (USFM ids) -- used for sorting and testament detection.
OT_BOOKS = ["GEN", "EXO", "LEV", "NUM", "DEU", "JOS", "JDG", "RUT", "1SA", "2SA", "1KI", "2KI",
            "1CH", "2CH", "EZR", "NEH", "EST", "JOB", "PSA", "PRO", "ECC", "SNG", "ISA", "JER",
            "LAM", "EZK", "DAN", "HOS", "JOL", "AMO", "OBA", "JON", "MIC", "NAM", "HAB", "ZEP",
            "HAG", "ZEC", "MAL"]
NT_BOOKS = ["MAT", "MRK", "LUK", "JHN", "ACT", "ROM", "1CO", "2CO", "GAL", "EPH", "PHP", "COL",
            "1TH", "2TH", "1TI", "2TI", "TIT", "PHM", "HEB", "JAS", "1PE", "2PE", "1JN", "2JN",
            "3JN", "JUD", "REV"]
BOOK_ORDER = {b: i for i, b in enumerate(OT_BOOKS + NT_BOOKS)}


def testament(book: str) -> str:
    return "OT" if book in OT_BOOKS else "NT"


class BibleIs:
    def __init__(self, api: str = "https://live.bible.is/api", delay: float = 0.25, timeout: float = 60):
        self.api = api.rstrip("/")
        self.delay = delay
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "Mozilla/5.0 (btp-corpus-builder)",
                                     "Accept": "application/json"})
        self._last = 0.0

    def _get(self, path: str, params: dict | None = None) -> dict:
        def call():
            wait = self.delay - (time.time() - self._last)
            if wait > 0:
                time.sleep(wait)
            self._last = time.time()
            r = self.session.get(f"{self.api}/{path}", params=params, timeout=self.timeout)
            if r.status_code == 404:
                return {"data": []}
            r.raise_for_status()
            return r.json()
        return retry(call, f"GET {path}")

    # ------------------------------------------------------------------ audio
    def audio_index(self, audio_set: str, style: str, keep_url: bool = False) -> list[dict]:
        """[{book, chapter, file, duration, size}] for every chapter of an audio set.

        keep_url=True also returns the signed download ``url`` (valid ~1 day -- never cache it).
        """
        kind = "audio_drama" if style == "drama" else "audio"
        rows = self._get(f"bibles/filesets/{audio_set}", {"type": kind}).get("data", [])
        if not rows and kind == "audio_drama":   # a few sets are filed under the other type
            rows = self._get(f"bibles/filesets/{audio_set}", {"type": "audio"}).get("data", [])
        out, seen = [], set()
        for r in rows:
            if not r.get("path"):
                continue
            fname = os.path.basename(urlparse(r["path"]).path)
            key = (r["book_id"], int(r["chapter_start"]))
            if key in seen:
                continue
            seen.add(key)
            row = {"book": r["book_id"], "chapter": int(r["chapter_start"]), "file": fname,
                   "duration": r.get("duration"), "size": r.get("filesize_in_bytes")}
            if keep_url:
                row["url"] = r["path"]
            out.append(row)
        out.sort(key=lambda x: (BOOK_ORDER.get(x["book"], 999), x["chapter"]))
        return out

    # ------------------------------------------------------------------- text
    def chapter_text(self, text_set: str, book: str, chapter: int) -> list[dict]:
        rows = self._get(f"bibles/filesets/{text_set}/{book}/{chapter}", {"type": "text_plain"}).get("data", [])
        verses = []
        for r in rows:
            text = clean_bible_text(r.get("verse_text") or "")
            if not text:
                continue
            vs = int(r["verse_start"])
            ve = int(r.get("verse_end") or vs)
            verses.append({"book": book, "chapter": chapter, "verse_start": vs,
                           "verse_end": max(vs, ve), "text": text})
        verses.sort(key=lambda v: v["verse_start"])
        return verses


# ---------------------------------------------------------------------- caching
def load_audio_index(path: Path) -> list[dict] | None:
    return read_json(path)


def build_audio_index(client: BibleIs, audio_set: str, style: str, path: Path, force: bool = False) -> list[dict]:
    if path.is_file() and not force:
        return read_json(path)
    idx = client.audio_index(audio_set, style)
    if idx:
        write_json(path, idx)
    return idx


def read_text_jsonl(path: Path) -> list[dict]:
    if not path.is_file():
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def fetch_text_set(client: BibleIs, text_set: str, chapters: list[tuple[str, int]], path: Path) -> dict:
    """Append verses of every (book, chapter) not yet cached in ``path`` (JSONL).

    Returns {"chapters": n_cached, "fetched": n_new, "empty": [...]}; chapters that the
    API returns empty are recorded in a sidecar so they are not re-requested forever.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    empty_path = path.with_suffix(".empty.json")
    empty = set(tuple(x) for x in (read_json(empty_path, []) or []))
    have = {(v["book"], v["chapter"]) for v in read_text_jsonl(path)}
    todo = [c for c in chapters if c not in have and c not in empty]
    fetched = 0
    with open(path, "a", encoding="utf-8") as f:
        for i, (book, ch) in enumerate(todo, 1):
            verses = client.chapter_text(text_set, book, ch)
            if not verses:
                empty.add((book, ch))
                continue
            for v in verses:
                f.write(json.dumps(v, ensure_ascii=False) + "\n")
            f.flush()
            fetched += 1
            if i % 100 == 0:
                log.info("  %s: %d/%d chapters fetched", text_set, i, len(todo))
                write_json(empty_path, sorted(empty))
    write_json(empty_path, sorted(empty))
    return {"chapters": len(have) + fetched, "fetched": fetched, "empty": sorted(empty)}

In [ ]:
%%writefile /tmp/btp/btp/book.py
"""Book -> translated text -> audio.

Two steps with a human in between, on purpose:
  1. translate_book: English book -> review TSV (machine translation + empty
     ``reviewed_translation`` column for a fluent speaker to correct).
  2. synthesize_book: reviewed TSV -> one WAV per chapter-sized part, with sentence /
     paragraph pauses and loudness normalisation, plus an automatic intelligibility check
     (the language's ASR model transcribes the synthesised audio; CER vs the text shows
     words the voice mispronounces).
"""
from __future__ import annotations

import csv
import json
import logging
from pathlib import Path

from btp import audio as A
from btp.text import chunk_book, load_book_text, normalize_for_tts

log = logging.getLogger(__name__)

REVIEW_FIELDS = ["unit", "paragraph_id", "sentence_id", "is_paragraph_end", "source_en", "scripture_refs",
                 "scripture_target", "scripture_substituted", "glossary_flags", "machine_translation",
                 "reviewed_translation"]


def translate_book(book_path: str | Path, lang, cfg: dict, paths, out_path: str | Path | None = None,
                   model_dir: str | Path | None = None, device: str | None = None) -> Path:
    from btp.mt import EN, load_mt, translate
    from btp.utils import device as auto_device

    device = device or auto_device()
    text = load_book_text(book_path)
    units = chunk_book(text, cfg["book"]["max_chars_per_unit"])
    model_dir = Path(model_dir) if model_dir else paths.mt_dir() / "final"
    if (model_dir / "config.json").is_file():
        model, tok, meta = load_mt(model_dir, device)
        code = meta.get("languages", {}).get(lang.code, lang.nllb_token)
    elif lang.nllb:
        log.warning("no fine-tuned MT model yet -- using base %s (zero-shot)", cfg["mt"]["base_model"])
        model, tok, _ = load_mt(cfg["mt"]["base_model"], device)
        code = lang.nllb
    else:
        raise RuntimeError(f"{lang.code} is not in NLLB-200 -- train the MT stage first")
    log.info("translating %d units of %s into %s (%s)", len(units), Path(book_path).name, lang.name, code)
    hyps = translate(model, tok, [u["text"] for u in units], EN, code, num_beams=cfg["mt"]["num_beams"],
                     device=device)

    # Scripture: quoted verses come from the language's own Bible, not from machine translation
    from btp.sop import BibleLookup, glossary_flags, scripture_for_unit
    en_bible = BibleLookup.english(paths)
    tgt_bible = BibleLookup.for_language(lang, paths) if lang.has_text else None
    scripture = [scripture_for_unit(u["text"], en_bible, tgt_bible) for u in units]
    n_sub = sum(1 for s in scripture if s["substitute"])
    hyps = [s["substitute"] or h for s, h in zip(scripture, hyps)]
    log.info("scripture: %d units with references, %d replaced by the %s Bible text%s",
             sum(1 for s in scripture if s["refs"]), n_sub, lang.name,
             "" if tgt_bible and len(tgt_bible) else " (no Bible text for this language)")
    out = Path(out_path) if out_path else paths.books() / lang.code / f"{Path(book_path).stem}.review.tsv"
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=REVIEW_FIELDS, delimiter="\t")
        w.writeheader()
        for i, (u, h) in enumerate(zip(units, hyps)):
            s = scripture[i]
            w.writerow({"unit": i, "paragraph_id": u["paragraph_id"], "sentence_id": u["sentence_id"],
                        "is_paragraph_end": u["is_paragraph_end"], "source_en": u["text"],
                        "scripture_refs": s["refs"], "scripture_target": s["target_verses"],
                        "scripture_substituted": bool(s["substitute"]),
                        "glossary_flags": "; ".join(glossary_flags(u["text"], h, lang.code)),
                        "machine_translation": h, "reviewed_translation": ""})
    log.info("review file: %s (fill 'reviewed_translation'; blank rows keep the machine translation)", out)
    return out


def read_review(path: str | Path) -> list[dict]:
    rows = []
    with open(path, encoding="utf-8", newline="") as f:
        for r in csv.DictReader(f, delimiter="\t"):
            text = (r.get("reviewed_translation") or "").strip() or (r.get("machine_translation") or "").strip()
            rows.append({"paragraph_id": int(r["paragraph_id"]),
                         "is_paragraph_end": str(r["is_paragraph_end"]).strip().lower() in ("true", "1", "yes"),
                         "text": normalize_for_tts(text), "reviewed": bool((r.get("reviewed_translation") or "").strip())})
    return rows


def synthesize_book(review_path: str | Path, lang, cfg: dict, paths, out_dir: str | Path | None = None,
                    tts_dir: str | None = None, max_minutes_per_file: float = 30.0, device: str | None = None,
                    asr_check: bool = True) -> dict:
    from btp.tts import load_tts, synthesize_with, tts_model_dir
    from btp.utils import device as auto_device
    from btp.utils import write_json

    device = device or auto_device()
    rows = [r for r in read_review(review_path) if r["text"]]
    if not rows:
        raise RuntimeError("review file has no text")
    unreviewed = sum(not r["reviewed"] for r in rows)
    if unreviewed:
        log.warning("%d of %d units have no human review -- machine translation will be spoken as-is",
                    unreviewed, len(rows))
    mdir = tts_dir or tts_model_dir(paths, lang)
    model, tok = load_tts(mdir, device)
    sr = model.config.sampling_rate
    bc = cfg["book"]
    clips = synthesize_with(model, tok, [r["text"] for r in rows], device)

    out_dir = Path(out_dir) if out_dir else paths.books() / lang.code / Path(review_path).stem.replace(".review", "")
    out_dir.mkdir(parents=True, exist_ok=True)
    parts, cur, cur_len, breaks = [], [], 0.0, []
    for r, c in zip(rows, clips):
        cur.append(c)
        if r["is_paragraph_end"]:
            breaks.append(len(cur) - 1)
        cur_len += len(c) / sr
        if cur_len >= max_minutes_per_file * 60 and r["is_paragraph_end"]:
            parts.append((cur, breaks))
            cur, cur_len, breaks = [], 0.0, []
    if cur:
        parts.append((cur, breaks))
    files = []
    for k, (pc, br) in enumerate(parts, 1):
        wav = A.concatenate_with_pauses(pc, sr, bc["sentence_pause_sec"], bc["paragraph_pause_sec"], br)
        wav = A.normalize_loudness(wav, bc["target_dbfs"])
        f = out_dir / f"{lang.code}_part{k:02d}.wav"
        A.save_audio(f, wav, sr)
        files.append(str(f))
    result = {"files": files, "units": len(rows), "unreviewed_units": unreviewed, "voice": mdir,
              "minutes": round(sum(len(c) for c in clips) / sr / 60, 2)}

    if asr_check:
        try:
            result["asr_check"] = intelligibility(lang, paths, cfg, [r["text"] for r in rows][:200], clips[:200], sr,
                                                  device)
        except Exception as e:  # noqa: BLE001 -- the check is advisory
            log.warning("intelligibility check skipped: %s", e)
    write_json(out_dir / "synthesis_report.json", result)
    log.info("book audio: %s", json.dumps({k: v for k, v in result.items() if k != "files"}))
    return result


def intelligibility(lang, paths, cfg, texts: list[str], clips, sr: int, device: str) -> dict:
    """Transcribe synthesised clips with the language's ASR; CER/WER vs the input text."""
    from btp.asr import error_rates, load_finetuned, load_pretrained_adapter, transcribe
    final = paths.asr_dir(lang.code) / "final"
    if (final / "adapter.safetensors").is_file():
        model, tok, fe = load_finetuned(final, device)
    elif lang.mms_asr:
        model, tok, fe = load_pretrained_adapter(cfg["asr"]["base_model"], lang.mms_asr, device)
    else:
        raise RuntimeError("no ASR model for this language")
    wavs = [A.resample(c, sr, A.SR) for c in clips]
    hyps = transcribe(model, tok, fe, wavs, device=device)
    return error_rates(texts, [h for h, _ in hyps])

In [ ]:
%%writefile /tmp/btp/btp/cli.py
"""Command-line entry point: python -m btp.cli <stage> [--langs ...] [options].

Stages
  check      environment / GPU / Drive / registry sanity report
  corpus     Bible.is audio indexes + verse text + parallel TSVs
  align      verse-level forced alignment -> aligned parquet shards
  pseudo     pseudo-label recordings that have no matching text (Luo, Kamba, ...)
  asr        fine-tune MMS adapters (one run per language)
  mt         fine-tune the multilingual NLLB model (one run for all languages)
  tts        fine-tune MMS-TTS voices (one run per language)
  evaluate   daily leaderboard on the held-out books (MT, ASR, TTS intelligibility)
  translate-book / synthesize-book   English book -> review TSV -> narrated audio
  queue / import-transcripts   human transcription loop for audio-only languages
  harvest    collect human-reviewed book sentences -> in-domain MT training/test data
  import-openbible   ready-made verse audio+text (Open.Bible, CC BY-SA) for luo/kik/swh/orm
  status     print a summary of everything in the workspace

Every stage logs to <workspace>/logs/<stage>-<timestamp>.log and writes a JSON result
to <workspace>/reports/<stage>_<lang>.json, so runs can be inspected after Colab exits.
"""
from __future__ import annotations

import argparse
import json
import logging
import platform
import sys
import traceback

from btp.config import Paths, load_config, load_env
from btp.languages import get_language, resolve_langs
from btp.utils import now_iso, setup_logging, write_json

log = logging.getLogger("btp.cli")


def _report(paths: Paths, name: str, data: dict) -> None:
    write_json(paths.reports() / f"{name}.json", {"finished": now_iso(), **data})


def stage_check(args, cfg, paths) -> dict:
    import torch
    info = {"audio_source": cfg["paths"].get("audio_source", "drive"), "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.cuda.is_available(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "raw_audio_exists": paths.raw_audio.is_dir(), "workspace": str(paths.ws)}
    from importlib.metadata import PackageNotFoundError, version
    for mod in ("transformers", "torchaudio", "datasets", "wandb", "uroman", "sacrebleu", "jiwer", "soundfile",
                "librosa", "pyarrow"):
        try:
            info[mod] = version(mod)
        except PackageNotFoundError:
            info[mod] = "MISSING"
    import soundfile
    info["libsndfile"] = soundfile.__libsndfile_version__
    if paths.raw_audio.is_dir():
        info["raw_language_folders"] = len([p for p in paths.raw_audio.iterdir() if p.is_dir()])
    langs = resolve_langs(args.langs)
    from btp.corpus import audio_set_dir
    missing = []
    for code in langs:
        for s in get_language(code).sets:
            if cfg["paths"].get("audio_source", "drive") == "drive" and paths.raw_audio.is_dir()                     and audio_set_dir(paths, s) is None:
                missing.append(s.audio)
    info["audio_sets_missing_on_drive"] = missing
    for k, v in info.items():
        log.info("%-28s %s", k, v)
    return info


def stage_corpus(args, cfg, paths) -> dict:
    from btp.corpus import build_language_corpus
    out = {}
    for code in resolve_langs(args.langs):
        lang = get_language(code)
        try:
            out[code] = build_language_corpus(lang, cfg, paths)
            _report(paths, f"corpus_{code}", out[code])
        except Exception as e:  # noqa: BLE001
            log.error("corpus %s failed: %s\n%s", code, e, traceback.format_exc())
            out[code] = {"error": str(e)}
    return out


def stage_align(args, cfg, paths) -> dict:
    from btp.align import align_audio_set, aligned_hours
    out = {}
    cap = cfg["align"].get("max_hours_per_lang")
    for code in resolve_langs(args.langs):
        lang = get_language(code)
        # plain narration first (single speaker: what TTS needs), drama afterwards
        sets = sorted(lang.text_sets, key=lambda s: s.style != "plain")
        for aset in sets:
            if args.sets and aset.audio not in args.sets:
                continue
            budget = None
            if cap and not args.sets:
                from btp.openbible import openbible_hours
                used = sum(aligned_hours(paths, s) for s in lang.text_sets) + openbible_hours(paths, code)
                budget = cap - used
                if budget <= 0:
                    log.info("%s: %.1f h aligned >= cap %.1f h -- %s skipped", code, used, cap, aset.audio)
                    continue
            try:
                out[aset.audio] = align_audio_set(lang, aset, cfg, paths, max_chapters=args.max_chapters,
                                                  books=args.books,
                                                  max_chapters_per_book=args.max_chapters_per_book,
                                                  hours_budget=budget)
                log.info("%s %s: %s", code, aset.audio, out[aset.audio])
            except Exception as e:  # noqa: BLE001
                log.error("align %s failed: %s\n%s", aset.audio, e, traceback.format_exc())
                out[aset.audio] = {"error": str(e)}
        _report(paths, f"align_{code}", {k: v for k, v in out.items() if k in {s.audio for s in lang.sets}})
    return out


def stage_pseudo(args, cfg, paths) -> dict:
    from btp.asr import pseudo_label_set
    out = {}
    for code in resolve_langs(args.langs):
        lang = get_language(code)
        targets = lang.untexted_sets if not args.sets else [s for s in lang.sets if s.audio in args.sets]
        if lang.has_text and not args.include_texted:
            # with real text available, pseudo labels are only an optional ASR top-up
            targets = [s for s in targets if args.sets and s.audio in args.sets]
        for aset in targets:
            try:
                out[aset.audio] = pseudo_label_set(lang, aset, cfg, paths, max_chapters=args.max_chapters)
            except Exception as e:  # noqa: BLE001
                log.error("pseudo %s failed: %s\n%s", aset.audio, e, traceback.format_exc())
                out[aset.audio] = {"error": str(e)}
        _report(paths, f"pseudo_{code}", out)
    return out


def _tracker(cfg, paths, stage, name, lang=None, config=None):
    from btp.tracking import Tracker
    return Tracker(cfg, stage=stage, name=name, lang=lang, config=config, workspace=paths.ws / "wandb")


def stage_asr(args, cfg, paths) -> dict:
    from btp.asr import train_asr
    out = {}
    for code in resolve_langs(args.langs):
        lang = get_language(code)
        tr = _tracker(cfg, paths, "asr", f"asr-{code}", code, cfg["asr"])
        try:
            out[code] = train_asr(lang, cfg, paths, tracker=tr, max_steps=args.max_steps)
        except Exception as e:  # noqa: BLE001
            log.error("asr %s failed: %s\n%s", code, e, traceback.format_exc())
            out[code] = {"error": str(e)}
        finally:
            tr.finish()
        _report(paths, f"asr_{code}", out[code])
    return out


def stage_mt(args, cfg, paths) -> dict:
    from btp.mt import train_mt
    langs = [get_language(c) for c in resolve_langs(args.langs)]
    tr = _tracker(cfg, paths, "mt", "mt-multilingual", None, {**cfg["mt"], "langs": [lg.code for lg in langs]})
    try:
        out = train_mt(langs, cfg, paths, tracker=tr, max_steps=args.max_steps, base_model=args.base_model)
    finally:
        tr.finish()
    _report(paths, "mt", out)
    return out


def stage_tts(args, cfg, paths) -> dict:
    from btp.tts import train_tts
    out = {}
    for code in resolve_langs(args.langs):
        lang = get_language(code)
        tr = _tracker(cfg, paths, "tts", f"tts-{code}", code, cfg["tts"])
        try:
            out[code] = train_tts(lang, cfg, paths, tracker=tr, max_steps=args.max_steps)
        except Exception as e:  # noqa: BLE001
            log.error("tts %s failed: %s\n%s", code, e, traceback.format_exc())
            out[code] = {"error": str(e)}
        finally:
            tr.finish()
        _report(paths, f"tts_{code}", out[code])
    return out


def stage_translate_book(args, cfg, paths) -> dict:
    from btp.book import translate_book
    if not args.book:
        raise SystemExit("--book PATH is required")
    out = {}
    for code in resolve_langs(args.langs):
        try:
            out[code] = {"review_file": str(translate_book(args.book, get_language(code), cfg, paths))}
        except Exception as e:  # noqa: BLE001
            log.error("translate-book %s failed: %s\n%s", code, e, traceback.format_exc())
            out[code] = {"error": str(e)}
    return out


def stage_synthesize_book(args, cfg, paths) -> dict:
    from btp.book import synthesize_book
    if not args.review:
        raise SystemExit("--review PATH (a .review.tsv) is required")
    out = {}
    for code in resolve_langs(args.langs):
        try:
            out[code] = synthesize_book(args.review, get_language(code), cfg, paths)
        except Exception as e:  # noqa: BLE001
            log.error("synthesize-book %s failed: %s\n%s", code, e, traceback.format_exc())
            out[code] = {"error": str(e)}
    return out


def stage_evaluate(args, cfg, paths) -> dict:
    from datetime import date

    from btp.evaluate import evaluate_all
    langs = [get_language(c) for c in resolve_langs(args.langs)]
    tr = _tracker(cfg, paths, "daily-eval", f"eval-{date.today().isoformat()}", None,
                  {"langs": [lg.code for lg in langs]})
    try:
        return evaluate_all(langs, cfg, paths, tracker=tr, mt_limit=args.max_steps)
    finally:
        tr.finish()


def stage_queue(args, cfg, paths) -> dict:
    from btp.humanloop import export_queue
    out = {}
    for code in resolve_langs(args.langs):
        try:
            out[code] = export_queue(get_language(code), cfg, paths, n=args.n)
        except Exception as e:  # noqa: BLE001
            log.error("queue %s failed: %s", code, e)
            out[code] = {"error": str(e)}
    return out


def stage_import_transcripts(args, cfg, paths) -> dict:
    from btp.humanloop import import_transcripts
    return {code: import_transcripts(get_language(code), cfg, paths) for code in resolve_langs(args.langs)}


def stage_import_openbible(args, cfg, paths) -> dict:
    from btp.openbible import import_openbible
    out = {}
    for code in resolve_langs(args.langs):
        lang = get_language(code)
        if not lang.openbible:
            continue
        try:
            out[code] = import_openbible(lang, cfg, paths, max_files=args.max_chapters)
        except Exception as e:  # noqa: BLE001
            log.error("import-openbible %s failed: %s\n%s", code, e, traceback.format_exc())
            out[code] = {"error": str(e)}
        _report(paths, f"openbible_{code}", out[code])
    return out


def stage_harvest(args, cfg, paths) -> dict:
    from btp.sop import harvest_reviews
    return {code: harvest_reviews(get_language(code), paths) for code in resolve_langs(args.langs)}


def stage_status(args, cfg, paths) -> dict:
    from btp.speechdata import load_metadata, summarize
    out = {}
    for code in resolve_langs(args.langs):
        try:
            rows = load_metadata(paths, code, include_partial=True)
        except Exception:  # noqa: BLE001
            rows = []
        par = paths.parallel(code)
        n_par = sum(1 for _ in open(par, encoding="utf-8")) - 1 if par.is_file() else 0
        out[code] = {"parallel_pairs": n_par, "speech": summarize(rows),
                     "asr_model": (paths.asr_dir(code) / "final").is_dir(),
                     "tts_model": (paths.tts_dir(code) / "final").is_dir()}
        log.info("%s: %s", code, json.dumps(out[code]))
    out["mt_model"] = (paths.mt_dir() / "final").is_dir()
    return out


STAGES = {"check": stage_check, "corpus": stage_corpus, "align": stage_align, "pseudo": stage_pseudo,
          "asr": stage_asr, "mt": stage_mt, "tts": stage_tts, "translate-book": stage_translate_book,
          "synthesize-book": stage_synthesize_book, "evaluate": stage_evaluate, "queue": stage_queue,
          "import-transcripts": stage_import_transcripts, "harvest": stage_harvest, "import-openbible": stage_import_openbible,
          "status": stage_status}


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="btp", description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("stage", choices=list(STAGES))
    p.add_argument("--langs", default="a", help="'a', 'b', 'b:text', 'all' or comma list (default: a)")
    p.add_argument("--sets", nargs="*", help="restrict to these audio set ids")
    p.add_argument("--max-chapters", type=int, default=None, help="limit chapters per set (smoke tests)")
    p.add_argument("--books", nargs="*", help="align: only these USFM book ids (e.g. MRK COL PHP)")
    p.add_argument("--max-chapters-per-book", type=int, default=None, help="align: chapters per book (smoke)")
    p.add_argument("--max-steps", type=int, default=None, help="override training steps (smoke tests)")
    p.add_argument("--base-model", default=None, help="override the MT base model")
    p.add_argument("--include-texted", action="store_true", help="pseudo: also label sets of languages with text")
    p.add_argument("--book", help="translate-book: English book (.txt/.md/.docx/.pdf/.epub)")
    p.add_argument("--review", help="synthesize-book: reviewed .review.tsv from translate-book")
    p.add_argument("--n", type=int, default=300, help="queue: clips per transcription batch")
    p.add_argument("--set", dest="overrides", action="append", default=[], help="config override key=value")
    return p


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    load_env()
    cfg = load_config(args.overrides)
    paths = Paths(cfg)
    log_dir = paths.logs() if paths.ws.parent.exists() else None
    log_path = setup_logging(log_dir, name=args.stage)
    log.info("btp %s | langs=%s | log=%s", args.stage, args.langs, log_path)
    try:
        result = STAGES[args.stage](args, cfg, paths)
    except Exception as e:  # noqa: BLE001
        log.error("stage %s crashed: %s\n%s", args.stage, e, traceback.format_exc())
        _report(paths, f"stage_{args.stage}", {"error": str(e), "traceback": traceback.format_exc()})
        return 1
    _report(paths, f"stage_{args.stage}", {"result": result})
    failed = [k for k, v in (result or {}).items() if isinstance(v, dict) and "error" in v]
    if failed:
        log.error("stage %s finished with failures: %s", args.stage, failed)
        return 2
    log.info("stage %s OK", args.stage)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /tmp/btp/btp/config.py
"""Configuration loading: config/pipeline.yaml + CLI overrides + environment (.env)."""
from __future__ import annotations

import copy
import os
from pathlib import Path
from typing import Any

import yaml

REPO_ROOT = Path(__file__).resolve().parent.parent
CONFIG_DIR = REPO_ROOT / "config"


def _parse_value(raw: str) -> Any:
    """Parse an override value with YAML rules (numbers, bools, lists, null)."""
    return yaml.safe_load(raw)


def apply_overrides(cfg: dict, overrides: list[str] | None) -> dict:
    """Apply ``a.b.c=value`` overrides to a nested dict (returns a new dict)."""
    cfg = copy.deepcopy(cfg)
    for item in overrides or []:
        if "=" not in item:
            raise ValueError(f"Override must look like key.sub=value, got {item!r}")
        key, raw = item.split("=", 1)
        node = cfg
        parts = key.strip().split(".")
        for p in parts[:-1]:
            node = node.setdefault(p, {})
            if not isinstance(node, dict):
                raise ValueError(f"Cannot set {key}: {p} is not a section")
        node[parts[-1]] = _parse_value(raw)
    return cfg


def load_env(path: str | Path | None = None) -> None:
    """Load KEY=VALUE lines from a .env file into os.environ (existing vars win)."""
    candidates = [Path(path)] if path else [REPO_ROOT / ".env"]
    for p in candidates:
        if not p.is_file():
            continue
        for line in p.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))


def load_config(overrides: list[str] | None = None, path: str | Path | None = None) -> dict:
    """Load pipeline.yaml, apply overrides, and honour BTP_WORKSPACE / BTP_RAW_AUDIO env vars."""
    with open(path or CONFIG_DIR / "pipeline.yaml", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    env_map = {
        "BTP_WORKSPACE": ("paths", "workspace"),
        "BTP_RAW_AUDIO": ("paths", "raw_audio_root"),
        "BTP_LOCAL_CACHE": ("paths", "local_cache"),
        "BTP_AUDIO_SOURCE": ("paths", "audio_source"),
    }
    for env, (sec, key) in env_map.items():
        if os.environ.get(env):
            cfg[sec][key] = os.environ[env]
    return apply_overrides(cfg, overrides)


class Paths:
    """All on-disk locations derived from the workspace root (kept in one place)."""

    def __init__(self, cfg: dict):
        self.raw_audio = Path(cfg["paths"]["raw_audio_root"])
        self.ws = Path(cfg["paths"]["workspace"])
        self.cache = Path(cfg["paths"]["local_cache"])

    # corpora ------------------------------------------------------------
    def text_set(self, fileset: str) -> Path:
        return self.ws / "corpus" / "text" / f"{fileset.replace(':', '_')}.jsonl"

    def audio_index(self, audio_set: str) -> Path:
        return self.ws / "corpus" / "audio_index" / f"{audio_set}.json"

    def parallel(self, lang: str) -> Path:
        return self.ws / "corpus" / "parallel" / f"{lang}.tsv"

    # aligned speech -------------------------------------------------------
    def aligned_dir(self, lang: str, audio_set: str) -> Path:
        return self.ws / "aligned" / lang / audio_set

    def pseudo_dir(self, lang: str, audio_set: str) -> Path:
        return self.ws / "pseudo" / lang / audio_set

    # models -------------------------------------------------------------
    def mt_dir(self) -> Path:
        return self.ws / "models" / "mt"

    def asr_dir(self, lang: str) -> Path:
        return self.ws / "models" / "asr" / lang

    def tts_dir(self, lang: str) -> Path:
        return self.ws / "models" / "tts" / lang

    # bookkeeping ----------------------------------------------------------
    def logs(self) -> Path:
        return self.ws / "logs"

    def reports(self) -> Path:
        return self.ws / "reports"

    def status(self) -> Path:
        return self.ws / "status"

    def books(self) -> Path:
        return self.ws / "books"

In [ ]:
%%writefile /tmp/btp/btp/corpus.py
"""Corpus building: audio-file lookup on Drive, verse tables, splits and parallel TSVs."""
from __future__ import annotations

import csv
import logging
from collections import defaultdict
from pathlib import Path

from btp.bibleis import BOOK_ORDER, BibleIs, build_audio_index, fetch_text_set, read_text_jsonl, testament
from btp.config import Paths
from btp.languages import AudioSet, Language, english_sets

log = logging.getLogger(__name__)

PARALLEL_FIELDS = ["ref", "book", "chapter", "split", "en", "tgt"]


# ---------------------------------------------------------------------------
# Splits (shared by MT, ASR, TTS)
# ---------------------------------------------------------------------------
def split_of(book: str, cfg: dict) -> str:
    s = cfg["splits"]
    if book in s["test_books"] or book in s.get("test_books_ot", []):
        return "test"
    if book in s["dev_books"] or book in s.get("dev_books_ot", []):
        return "dev"
    return "train"


# ---------------------------------------------------------------------------
# Drive audio lookup
# ---------------------------------------------------------------------------
def audio_set_dir(paths: Paths, aset: AudioSet) -> Path | None:
    """Folder '<raw>/<drive_folder>/<AUDIO_SET> (<style> <part>)' created by the Drive copy job."""
    def match(base: Path) -> Path | None:
        if not base.is_dir():
            return None
        for p in base.iterdir():
            if p.is_dir() and (p.name == aset.audio or p.name.startswith(aset.audio + " ")):
                return p
        return None

    found = match(paths.raw_audio / aset.drive_folder)
    if found is None and paths.raw_audio.is_dir():
        # registry folder wrong or renamed on Drive: the set id is unique, so search every language folder
        for lang_dir in sorted(paths.raw_audio.iterdir()):
            found = match(lang_dir)
            if found is not None:
                log.warning("%s found under %s (registry says %s)", aset.audio, lang_dir.name, aset.drive_folder)
                break
    return found


def chapter_audio_files(paths: Paths, aset: AudioSet, index: list[dict]) -> list[dict]:
    """Attach the local (Drive-mounted) MP3 path to each audio-index row that exists on disk."""
    d = audio_set_dir(paths, aset)
    if d is None:
        log.warning("Audio folder for %s not found under %s", aset.audio, paths.raw_audio / aset.drive_folder)
        return []
    present = {p.name: p for p in d.iterdir() if p.suffix.lower() == ".mp3"}
    rows = []
    for r in index:
        if r["file"] in present:
            rows.append({**r, "path": str(present[r["file"]])})
    missing = len(index) - len(rows)
    if missing:
        log.warning("%s: %d of %d indexed chapters missing on Drive", aset.audio, missing, len(index))
    return rows


def chapter_sources(cfg: dict, paths: Paths, aset: AudioSet, index: list[dict]) -> list[dict]:
    """Chapter rows carrying either a local ``path`` (Drive) or a fresh signed ``url`` (Bible.is)."""
    if cfg["paths"].get("audio_source", "drive") == "bibleis":
        client = BibleIs(cfg["bibleis"]["api"], cfg["bibleis"]["request_delay_sec"], cfg["bibleis"]["timeout_sec"])
        live = {(r["book"], r["chapter"]): r for r in client.audio_index(aset.audio, aset.style, keep_url=True)}
        return [{**r, "url": live[(r["book"], r["chapter"])]["url"]} for r in index if (r["book"], r["chapter"]) in live]
    return chapter_audio_files(paths, aset, index)


def fetch_chapter_audio(row: dict):
    """Decoded 16 kHz mono audio of one chapter row (local file or in-memory download)."""
    from btp import audio as A
    if row.get("path"):
        return A.load_audio(row["path"])
    import requests

    from btp.utils import retry

    def get():
        r = requests.get(row["url"], timeout=120, headers={"User-Agent": "Mozilla/5.0 (btp)"})
        r.raise_for_status()
        return r.content
    return A.load_audio(retry(get, f"download {row.get('file')}"))


# ---------------------------------------------------------------------------
# Verse tables
# ---------------------------------------------------------------------------
def verses_by_chapter(rows: list[dict]) -> dict[tuple[str, int], list[dict]]:
    out: dict[tuple[str, int], list[dict]] = defaultdict(list)
    seen = set()
    for v in rows:
        key = (v["book"], int(v["chapter"]), int(v["verse_start"]))
        if key in seen:              # tolerate duplicate lines from an interrupted append
            continue
        seen.add(key)
        out[(v["book"], int(v["chapter"]))].append(v)
    for k in out:
        out[k].sort(key=lambda v: v["verse_start"])
    return dict(out)


def merge_parallel(tgt: list[dict], en: list[dict]) -> list[dict]:
    """Pair target and English verses of ONE chapter, merging overlapping verse ranges.

    Translations sometimes combine verses ("3-4") differently; any target and English
    units whose [verse_start, verse_end] ranges overlap are merged into one pair so the
    two sides always cover exactly the same verses.
    """
    items = [("t", v["verse_start"], v["verse_end"], v["text"]) for v in tgt] + \
            [("e", v["verse_start"], v["verse_end"], v["text"]) for v in en]
    items.sort(key=lambda x: (x[1], x[2]))
    groups: list[dict] = []
    for side, vs, ve, text in items:
        if groups and vs <= groups[-1]["ve"]:
            g = groups[-1]
            g["ve"] = max(g["ve"], ve)
        else:
            g = {"vs": vs, "ve": ve, "t": [], "e": []}
            groups.append(g)
        g[side].append((vs, text))
    pairs = []
    for g in groups:
        if not g["t"] or not g["e"]:
            continue               # verse missing on one side (e.g. textual variants)
        t = " ".join(x[1] for x in sorted(g["t"]))
        e = " ".join(x[1] for x in sorted(g["e"]))
        pairs.append({"verse_start": g["vs"], "verse_end": g["ve"], "tgt": t, "en": e})
    return pairs


def ref_id(book: str, chapter: int, vs: int, ve: int) -> str:
    return f"{book}.{chapter}.{vs}" if vs == ve else f"{book}.{chapter}.{vs}-{ve}"


# ---------------------------------------------------------------------------
# Stage: build corpus for one language
# ---------------------------------------------------------------------------
def build_language_corpus(lang: Language, cfg: dict, paths: Paths, client: BibleIs | None = None,
                          fetch_english: bool = True) -> dict:
    """Audio indexes + verse text for every set of ``lang``, then the parallel TSV.

    Returns a summary dict (also written to <workspace>/status/corpus_<lang>.json).
    """
    client = client or BibleIs(cfg["bibleis"]["api"], cfg["bibleis"]["request_delay_sec"],
                               cfg["bibleis"]["timeout_sec"])
    summary: dict = {"lang": lang.code, "sets": {}}
    needed_text: dict[str, set[tuple[str, int]]] = defaultdict(set)
    all_chapters: set[tuple[str, int]] = set()

    for aset in lang.sets:
        idx = build_audio_index(client, aset.audio, aset.style, paths.audio_index(aset.audio))
        on_drive = chapter_audio_files(paths, aset, idx)             if cfg["paths"].get("audio_source", "drive") == "drive" and paths.raw_audio.is_dir() else []
        chapters = {(r["book"], r["chapter"]) for r in idx}
        all_chapters |= chapters
        if aset.text:
            needed_text[aset.text] |= chapters
        summary["sets"][aset.audio] = {"indexed_chapters": len(idx), "on_drive": len(on_drive),
                                       "text": aset.text, "style": aset.style}

    from btp import ebible
    for text_set in lang.extra_texts:
        needed_text.setdefault(text_set, set())
    for text_set, chapters in list(needed_text.items()):
        if ebible.is_ebible(text_set):
            # whole translation (also chapters without audio -> more translation pairs)
            res = ebible.fetch_text(text_set, paths.text_set(text_set))
            needed_text[text_set] = ebible.chapters_in(paths.text_set(text_set))
            summary.setdefault("text", {})[text_set] = {"chapters": res["chapters"], "verses": res["verses"],
                                                        "source": "eBible.org"}
            log.info("%s text %s (eBible): %d chapters, %d verses", lang.code, text_set, res["chapters"], res["verses"])
            continue
        res = fetch_text_set(client, text_set, sorted(chapters, key=lambda c: (BOOK_ORDER.get(c[0], 999), c[1])),
                             paths.text_set(text_set))
        summary.setdefault("text", {})[text_set] = {"chapters": res["chapters"], "empty": len(res["empty"])}
        log.info("%s text %s: %d chapters cached (%d empty)", lang.code, text_set, res["chapters"], len(res["empty"]))

    if fetch_english and needed_text:
        en = english_sets()
        text_chapters = set().union(*needed_text.values())
        for key, tset in (("nt", en["nt"]), ("ot", en["ot"])):
            chs = sorted((c for c in text_chapters if (testament(c[0]) == "NT") == (key == "nt")),
                         key=lambda c: (BOOK_ORDER.get(c[0], 999), c[1]))
            if chs:
                fetch_text_set(client, tset, chs, paths.text_set(tset))

    if needed_text:
        summary["parallel"] = build_parallel(lang, cfg, paths)
    return summary


def build_parallel(lang: Language, cfg: dict, paths: Paths) -> dict:
    """Write <workspace>/corpus/parallel/<lang>.tsv from all of the language's text sets."""
    en_sets = english_sets()
    en_rows = read_text_jsonl(paths.text_set(en_sets["nt"])) + read_text_jsonl(paths.text_set(en_sets["ot"]))
    en_by = verses_by_chapter(en_rows)
    seen_refs: set[str] = set()
    counts = {"train": 0, "dev": 0, "test": 0}
    out_path = paths.parallel(lang.code)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(".tsv.tmp")
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=PARALLEL_FIELDS, delimiter="\t", quoting=csv.QUOTE_MINIMAL)
        w.writeheader()
        for text_set in dict.fromkeys([s.text for s in lang.text_sets] + list(lang.extra_texts)):
            tgt_by = verses_by_chapter(read_text_jsonl(paths.text_set(text_set)))
            for (book, ch) in sorted(tgt_by, key=lambda c: (BOOK_ORDER.get(c[0], 999), c[1])):
                if (book, ch) not in en_by:
                    continue
                for p in merge_parallel(tgt_by[(book, ch)], en_by[(book, ch)]):
                    ref = ref_id(book, ch, p["verse_start"], p["verse_end"])
                    # the same text set listed for two audio sets (e.g. SOMSIM) must not duplicate pairs,
                    # but two different versions of one verse are two genuine translation pairs
                    if (text_set, ref) in seen_refs:
                        continue
                    seen_refs.add((text_set, ref))
                    sp = split_of(book, cfg)
                    counts[sp] += 1
                    w.writerow({"ref": ref, "book": book, "chapter": ch, "split": sp,
                                "en": p["en"], "tgt": p["tgt"]})
    tmp.replace(out_path)
    log.info("%s parallel corpus: %s -> %s", lang.code, counts, out_path)
    return counts


def read_parallel(path: str | Path, split: str | None = None) -> list[dict]:
    p = Path(path)
    if not p.is_file():
        return []
    with open(p, encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f, delimiter="\t"))
    return [r for r in rows if split is None or r["split"] == split]

In [ ]:
%%writefile /tmp/btp/btp/ebible.py
"""eBible.org as a second Bible text source (verse-per-line downloads).

Used where Bible.is has audio but no text (e.g. Dholuo: eBible "luo" is the same Bible as the
Bible.is LUOBIB recordings) and for extra translation pairs from openly licensed versions.

A registry text set "ebible:<translationId>" is downloaded once from
https://ebible.org/Scriptures/<id>_vpl.zip and cached in the same JSONL verse format as the
Bible.is text (book, chapter, verse_start, verse_end, text), so alignment and parallel-corpus
building treat both sources identically.

eBible uses its own codes for 11 books (MAR, JOH, PHI, ...); they are mapped to the USFM ids used
everywhere else. Without this, Mark/John/Philippians... would silently never pair with the English
verses or the audio -- and Philippians is a held-out test book. Deuterocanonical books (no English
counterpart in our corpus) are dropped.
"""
from __future__ import annotations

import io
import json
import logging
import re
import zipfile
from pathlib import Path

import requests

from btp.bibleis import BOOK_ORDER
from btp.text import clean_bible_text
from btp.utils import retry

log = logging.getLogger(__name__)

URL = "https://ebible.org/Scriptures/{tid}_vpl.zip"
EBIBLE_TO_USFM = {"MAR": "MRK", "JOH": "JHN", "PHI": "PHP", "JAM": "JAS", "1JO": "1JN", "2JO": "2JN",
                  "3JO": "3JN", "SOL": "SNG", "EZE": "EZK", "JOE": "JOL", "NAH": "NAM"}
_LINE = re.compile(r"^([0-9A-Z]{3}) (\d+):(\d+)(?:-(\d+))? ?(.*)$")
PREFIX = "ebible:"


def is_ebible(text_set: str | None) -> bool:
    return bool(text_set) and text_set.startswith(PREFIX)


def translation_id(text_set: str) -> str:
    return text_set[len(PREFIX):]


def parse_vpl(lines: list[str]) -> tuple[list[dict], dict]:
    """VPL lines -> verse rows in our JSONL schema, plus stats of what was skipped."""
    rows, stats = [], {"lines": 0, "deuterocanon_skipped": 0, "unparsed": 0, "empty": 0}
    for line in lines:
        line = line.strip()
        if not line:
            continue
        stats["lines"] += 1
        m = _LINE.match(line)
        if not m:
            stats["unparsed"] += 1
            continue
        book = EBIBLE_TO_USFM.get(m.group(1), m.group(1))
        if book not in BOOK_ORDER:
            stats["deuterocanon_skipped"] += 1
            continue
        text = clean_bible_text(m.group(5))
        if not text:
            stats["empty"] += 1
            continue
        vs = int(m.group(3))
        ve = int(m.group(4) or vs)
        rows.append({"book": book, "chapter": int(m.group(2)), "verse_start": vs, "verse_end": max(vs, ve),
                     "text": text})
    return rows, stats


def fetch_text(text_set: str, path: Path, force: bool = False) -> dict:
    """Download + cache one eBible translation as JSONL. Returns {"chapters", "verses", ...stats}."""
    if path.is_file() and not force:
        chapters = set()
        n = 0
        with open(path, encoding="utf-8") as f:
            for line in f:
                v = json.loads(line)
                chapters.add((v["book"], v["chapter"]))
                n += 1
        return {"chapters": len(chapters), "verses": n, "cached": True}
    tid = translation_id(text_set)

    def get():
        r = requests.get(URL.format(tid=tid), timeout=180, headers={"User-Agent": "Mozilla/5.0 (btp-corpus-builder)"})
        r.raise_for_status()
        return r.content
    z = zipfile.ZipFile(io.BytesIO(retry(get, f"eBible {tid}")))
    name = next(n for n in z.namelist() if n.endswith("_vpl.txt"))
    rows, stats = parse_vpl(z.read(name).decode("utf-8-sig").splitlines())
    if not rows:
        raise RuntimeError(f"eBible {tid}: no verses parsed ({stats})")
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    tmp.replace(path)
    chapters = {(r["book"], r["chapter"]) for r in rows}
    log.info("eBible %s: %d verses in %d chapters cached (%s)", tid, len(rows), len(chapters), stats)
    return {"chapters": len(chapters), "verses": len(rows), **stats}


def chapters_in(path: Path) -> set[tuple[str, int]]:
    out = set()
    if path.is_file():
        with open(path, encoding="utf-8") as f:
            for line in f:
                v = json.loads(line)
                out.add((v["book"], v["chapter"]))
    return out

In [ ]:
%%writefile /tmp/btp/btp/evaluate.py
"""Daily evaluation: score the latest models on the fixed held-out books -> leaderboard.

One W&B run per day (job_type "daily-eval") with:
  * mt/<lang>/<en2x|x2en>/chrf, bleu      latest multilingual MT model, beam search
  * asr/<lang>/wer, cer                   latest fine-tuned adapter (and MMS zero-shot baseline)
  * tts/<lang>/asr_cer                     intelligibility of the latest voice (synthesise -> ASR)
  * a leaderboard table + audio samples
and a dated JSON copy in <workspace>/reports/leaderboard/. Because the test books never
change, these numbers are comparable from day to day: the W&B line charts ARE the trend.
"""
from __future__ import annotations

import logging
from datetime import date

from btp import audio as A

log = logging.getLogger(__name__)


def _test_rows(paths, cfg, lang_code: str, limit: int) -> list[dict]:
    from btp.corpus import split_of
    from btp.speechdata import load_audio_for, load_metadata
    meta = [r for r in load_metadata(paths, lang_code) if split_of(r["book"], cfg) == "test"
            and (r["source"] == "human" or (r["source"] == "aligned" and r["score"] >= cfg["align"]["min_score"]))
            and r["duration"] <= cfg["asr"]["max_audio_sec"]]
    meta = sorted(meta, key=lambda r: r["id"])[:limit]
    audio = load_audio_for(paths, meta)
    return [{**r, "audio": audio[r["id"]]} for r in meta if r["id"] in audio]


def evaluate_all(langs, cfg: dict, paths, tracker=None, device: str | None = None, mt_limit: int | None = None,
                 asr_limit: int = 200, tts_sentences: int = 20, baselines: bool = True) -> dict:
    from btp.utils import device as auto_device
    from btp.utils import write_json

    device = device or auto_device()
    board: dict = {"date": date.today().isoformat(), "mt": {}, "asr": {}, "tts": {}}

    # ---- MT -------------------------------------------------------------
    mt_final = paths.mt_dir() / "final"
    if (mt_final / "config.json").is_file():
        from btp.mt import evaluate as mt_eval
        from btp.mt import gather_examples, load_mt
        model, tok, meta = load_mt(mt_final, device)
        groups = gather_examples(langs, cfg, paths, "test")
        from btp.langid import profiles_from_parallel
        from btp.languages import list_languages
        profiles = profiles_from_parallel(paths, list_languages())
        m, samples = mt_eval(model, tok, groups, limit=mt_limit, num_beams=cfg["mt"]["num_beams"], device=device,
                             profiles=profiles)
        board["mt"] = {"model_step": meta.get("step"), **m}
        if tracker:
            tracker.table("mt/samples", ["group", "ref", "source", "reference", "hypothesis"],
                          [row for v in samples.values() for row in v])
        del model
    else:
        log.info("no MT model yet")

    # ---- ASR + TTS ----------------------------------------------------------
    from btp.asr import error_rates, evaluate_rows, load_finetuned, load_pretrained_adapter, transcribe
    from btp.tts import load_tts, synthesize_with, tts_model_dir
    for lang in langs:
        rows = _test_rows(paths, cfg, lang.code, asr_limit)
        final = paths.asr_dir(lang.code) / "final"
        asr = None
        if rows and (final / "adapter.safetensors").is_file():
            asr = load_finetuned(final, device)
            m, ex = evaluate_rows(*asr, rows, device)
            board["asr"][lang.code] = {**m, "model": "finetuned"}
            if tracker:
                tracker.table(f"asr/{lang.code}/examples", ["id", "reference", "hypothesis"], [list(e) for e in ex])
        if rows and baselines and lang.mms_asr:
            base = load_pretrained_adapter(cfg["asr"]["base_model"], lang.mms_asr, device)
            m0, _ = evaluate_rows(*base, rows, device)
            board["asr"].setdefault(lang.code, {})["baseline_cer"] = m0["cer"]
            board["asr"][lang.code]["baseline_wer"] = m0["wer"]
            if asr is None:
                asr = base
                board["asr"][lang.code].update({**m0, "model": "mms-zero-shot"})
        # TTS intelligibility: synthesise held-out sentences, transcribe with the ASR model
        if rows and asr is not None:
            try:
                mdir = tts_model_dir(paths, lang)
                tts_model, tts_tok = load_tts(mdir, device)
                texts = [r["text"] for r in rows[:tts_sentences]]
                clips = synthesize_with(tts_model, tts_tok, texts, device)
                sr = tts_model.config.sampling_rate
                hyps = transcribe(*asr, [A.resample(c, sr, A.SR) for c in clips], device=device)
                er = error_rates(texts, [h for h, _ in hyps])
                board["tts"][lang.code] = {"asr_cer": er["cer"], "asr_wer": er["wer"], "n": er["n"],
                                           "voice": "finetuned" if "facebook/" not in mdir else mdir}
                if tracker:
                    tracker.audio(f"tts/{lang.code}/samples", clips[:3], texts[:3], sr)
            except Exception as e:  # noqa: BLE001
                log.warning("%s TTS eval skipped: %s", lang.code, e)

    # ---- log -------------------------------------------------------------------
    write_json(paths.reports() / "leaderboard" / f"{board['date']}.json", board)
    write_json(paths.reports() / "leaderboard" / "latest.json", board)
    if tracker:
        flat = {}
        for k, v in board["mt"].items():
            if isinstance(v, dict):
                for mk in ("chrf", "bleu", "off_target"):
                    if v.get(mk) is not None:
                        flat[f"mt/{k.replace(':', '/')}/{mk}"] = v[mk]
        for code, v in board["asr"].items():
            for mk in ("wer", "cer", "baseline_cer"):
                if v.get(mk) is not None:
                    flat[f"asr/{code}/{mk}"] = v[mk]
        for code, v in board["tts"].items():
            flat[f"tts/{code}/asr_cer"] = v["asr_cer"]
        tracker.log(flat)
        tracker.summary(flat)
        rows = []
        for lang in langs:
            c = lang.code
            rows.append([c, lang.name,
                         board["mt"].get(f"{c}:en2x", {}).get("chrf"), board["mt"].get(f"{c}:x2en", {}).get("chrf"),
                         board["mt"].get(f"{c}:en2x", {}).get("off_target"),
                         board["asr"].get(c, {}).get("cer"), board["asr"].get(c, {}).get("baseline_cer"),
                         board["tts"].get(c, {}).get("asr_cer")])
        tracker.table("leaderboard", ["lang", "name", "mt_en2x_chrf", "mt_x2en_chrf", "mt_off_target", "asr_cer",
                                      "asr_baseline_cer",
                                      "tts_intelligibility_cer"], rows)
    return board

In [ ]:
%%writefile /tmp/btp/btp/humanloop.py
"""Human-in-the-loop transcription for languages with audio but no text (pipeline B, audio_only).

For these languages no Bible text matches the recordings, and a related language's ASR
(the bridge) only produces rough drafts. The fastest route to a real model is a few hours
of corrected transcripts:

  1. `pseudo` (btp.asr.pseudo_label_set) segments recordings and stores bridge drafts.
  2. `queue` exports N clips (WAV) + a TSV with the draft for a fluent speaker to correct.
  3. `import-transcripts` reads the corrected TSV back as gold data (source="human"); ASR
     and TTS then train on it exactly like aligned Bible verses.

Clips are chosen across many chapters (not one book) so the corrected set covers varied
vocabulary, and clips already queued are never exported twice.
"""
from __future__ import annotations

import csv
import logging
import random
from pathlib import Path

from btp import audio as A

log = logging.getLogger(__name__)

QUEUE_FIELDS = ["id", "file", "duration", "draft", "corrected", "skip"]


def queue_dir(paths, lang_code: str) -> Path:
    return paths.ws / "transcription" / lang_code


def export_queue(lang, cfg: dict, paths, n: int = 300, seed: int = 0) -> dict:
    """Write <workspace>/transcription/<lang>/batch_XX/{clips/*.wav, queue.tsv}."""
    from btp.speechdata import load_audio_for, load_metadata
    rows = [r for r in load_metadata(paths, lang.code) if r["source"] == "pseudo"]
    if not rows:
        raise RuntimeError(f"No pseudo-labelled segments for {lang.code} -- run the pseudo stage first")
    qdir = queue_dir(paths, lang.code)
    done_ids = set()
    for tsv in qdir.glob("batch_*/queue.tsv"):
        with open(tsv, encoding="utf-8", newline="") as f:
            done_ids |= {r["id"] for r in csv.DictReader(f, delimiter="\t")}
    rng = random.Random(seed + len(done_ids))
    pool = [r for r in rows if r["id"] not in done_ids and 2.0 <= r["duration"] <= 12.0]
    by_chapter: dict[tuple, list[dict]] = {}
    for r in pool:
        by_chapter.setdefault((r["audio_set"], r["book"], r["chapter"]), []).append(r)
    chapters = list(by_chapter)
    rng.shuffle(chapters)
    picked: list[dict] = []
    while len(picked) < n and chapters:          # round-robin over chapters for coverage
        for key in list(chapters):
            lst = by_chapter[key]
            if not lst:
                chapters.remove(key)
                continue
            picked.append(lst.pop(rng.randrange(len(lst))))
            if len(picked) >= n:
                break
    if not picked:
        return {"exported": 0, "note": "every segment has already been queued"}
    batch = qdir / f"batch_{len(list(qdir.glob('batch_*'))) + 1:02d}"
    (batch / "clips").mkdir(parents=True, exist_ok=True)
    audio = load_audio_for(paths, picked)
    with open(batch / "queue.tsv", "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=QUEUE_FIELDS, delimiter="\t")
        w.writeheader()
        for r in picked:
            wav_name = f"{r['id']}.wav"
            A.save_audio(batch / "clips" / wav_name, A.decode_audio_bytes(audio[r["id"]]))
            w.writerow({"id": r["id"], "file": f"clips/{wav_name}", "duration": r["duration"],
                        "draft": r["text"], "corrected": "", "skip": ""})
    hours = sum(r["duration"] for r in picked) / 3600
    log.info("%s: exported %d clips (%.2f h) to %s", lang.code, len(picked), hours, batch)
    return {"exported": len(picked), "hours": round(hours, 2), "batch": str(batch)}


def import_transcripts(lang, cfg: dict, paths) -> dict:
    """Turn every corrected row of every batch into gold shards (source='human')."""
    from btp.align import write_shard
    from btp.speechdata import load_audio_for, load_metadata
    meta = {r["id"]: r for r in load_metadata(paths, lang.code) if r["source"] == "pseudo"}
    corrected: dict[str, str] = {}
    for tsv in sorted(queue_dir(paths, lang.code).glob("batch_*/queue.tsv")):
        with open(tsv, encoding="utf-8", newline="") as f:
            for r in csv.DictReader(f, delimiter="\t"):
                text = (r.get("corrected") or "").strip()
                if text and not (r.get("skip") or "").strip() and r["id"] in meta:
                    corrected[r["id"]] = text
    if not corrected:
        return {"imported": 0, "note": "no corrected rows yet"}
    rows = [meta[i] for i in corrected]
    audio = load_audio_for(paths, rows)
    out_rows = []
    for r in rows:
        out_rows.append({**r, "id": r["id"] + ".human", "text": corrected[r["id"]], "score": 0.0, "source": "human",
                         "audio": audio[r["id"]]})
    by_set: dict[str, list[dict]] = {}
    for r in out_rows:
        by_set.setdefault(r["audio_set"], []).append(r)
    for aset, rs in by_set.items():
        write_shard(rs, paths.ws / "human" / lang.code / aset / "transcribed.parquet")
    hours = sum(r["duration"] for r in out_rows) / 3600
    log.info("%s: imported %d human transcripts (%.2f h)", lang.code, len(out_rows), hours)
    return {"imported": len(out_rows), "hours": round(hours, 2)}

In [ ]:
%%writefile /tmp/btp/btp/langid.py
"""Tiny character-trigram language identifier for catching off-target translations.

A new NLLB language token starts from a related language's embedding (e.g. Samburu from
Swahili), so an under-trained model can quietly answer in the *init* language. chrF alone
does not reveal that clearly. Profiles are built from each language's own training
targets; a hypothesis is "off target" when its nearest profile is another language.
Only meaningful when 2+ profiles exist (the full run has one per language).
"""
from __future__ import annotations

import math
import random
from collections import Counter

from btp.text import normalize_transcript


def _grams(text: str) -> Counter:
    t = f"  {normalize_transcript(text)}  "
    return Counter(t[i:i + 3] for i in range(len(t) - 2))


def build_profile(texts: list[str], max_texts: int = 3000, top: int = 3000, seed: int = 0) -> dict[str, float]:
    rng = random.Random(seed)
    sample = texts if len(texts) <= max_texts else rng.sample(texts, max_texts)
    c: Counter = Counter()
    for t in sample:
        c.update(_grams(t))
    common = dict(c.most_common(top))
    norm = math.sqrt(sum(v * v for v in common.values())) or 1.0
    return {g: v / norm for g, v in common.items()}


def similarity(text: str, profile: dict[str, float]) -> float:
    g = _grams(text)
    norm = math.sqrt(sum(v * v for v in g.values())) or 1.0
    return sum(v * profile.get(k, 0.0) for k, v in g.items()) / norm


def identify(text: str, profiles: dict[str, dict[str, float]]) -> str | None:
    if not profiles or not text.strip():
        return None
    return max(profiles, key=lambda code: similarity(text, profiles[code]))


def off_target_rate(hyps: list[str], target: str, profiles: dict[str, dict[str, float]]) -> float | None:
    """Fraction of hypotheses identified as a language other than ``target`` (None if not measurable)."""
    if target not in profiles or len(profiles) < 2:
        return None
    labels = [identify(h, profiles) for h in hyps if h.strip()]
    if not labels:
        return None
    return round(sum(lab != target for lab in labels) / len(labels), 4)


def profiles_from_parallel(paths, codes: list[str]) -> dict[str, dict[str, float]]:
    """Profiles for every language that has a parallel corpus (target side of the train split)."""
    from btp.corpus import read_parallel
    out = {}
    for code in codes:
        rows = read_parallel(paths.parallel(code), "train")
        if len(rows) >= 50:
            out[code] = build_profile([r["tgt"] for r in rows])
    return out

In [ ]:
%%writefile /tmp/btp/btp/languages.py
"""Language registry backed by config/languages.yaml (single source of truth)."""
from __future__ import annotations

from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path

import yaml

from btp.config import CONFIG_DIR


@dataclass(frozen=True)
class AudioSet:
    audio: str                 # Bible.is audio fileset id, e.g. SAQBTLN1DA
    text: str | None           # matching Bible.is text fileset id, or None
    style: str                 # plain | drama
    drive_folder: str          # language folder on Drive holding this set
    variety: str | None = None

    @property
    def testament(self) -> str:
        """'N' (New Testament), 'O' (Old) or 'P'/'S' (portions) from the id."""
        return self.audio[6] if len(self.audio) > 6 else "?"


@dataclass(frozen=True)
class Language:
    code: str
    name: str
    pipeline: str              # "a" | "b"
    tier: str                  # "supported" (A) | "text" | "audio_only" (B)
    drive_folder: str
    mms_asr: str | None
    mms_tts: str | None
    nllb: str | None
    nllb_init: str | None
    tts_bridge: str | None
    asr_bridge: str | None
    sets: tuple[AudioSet, ...] = field(default_factory=tuple)
    note: str | None = None
    extra_texts: tuple[str, ...] = field(default_factory=tuple)   # parallel-only Bible versions
    openbible: str | None = None        # config of davidguzmanr/open-bible-resources (verse-level speech)
    licence: str | None = None          # usage restriction notes for this language's data

    # ---- derived helpers -------------------------------------------------
    @property
    def nllb_token(self) -> str:
        """NLLB language token: the real NLLB code, or a new '<code>_Latn' token."""
        return self.nllb or f"{self.code}_Latn"

    @property
    def needs_new_mt_token(self) -> bool:
        return self.nllb is None

    @property
    def text_sets(self) -> list[AudioSet]:
        return [s for s in self.sets if s.text]

    @property
    def untexted_sets(self) -> list[AudioSet]:
        return [s for s in self.sets if not s.text]

    @property
    def has_text(self) -> bool:
        return bool(self.text_sets)

    @property
    def asr_start(self) -> str | None:
        """MMS adapter to initialise ASR from (own adapter, else the bridge's)."""
        return self.mms_asr or self.asr_bridge

    @property
    def tts_start(self) -> str | None:
        """MMS-TTS checkpoint code to initialise TTS from (own, else the bridge's)."""
        return self.mms_tts or self.tts_bridge

    @property
    def tts_needs_bootstrap(self) -> bool:
        return self.mms_tts is None


def _build(code: str, entry: dict, pipeline: str) -> Language:
    folder = entry["drive_folder"]
    sets = tuple(
        AudioSet(
            audio=s["audio"],
            text=s.get("text"),
            style=s.get("style", "plain"),
            drive_folder=s.get("drive_folder", folder),
            variety=s.get("variety"),
        )
        for s in entry.get("sets", [])
    )
    return Language(
        code=code,
        name=entry["name"],
        pipeline=pipeline,
        tier=entry.get("tier", "supported" if pipeline == "a" else "text"),
        drive_folder=folder,
        mms_asr=entry.get("mms_asr"),
        mms_tts=entry.get("mms_tts"),
        nllb=entry.get("nllb"),
        nllb_init=entry.get("nllb_init") or entry.get("nllb"),
        tts_bridge=entry.get("tts_bridge"),
        asr_bridge=entry.get("asr_bridge"),
        sets=sets,
        note=entry.get("note"),
        extra_texts=tuple(entry.get("extra_texts") or ()),
        openbible=entry.get("openbible"),
        licence=entry.get("licence"),
    )


@lru_cache(maxsize=4)
def _registry(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        raw = yaml.safe_load(f)
    langs: dict[str, Language] = {}
    for pipeline, key in (("a", "pipeline_a"), ("b", "pipeline_b")):
        for code, entry in (raw.get(key) or {}).items():
            if code in langs:
                raise ValueError(f"Language {code} registered twice in languages.yaml")
            langs[code] = _build(code, entry, pipeline)
    return {"langs": langs, "english": raw["english"]}


def registry(path: str | Path | None = None) -> dict:
    return _registry(str(path or CONFIG_DIR / "languages.yaml"))


def get_language(code: str) -> Language:
    langs = registry()["langs"]
    if code not in langs:
        raise KeyError(f"'{code}' is not registered in config/languages.yaml "
                       f"(known: {', '.join(sorted(langs))})")
    return langs[code]


def list_languages(pipeline: str | None = None, tier: str | None = None) -> list[str]:
    out = []
    for code, lang in registry()["langs"].items():
        if pipeline and lang.pipeline != pipeline:
            continue
        if tier and lang.tier != tier:
            continue
        out.append(code)
    return out


def english_sets() -> dict:
    return dict(registry()["english"])


def resolve_langs(spec: str | list[str] | None) -> list[str]:
    """Turn 'all' / 'a' / 'b' / 'b:text' / 'swh,kik' into a list of codes."""
    if spec is None or spec == "" or spec == "all":
        return list_languages()
    if isinstance(spec, (list, tuple)):
        items = list(spec)
    else:
        items = [s.strip() for s in str(spec).split(",") if s.strip()]
    out: list[str] = []
    for item in items:
        if item in ("a", "b"):
            out += list_languages(pipeline=item)
        elif item.startswith(("a:", "b:")):
            p, t = item.split(":", 1)
            out += list_languages(pipeline=p, tier=t)
        else:
            get_language(item)  # validate
            out.append(item)
    return list(dict.fromkeys(out))

In [ ]:
%%writefile /tmp/btp/btp/mas.py
"""Monotonic Alignment Search (VITS) without Cython.

Drop-in replacement for the Cython ``maximum_path`` shipped with VITS /
finetune-hf-vits (whose build uses distutils, removed in Python 3.12). The dynamic
programme is identical to the reference implementation; the inner loop over text
positions is vectorised with NumPy, leaving a Python loop only over spectrogram frames.
"""
from __future__ import annotations

import numpy as np
import torch

_NEG = -1e9


def _maximum_path_single(value: np.ndarray, t_y: int, t_x: int) -> np.ndarray:
    """value: [t_y, t_x] float32 (modified in place). Returns int32 path [t_y, t_x]."""
    path = np.zeros_like(value, dtype=np.int32)
    if t_y == 0 or t_x == 0:
        return path
    for y in range(t_y):
        lo, hi = max(0, t_x + y - t_y), min(t_x, y + 1)
        if lo >= hi:
            continue
        xs = np.arange(lo, hi)
        if y == 0:
            v_cur = np.full(xs.shape, _NEG, dtype=value.dtype)
            v_prev = np.where(xs == 0, 0.0, _NEG).astype(value.dtype)
        else:
            v_cur = np.where(xs == y, _NEG, value[y - 1, xs])
            prev_idx = np.maximum(xs - 1, 0)
            v_prev = np.where(xs == 0, _NEG, value[y - 1, prev_idx])
        value[y, xs] += np.maximum(v_prev, v_cur)
    index = t_x - 1
    for y in range(t_y - 1, -1, -1):
        path[y, index] = 1
        if index != 0 and (index == y or value[y - 1, index] < value[y - 1, index - 1]):
            index -= 1
    return path


def maximum_path(neg_cent: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """neg_cent, mask: [b, t_t, t_s] -> hard monotonic alignment path [b, t_t, t_s]."""
    device, dtype = neg_cent.device, neg_cent.dtype
    nc = neg_cent.detach().float().cpu().numpy().astype(np.float32)
    m = mask.detach().cpu().numpy()
    t_t_max = m.sum(1)[:, 0].astype(np.int32)
    t_s_max = m.sum(2)[:, 0].astype(np.int32)
    out = np.zeros(nc.shape, dtype=np.int32)
    for b in range(nc.shape[0]):
        ty, tx = int(t_t_max[b]), int(t_s_max[b])
        out[b, :ty, :tx] = _maximum_path_single(nc[b, :ty, :tx].copy(), ty, tx)
    return torch.from_numpy(out).to(device=device, dtype=dtype)


def maximum_path_reference(neg_cent: np.ndarray, t_y: int, t_x: int) -> np.ndarray:
    """Literal scalar port of the original Cython loop -- used only by the tests."""
    value = neg_cent.astype(np.float32).copy()
    path = np.zeros_like(value, dtype=np.int32)
    for y in range(t_y):
        for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
            v_cur = _NEG if x == y else value[y - 1, x]
            if x == 0:
                v_prev = 0.0 if y == 0 else _NEG
            else:
                v_prev = value[y - 1, x - 1]
            value[y, x] += max(v_prev, v_cur)
    index = t_x - 1
    for y in range(t_y - 1, -1, -1):
        path[y, index] = 1
        if index != 0 and (index == y or value[y - 1, index] < value[y - 1, index - 1]):
            index -= 1
    return path

In [ ]:
%%writefile /tmp/btp/btp/mt.py
"""Text-to-text translation: one multilingual NLLB-200 model for English <-> every language.

* Languages NLLB already knows (swh, kik, luo, kam, som, orm->gaz_Latn) keep their codes.
* Every other language gets a NEW language token ('saq_Latn', ...) whose embedding is
  initialised from a related NLLB language (``nllb_init`` in languages.yaml), then the
  whole model is fine-tuned on the verse-aligned Bible corpora of all languages at once
  (cross-lingual transfer between the many small related languages helps each of them).
* Small corpora are up-sampled with temperature sampling (p ~ n^T).
* Languages with no parallel text at all (Luo, Kamba) get a small "replay" set of the
  base model's own translations so multilingual fine-tuning does not erase what NLLB
  already knew about them.
* Inputs are encoded explicitly as  [src_lang] tokens </s>  and labels as
  [tgt_lang] tokens </s>  -- NLLB's format -- without relying on tokenizer src/tgt
  state, which differs between transformers versions.
"""
from __future__ import annotations

import json
import logging
import math
import random
import time
from pathlib import Path

log = logging.getLogger(__name__)

EN = "eng_Latn"


# ---------------------------------------------------------------------------
# Tokenizer / model
# ---------------------------------------------------------------------------
def add_language_tokens(tokenizer, tokens: list[str]) -> list[str]:
    """Register new NLLB language codes as special tokens; returns the ones actually added."""
    vocab = tokenizer.get_vocab()
    new = [t for t in tokens if t not in vocab]
    if new:
        try:
            tokenizer.add_special_tokens({"additional_special_tokens": new},
                                         replace_additional_special_tokens=False)
        except TypeError:   # very old / very new signatures
            tokenizer.add_tokens(new, special_tokens=True)
    for t in tokens:
        tid = tokenizer.convert_tokens_to_ids(t)
        if tid is None or tid == tokenizer.unk_token_id:
            raise RuntimeError(f"language token {t} could not be registered")
    return new


def lang_id(tokenizer, code: str) -> int:
    tid = tokenizer.convert_tokens_to_ids(code)
    if tid is None or tid == tokenizer.unk_token_id:
        raise KeyError(f"{code} is not a language token of this tokenizer")
    return int(tid)


def init_new_embeddings(model, tokenizer, init_map: dict[str, str]) -> None:
    """Resize embeddings and copy each new token's row from its related language."""
    import torch
    model.resize_token_embeddings(len(tokenizer))
    emb_in = model.get_input_embeddings().weight
    emb_out = model.get_output_embeddings().weight if model.get_output_embeddings() is not None else None
    with torch.no_grad():
        for new_tok, src_tok in init_map.items():
            n, s = lang_id(tokenizer, new_tok), lang_id(tokenizer, src_tok)
            emb_in[n] = emb_in[s].clone()
            if emb_out is not None and emb_out.data_ptr() != emb_in.data_ptr():
                emb_out[n] = emb_out[s].clone()
    if hasattr(model, "final_logits_bias") and model.final_logits_bias is not None:
        if model.final_logits_bias.shape[-1] != len(tokenizer):
            import torch.nn.functional as Fn
            pad = len(tokenizer) - model.final_logits_bias.shape[-1]
            model.final_logits_bias = Fn.pad(model.final_logits_bias, (0, pad))


def encode_pair(tokenizer, src_text: str, tgt_text: str, src_code: str, tgt_code: str,
                max_src: int, max_tgt: int) -> dict:
    eos = tokenizer.eos_token_id
    s = tokenizer(src_text, add_special_tokens=False)["input_ids"][: max_src - 2]
    t = tokenizer(tgt_text, add_special_tokens=False)["input_ids"][: max_tgt - 2]
    return {"input_ids": [lang_id(tokenizer, src_code)] + s + [eos],
            "labels": [lang_id(tokenizer, tgt_code)] + t + [eos]}


def collate(batch: list[dict], pad_id: int) -> dict:
    import torch
    n = max(len(b["input_ids"]) for b in batch)
    m = max(len(b["labels"]) for b in batch)
    ids = torch.full((len(batch), n), pad_id, dtype=torch.long)
    am = torch.zeros((len(batch), n), dtype=torch.long)
    lab = torch.full((len(batch), m), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        ids[i, : len(b["input_ids"])] = torch.tensor(b["input_ids"])
        am[i, : len(b["input_ids"])] = 1
        lab[i, : len(b["labels"])] = torch.tensor(b["labels"])
    return {"input_ids": ids, "attention_mask": am, "labels": lab}


def translate(model, tokenizer, texts: list[str], src_code: str, tgt_code: str, *, num_beams: int = 4,
              batch_size: int = 16, max_len: int = 200, device: str = "cpu") -> list[str]:
    """Batch translation with explicit [src_lang] prefix and forced target-language BOS."""
    import torch
    eos = tokenizer.eos_token_id
    src_id, tgt_id = lang_id(tokenizer, src_code), lang_id(tokenizer, tgt_code)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    out: dict[int, str] = {}
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        enc = [[src_id] + tokenizer(texts[i], add_special_tokens=False)["input_ids"][: max_len - 2] + [eos] for i in idx]
        batch = collate([{"input_ids": e, "labels": [0]} for e in enc], tokenizer.pad_token_id)
        with torch.inference_mode():
            gen = model.generate(input_ids=batch["input_ids"].to(device),
                                 attention_mask=batch["attention_mask"].to(device),
                                 forced_bos_token_id=tgt_id, num_beams=num_beams, max_new_tokens=max_len)
        for i, g in zip(idx, tokenizer.batch_decode(gen, skip_special_tokens=True)):
            out[i] = g.strip()
    return [out[i] for i in range(len(texts))]


def score(hyps: list[str], refs: list[str]) -> dict:
    import sacrebleu
    if not hyps:
        return {"chrf": None, "bleu": None, "n": 0}
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    return {"chrf": round(chrf, 2), "bleu": round(bleu, 2), "n": len(hyps)}


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def gather_examples(langs, cfg: dict, paths, split: str) -> dict[str, list[dict]]:
    """{'<lang>:en2x' | '<lang>:x2en': [ {src, tgt, src_code, tgt_code, ref} ]} for one split."""
    from btp.corpus import read_parallel
    out: dict[str, list[dict]] = {}
    for lang in langs:
        rows = read_parallel(paths.parallel(lang.code), split)
        if not rows:
            continue
        code = lang.nllb_token
        for d in cfg["mt"]["directions"]:
            key = f"{lang.code}:{d}"
            if d == "en2x":
                out[key] = [{"src": r["en"], "tgt": r["tgt"], "src_code": EN, "tgt_code": code, "ref": r["ref"]}
                            for r in rows]
            else:
                out[key] = [{"src": r["tgt"], "tgt": r["en"], "src_code": code, "tgt_code": EN, "ref": r["ref"]}
                            for r in rows]
    # Human-reviewed Spirit of Prophecy sentences (btp.sop.harvest_reviews): in-domain data in their
    # own groups, so temperature sampling up-samples them and they are scored separately.
    from btp.sop import read_reviewed
    for lang in langs:
        rows = read_reviewed(paths, lang.code, split)
        if not rows:
            continue
        code = lang.nllb_token
        for d in cfg["mt"]["directions"]:
            key = f"{lang.code}:sop:{d}"
            if d == "en2x":
                out[key] = [{"src": r["en"], "tgt": r["tgt"], "src_code": EN, "tgt_code": code, "ref": r["ref"]}
                            for r in rows]
            else:
                out[key] = [{"src": r["tgt"], "tgt": r["en"], "src_code": code, "tgt_code": EN, "ref": r["ref"]}
                            for r in rows]
    return out


def sampling_weights(sizes: dict[str, int], temperature: float) -> dict[str, float]:
    tot = sum(sizes.values())
    w = {k: (n / tot) ** temperature for k, n in sizes.items() if n > 0}
    z = sum(w.values())
    return {k: v / z for k, v in w.items()}


def sample_stream(groups: dict[str, list[dict]], temperature: float, seed: int):
    """Infinite stream of examples: pick a group ~ n^T, then a random example from it."""
    rng = random.Random(seed)
    w = sampling_weights({k: len(v) for k, v in groups.items()}, temperature)
    keys, probs = list(w), list(w.values())
    while True:
        k = rng.choices(keys, probs)[0]
        yield rng.choice(groups[k])


def build_replay(model, tokenizer, langs_without_data, english_sentences: list[str], n: int, device: str,
                 out_path: Path, batch_size: int = 32) -> dict[str, list[dict]]:
    """Distillation pairs (English -> base-model translation) for languages with no parallel data."""
    from btp.utils import read_json, write_json
    cached = read_json(out_path, {}) or {}
    rng = random.Random(7)
    sents = rng.sample(english_sentences, min(n, len(english_sentences)))
    for lang in langs_without_data:
        if lang.code in cached and len(cached[lang.code]) >= len(sents):
            continue
        log.info("building %d replay pairs for %s with the base model", len(sents), lang.code)
        hyps = translate(model, tokenizer, sents, EN, lang.nllb_token, num_beams=4, batch_size=batch_size, device=device)
        cached[lang.code] = [{"en": s, "tgt": h} for s, h in zip(sents, hyps) if h]
        write_json(out_path, cached)
    out = {}
    for lang in langs_without_data:
        pairs = cached.get(lang.code, [])
        out[f"{lang.code}:en2x"] = [{"src": p["en"], "tgt": p["tgt"], "src_code": EN, "tgt_code": lang.nllb_token,
                                     "ref": "replay"} for p in pairs]
        out[f"{lang.code}:x2en"] = [{"src": p["tgt"], "tgt": p["en"], "src_code": lang.nllb_token, "tgt_code": EN,
                                     "ref": "replay"} for p in pairs]
    return out


def evaluate(model, tokenizer, groups: dict[str, list[dict]], *, limit: int | None, num_beams: int, device: str,
             batch_size: int = 32, profiles: dict | None = None) -> tuple[dict, dict[str, list]]:
    """Per-group chrF++/BLEU on (a fixed prefix of) each group's examples.

    With language ``profiles`` (btp.langid), en2x groups also get ``off_target``: the share of
    translations that look like another language (e.g. a new token still answering in its
    Swahili initialisation).
    """
    from btp.langid import off_target_rate
    metrics, samples = {}, {}
    for key, rows in sorted(groups.items()):
        rows = rows[:limit] if limit else rows
        if not rows:
            continue
        hyps = translate(model, tokenizer, [r["src"] for r in rows], rows[0]["src_code"], rows[0]["tgt_code"],
                         num_beams=num_beams, batch_size=batch_size, device=device)
        metrics[key] = score(hyps, [r["tgt"] for r in rows])
        if profiles and key.endswith(":en2x"):
            metrics[key]["off_target"] = off_target_rate(hyps, key.split(":")[0], profiles)
        samples[key] = [[key, r["ref"], r["src"], r["tgt"], h] for r, h in zip(rows[:3], hyps[:3])]
    for d in ("en2x", "x2en"):
        vals = [m["chrf"] for k, m in metrics.items() if k.endswith(d) and m["chrf"] is not None]
        if vals:
            metrics[f"macro:{d}"] = {"chrf": round(sum(vals) / len(vals), 2), "n": len(vals)}
    ot = [m["off_target"] for k, m in metrics.items() if m.get("off_target") is not None]
    if ot:
        metrics.setdefault("macro:en2x", {})["off_target"] = round(sum(ot) / len(ot), 4)
    return metrics, samples


def _flat_metrics(prefix: str, metrics: dict) -> dict:
    out = {}
    for k, m in metrics.items():
        for mk, mv in m.items():
            if mk != "n" and mv is not None:
                out[f"{prefix}/{k.replace(':', '/')}/{mk}"] = mv
    return out


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def train_mt(langs, cfg: dict, paths, tracker=None, device: str | None = None, max_steps: int | None = None,
             base_model: str | None = None, resume: bool = True) -> dict:
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_linear_schedule_with_warmup

    from btp.utils import atomic_dir, deadline_passed, resolve_dir, safe_torch_load, seed_everything, write_json
    from btp.utils import device as auto_device

    seed_everything()
    device = device or auto_device()
    c = cfg["mt"]
    base_model = base_model or c["base_model"]
    max_steps = max_steps or c["max_steps"]
    out_dir = paths.mt_dir()
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = out_dir / "checkpoint"
    from btp.utils import mark_training_done, training_done
    run_key = "mt_" + "-".join(sorted(lg.code for lg in langs))
    if resume and (prev := training_done(paths, run_key, max_steps)) is not None:
        log.info("MT already trained to %d steps for these languages -- skipped", max_steps)
        return prev

    train = gather_examples(langs, cfg, paths, "train")
    dev = gather_examples(langs, cfg, paths, "dev")
    test = gather_examples(langs, cfg, paths, "test")
    with_data = {k.split(":")[0] for k in train}
    no_data = [lg for lg in langs if lg.code not in with_data and lg.nllb]
    if not train and not no_data:
        raise RuntimeError("No parallel data -- run the corpus stage first")

    tokenizer = AutoTokenizer.from_pretrained(base_model)
    new_tokens = sorted({lg.nllb_token for lg in langs if lg.needs_new_mt_token})
    add_language_tokens(tokenizer, new_tokens)
    found = resolve_dir(ckpt_dir) if resume else None
    start_from = found if (found is not None and (found / "model.safetensors").is_file()) else None
    warm_step = 0
    vocab_grew = False
    if start_from is None and resume:
        # Training finished earlier and its checkpoint was pruned; max_steps has since been raised.
        # Continue from the finished model (fresh optimiser) instead of restarting from the base model.
        from btp.utils import read_json
        rec = read_json(paths.status() / f"train_{run_key}.json", None)
        fin = resolve_dir(out_dir / "final")
        if rec and fin is not None and (fin / "model.safetensors").is_file():
            start_from, warm_step = fin, int(rec.get("steps", 0))
            log.info("MT checkpoint was pruned -- continuing from final/ at step %d", warm_step)
    if start_from:
        model = AutoModelForSeq2SeqLM.from_pretrained(str(start_from)).float()   # final/ is stored in fp16
        tokenizer = AutoTokenizer.from_pretrained(str(start_from))
        add_language_tokens(tokenizer, new_tokens)   # tokens of newly added languages
        if model.get_input_embeddings().weight.shape[0] != len(tokenizer):
            # Languages were added since the checkpoint: the embedding matrix grows, so the saved
            # optimiser state no longer matches the parameters (Kaggle 2026-11-19: 256204 vs 256215
            # crash in the first optimiser step). Keep the weights, start a fresh optimiser.
            vocab_grew = True
            init_new_embeddings(model, tokenizer, {lg.nllb_token: lg.nllb_init or "swh_Latn" for lg in langs
                                                   if lg.needs_new_mt_token})
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(base_model)
        init_new_embeddings(model, tokenizer, {lg.nllb_token: lg.nllb_init or "swh_Latn" for lg in langs
                                               if lg.needs_new_mt_token})
    model.to(device)
    if c.get("gradient_checkpointing"):
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    if no_data:
        english = sorted({ex["src"] for k, v in train.items() if k.endswith(":en2x") for ex in v})
        model.eval()
        train.update(build_replay(model, tokenizer, no_data, english, c.get("replay_pairs", 2000), device,
                                  out_dir / "replay.json"))
    sizes = {k: len(v) for k, v in train.items()}
    log.info("MT training groups: %s", sizes)
    from btp.langid import profiles_from_parallel
    from btp.languages import list_languages
    profiles = profiles_from_parallel(paths, list_languages())
    log.info("language-ID profiles for off-target checks: %s", sorted(profiles))

    params = [p for p in model.parameters() if p.requires_grad]
    if c.get("optimizer", "adamw") == "adafactor":
        from transformers.optimization import Adafactor
        opt = Adafactor(params, lr=c["lr"], relative_step=False, scale_parameter=False, warmup_init=False)
    else:
        opt = torch.optim.AdamW(params, lr=c["lr"], weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, c["warmup_steps"], max_steps)
    use_amp = c.get("fp16", True) and device.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    st = None
    step = warm_step
    if start_from is not None:
        st, _src = safe_torch_load(start_from / "trainer_state.pt")
    if st is not None:
        if vocab_grew:
            log.warning("vocabulary grew by new language tokens -- optimiser state reset (weights kept)")
        elif not _opt_state_fits(st["opt"], opt):
            log.warning("saved optimiser state does not match the parameters -- optimiser state reset")
        else:
            try:
                opt.load_state_dict(st["opt"])
                sched.load_state_dict(st["sched"])
                scaler.load_state_dict(st["scaler"])
            except Exception as e:  # noqa: BLE001 -- keep weights, reset optimiser
                log.warning("optimizer state not restored (%s)", e)
        step = st["step"]
        log.info("resumed MT from step %d", step)

    prev_final = resolve_dir(out_dir / "final")
    prev_meta = (json.loads((prev_final / "btp_mt.json").read_text())
                 if prev_final is not None and (prev_final / "btp_mt.json").is_file() else {})
    best = prev_meta.get("best_dev_chrf")
    if set(prev_meta.get("languages") or {}) != {lg.code for lg in langs}:
        # a best score over a different language set (e.g. Luo alone) is not comparable with this macro
        # average -- keeping it would stop final/ from ever being updated
        best = None
    if step == 0 and tracker:
        model.eval()
        m0, _ = evaluate(model, tokenizer, dev, limit=c["eval_samples_per_lang"] // 2, num_beams=1, device=device,
                         profiles=profiles)
        tracker.log(_flat_metrics("dev", m0), step=0)
        log.info("zero-shot dev macro: %s", {k: v for k, v in m0.items() if k.startswith("macro")})

    stream = sample_stream(train, c["temperature_sampling"], seed=1000 + step)
    bs, accum = c["per_device_batch"], c["grad_accum"]
    pad = tokenizer.pad_token_id
    model.train()
    t0, toks = time.time(), 0
    while step < max_steps:
        for _ in range(accum):
            exs = [next(stream) for _ in range(bs)]
            enc = [encode_pair(tokenizer, e["src"], e["tgt"], e["src_code"], e["tgt_code"], c["max_source_len"],
                               c["max_target_len"]) for e in exs]
            batch = {k: v.to(device) for k, v in collate(enc, pad).items()}
            toks += int((batch["labels"] != -100).sum())
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss / accum
            scaler.scale(loss).backward()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
        if torch.isfinite(gn):
            scaler.step(opt)
        scaler.update()
        sched.step()
        opt.zero_grad(set_to_none=True)
        step += 1
        if step % 25 == 0:
            dt = time.time() - t0
            if tracker:
                tracker.log({"train/loss": loss.item() * accum, "train/grad_norm": float(gn),
                             "train/lr": sched.get_last_lr()[0], "train/tokens_per_sec": toks / max(dt, 1e-6)},
                            step=step)
            t0, toks = time.time(), 0
        if step % c["eval_steps"] == 0 or step == max_steps:
            model.eval()
            m, samples = evaluate(model, tokenizer, dev, limit=c["eval_samples_per_lang"], num_beams=1, device=device,
                                  profiles=profiles)
            model.train()
            macro = m.get("macro:en2x", {}).get("chrf")
            log.info("MT step %d dev macro en2x chrF++ %s, x2en %s", step, macro, m.get("macro:x2en", {}).get("chrf"))
            if tracker:
                tracker.log(_flat_metrics("dev", m), step=step)
                tracker.table("dev/samples", ["group", "ref", "source", "reference", "hypothesis"],
                              [row for v in samples.values() for row in v], step=step)
            if macro is not None and (best is None or macro > best):
                best = macro
                _save_final(model, tokenizer, out_dir / "final", langs, step, best)
        out_of_time = deadline_passed(margin_sec=900)
        if out_of_time:
            log.warning("MT: deadline near -- checkpointing at step %d and stopping (resume next run)", step)
        if step % c["save_steps"] == 0 or step == max_steps or out_of_time:
            with atomic_dir(ckpt_dir) as tmp:          # model + optimiser saved as one unit
                model.save_pretrained(str(tmp), safe_serialization=True)
                tokenizer.save_pretrained(str(tmp))
                torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                            "step": step}, tmp / "trainer_state.pt")
            log.info("MT checkpoint saved at step %d", step)
        if out_of_time:
            break
    if resolve_dir(out_dir / "final") is None:
        _save_final(model, tokenizer, out_dir / "final", langs, step, best)
    if step >= max_steps and c.get("prune_checkpoint_when_done") and not deadline_passed():
        import shutil
        shutil.rmtree(ckpt_dir, ignore_errors=True)       # training finished: keep only final/ (storage)
        log.info("MT finished -- training checkpoint removed to save storage (final/ kept)")

    # held-out test with beam search on the best model
    final_dir = resolve_dir(out_dir / "final")
    final = AutoModelForSeq2SeqLM.from_pretrained(str(final_dir)).float().to(device).eval()
    ftok = AutoTokenizer.from_pretrained(str(final_dir))
    tm, samples = evaluate(final, ftok, test, limit=None, num_beams=c["num_beams"], device=device,
                          profiles=profiles)
    write_json(paths.reports() / "mt_test.json", tm)
    if tracker:
        tracker.summary(_flat_metrics("test", tm))
        tracker.table("test/samples", ["group", "ref", "source", "reference", "hypothesis"],
                      [row for v in samples.values() for row in v])
    result = {"steps": step, "best_dev_chrf": best, "test": tm}
    if step >= max_steps:
        mark_training_done(paths, run_key, step, result)
    return result


def _save_final(model, tokenizer, out: Path, langs, step: int, best) -> None:
    import torch

    from btp.utils import atomic_dir
    meta = {"step": step, "best_dev_chrf": best, "dtype": "float16",
            "languages": {lg.code: lg.nllb_token for lg in langs}, "english": EN}
    with atomic_dir(out) as tmp:
        # fp16 copy (half the size: 1.2 instead of 2.4 GB -- Kaggle's 20 GB output limit); tied
        # weights (shared embeddings / lm_head) stay one tensor so they are stored once
        half: dict = {}
        sd = {k: half.setdefault(v.data_ptr(), v.detach().to("cpu", torch.float16)) if v.is_floating_point() else v
              for k, v in model.state_dict().items()}
        model.save_pretrained(str(tmp), state_dict=sd, safe_serialization=True)
        tokenizer.save_pretrained(str(tmp))
        (tmp / "btp_mt.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")


def _opt_state_fits(saved: dict, opt) -> bool:
    """Do the saved optimiser state's tensors have the shapes of the current parameters?
    (load_state_dict does not check; a mismatch only crashes later inside opt.step)."""
    try:
        groups = [p for g in opt.param_groups for p in g["params"]]
        state = saved.get("state", {})
        if len(groups) != sum(len(g["params"]) for g in saved.get("param_groups", [])):
            return False
        for i, p in enumerate(groups):
            for name, v in state.get(i, {}).items():
                if not hasattr(v, "shape") or v.dim() == 0:
                    continue
                if name in ("exp_avg_sq_row",):
                    ok = tuple(v.shape) == tuple(p.shape[:-1])
                elif name in ("exp_avg_sq_col",):
                    ok = tuple(v.shape) == tuple(p.shape[:-2]) + tuple(p.shape[-1:])
                else:
                    ok = tuple(v.shape) == tuple(p.shape)
                if not ok:
                    return False
        return True
    except Exception:  # noqa: BLE001
        return False


def load_mt(model_dir: str | Path, device: str = "cpu"):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    model = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir)).float().to(device).eval()
    tok = AutoTokenizer.from_pretrained(str(model_dir))
    meta_p = Path(model_dir) / "btp_mt.json"
    meta = json.loads(meta_p.read_text(encoding="utf-8")) if meta_p.is_file() else {}
    return model, tok, meta


def steps_per_epoch(sizes: dict[str, int], batch: int) -> int:
    return math.ceil(sum(sizes.values()) / max(1, batch))

In [ ]:
%%writefile /tmp/btp/btp/openbible.py
"""Import ready-made verse-level speech from the Open.Bible dataset on Hugging Face.

``davidguzmanr/open-bible-resources`` holds Biblica Open.Bible recordings already split per verse
(audio + text + book/chapter/verse + speaker), for Dholuo, Kikuyu, Swahili and Oromo among others.
For those languages this replaces forced alignment entirely -- no GPU time spent cutting audio.

Licence: Open.Bible content is CC BY-SA 4.0 (attribution required; derivatives that are distributed
must carry the same licence; the Biblica trademark may not appear on altered works). The attribution
is written to ATTRIBUTION.md next to the imported data.

Output uses the aligned-clip shard format (source="aligned", score 0.0) under
    <workspace>/aligned/<lang>/OB-<config>-<speaker>/<BOOK>.ob<shard>.parquet
so speech data selection, held-out splits and TTS single-voice selection need no changes (each
speaker is its own "audio set"). Parquet files are read directly with pyarrow -- no datasets.Audio
decoding (which needs torchcodec) -- one file at a time, resumable per file.
"""
from __future__ import annotations

import logging
from collections import defaultdict
from pathlib import Path

from btp import audio as A

log = logging.getLogger(__name__)

REPO_ID = "davidguzmanr/open-bible-resources"
ATTRIBUTION = """# Attribution -- Open.Bible recordings

Audio and text imported from the Hugging Face dataset `davidguzmanr/open-bible-resources`
(configuration: {config}), which redistributes Biblica Open.Bible resources.

Original work available for free at www.open.bible, (c) Biblica, Inc., licensed under the Creative
Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0):
https://creativecommons.org/licenses/by-sa/4.0/

Changes: audio resampled to 16 kHz mono FLAC and re-packaged; clips filtered by duration.
Any distributed derivative must carry the same licence; "Biblica" may not appear on altered works.
"""


def _book_id(name: str) -> str | None:
    from btp.sop import _NAME_TO_ID
    return _NAME_TO_ID.get(str(name).strip().lower()) or _NAME_TO_ID.get(str(name).strip().lower().replace(" ", ""))


def list_files(config: str) -> list[str]:
    from huggingface_hub import HfApi
    files = HfApi().list_repo_files(REPO_ID, repo_type="dataset")
    return sorted(f for f in files if f.startswith(f"{config}/") and f.endswith(".parquet"))


def openbible_hours(paths, lang_code: str) -> float:
    from btp.utils import read_json
    st = read_json(paths.status() / f"openbible_{lang_code}.json", {}) or {}
    return float(st.get("hours", 0.0))


def import_openbible(lang, cfg: dict, paths, max_files: int | None = None, hours_cap: float | None = None,
                     heldout_cap: int = 400) -> dict:
    """Import ``lang.openbible`` (an Open Bible config name). Train clips stop at ``hours_cap``;
    held-out (dev/test) book clips are always kept, up to ``heldout_cap`` per split."""
    import pyarrow.parquet as pq
    from huggingface_hub import hf_hub_download

    from btp.align import write_shard
    from btp.corpus import split_of
    from btp.utils import Status, deadline_passed

    config = getattr(lang, "openbible", None)
    if not config:
        return {"skipped": "no Open Bible config for this language"}
    a = cfg["align"]
    hours_cap = a.get("max_hours_per_lang") if hours_cap is None else hours_cap
    status = Status(paths.status() / f"openbible_{lang.code}.json")
    hours = float(status.data.get("hours", 0.0))
    held = dict(status.data.get("heldout", {"dev": 0, "test": 0}))
    base = paths.ws / "aligned" / lang.code
    base.mkdir(parents=True, exist_ok=True)
    (base / "ATTRIBUTION-openbible.md").write_text(ATTRIBUTION.format(config=config), encoding="utf-8")
    stats = {"files": 0, "kept": 0, "skipped_duration": 0, "skipped_book": 0, "skipped_cap": 0}
    files = list_files(config)
    if not files:
        raise RuntimeError(f"Open Bible config {config!r} has no parquet files")
    cache = paths.cache / "openbible"
    for i, fname in enumerate(files):
        if max_files is not None and stats["files"] >= max_files:
            break
        if status.is_done(fname):
            continue
        if deadline_passed(margin_sec=900):
            log.warning("%s: deadline near -- Open Bible import stops before %s", lang.code, fname)
            break
        local = hf_hub_download(REPO_ID, fname, repo_type="dataset", cache_dir=str(cache))
        pf = pq.ParquetFile(local)
        by_set_book: dict[tuple[str, str], list[dict]] = defaultdict(list)
        for batch in pf.iter_batches(batch_size=256):
            for r in batch.to_pylist():
                book = _book_id(r.get("book"))
                if not book:
                    stats["skipped_book"] += 1
                    continue
                dur = float(r.get("duration_seconds") or 0)
                if not (a["min_clip_sec"] <= dur <= a["max_clip_sec"]):
                    stats["skipped_duration"] += 1
                    continue
                sp = split_of(book, cfg)
                if sp == "train" and hours_cap is not None and hours >= hours_cap:
                    stats["skipped_cap"] += 1
                    continue
                if sp != "train" and held.get(sp, 0) >= heldout_cap:
                    continue
                audio = r.get("audio") or {}
                raw = audio.get("bytes") if isinstance(audio, dict) else None
                if not raw:
                    continue
                try:
                    wav = A.load_audio(raw)
                except Exception:  # noqa: BLE001 -- skip an undecodable clip, keep the file going
                    continue
                ch, vs = int(r["chapter"]), int(str(r["verse"]).split("-")[0])
                ve = int(str(r["verse"]).split("-")[-1])
                aset = f"OB-{config}-{r.get('speaker_id') or 'SPEAKER_00'}"
                by_set_book[(aset, book)].append({
                    "id": f"{aset}.{book}.{ch}.{vs}", "lang": lang.code, "audio_set": aset, "style": "plain",
                    "book": book, "chapter": ch, "verse_start": vs, "verse_end": ve, "text": r.get("text") or "",
                    "start": 0.0, "end": round(A.duration(wav), 3), "duration": round(A.duration(wav), 3),
                    "score": 0.0, "source": "aligned", "audio": A.encode_flac(wav)})
                if sp == "train":
                    hours += A.duration(wav) / 3600
                else:
                    held[sp] = held.get(sp, 0) + 1
                stats["kept"] += 1
        for (aset, book), rows in by_set_book.items():
            write_shard(rows, base / aset / f"{book}.ob{i:03d}.parquet")
        Path(local).unlink(missing_ok=True)                    # keep local disk small
        stats["files"] += 1
        status.mark_done(fname, kept=sum(len(v) for v in by_set_book.values()))
        status.set("hours", round(hours, 3))
        status.set("heldout", held)
        log.info("%s Open Bible %s: %d/%d files, %.2f h train, held-out %s", lang.code, config, i + 1, len(files),
                 hours, held)
    return {**stats, "hours": round(hours, 3), "heldout": held, "config": config}

In [ ]:
%%writefile /tmp/btp/btp/sop.py
"""Spirit of Prophecy (Ellen G. White) book support.

* Scripture references: parse "John 3:16", "1 Cor. 13:4-7", "Ps. 23:1, 2" ... and look up the
  verse in each language's own Bible (the Bible.is text corpus), so quoted Scripture is not
  machine-translated. EGW quotes mostly the KJV while our English Bible is the WEB, so a quote is
  matched to its verse by reference + character similarity, not exact text.
* Glossary: agreed translations of key terms (config/glossary.yaml); review sheets flag rows whose
  machine translation does not use the agreed term.
* Review -> retrain loop: every reviewed sentence of every book becomes an in-domain parallel pair
  (corpus/reviewed/<lang>.tsv), used for MT training and as a held-out SoP test set.
"""
from __future__ import annotations

import csv
import hashlib
import logging
import re
from functools import lru_cache
from pathlib import Path

import yaml

from btp.config import CONFIG_DIR

log = logging.getLogger(__name__)

# English book names / common abbreviations -> USFM id
_BOOKS = {
    "GEN": ["Genesis", "Gen"], "EXO": ["Exodus", "Ex", "Exod"], "LEV": ["Leviticus", "Lev"],
    "NUM": ["Numbers", "Num"], "DEU": ["Deuteronomy", "Deut", "Dt"], "JOS": ["Joshua", "Josh"],
    "JDG": ["Judges", "Judg"], "RUT": ["Ruth"], "1SA": ["1 Samuel", "1 Sam"], "2SA": ["2 Samuel", "2 Sam"],
    "1KI": ["1 Kings", "1 Kgs"], "2KI": ["2 Kings", "2 Kgs"], "1CH": ["1 Chronicles", "1 Chron", "1 Chr"],
    "2CH": ["2 Chronicles", "2 Chron", "2 Chr"], "EZR": ["Ezra"], "NEH": ["Nehemiah", "Neh"],
    "EST": ["Esther", "Esth"], "JOB": ["Job"], "PSA": ["Psalms", "Psalm", "Ps", "Psa"],
    "PRO": ["Proverbs", "Prov", "Pr"], "ECC": ["Ecclesiastes", "Eccl", "Eccles"],
    "SNG": ["Song of Solomon", "Song of Songs", "Song"], "ISA": ["Isaiah", "Isa"], "JER": ["Jeremiah", "Jer"],
    "LAM": ["Lamentations", "Lam"], "EZK": ["Ezekiel", "Ezek"], "DAN": ["Daniel", "Dan"],
    "HOS": ["Hosea", "Hos"], "JOL": ["Joel"], "AMO": ["Amos"], "OBA": ["Obadiah", "Obad"],
    "JON": ["Jonah", "Jon"], "MIC": ["Micah", "Mic"], "NAM": ["Nahum", "Nah"], "HAB": ["Habakkuk", "Hab"],
    "ZEP": ["Zephaniah", "Zeph"], "HAG": ["Haggai", "Hag"], "ZEC": ["Zechariah", "Zech"],
    "MAL": ["Malachi", "Mal"], "MAT": ["Matthew", "Matt", "Mt"], "MRK": ["Mark", "Mk"],
    "LUK": ["Luke", "Lk"], "JHN": ["John", "Jn"], "ACT": ["Acts"], "ROM": ["Romans", "Rom"],
    "1CO": ["1 Corinthians", "1 Cor"], "2CO": ["2 Corinthians", "2 Cor"], "GAL": ["Galatians", "Gal"],
    "EPH": ["Ephesians", "Eph"], "PHP": ["Philippians", "Phil"], "COL": ["Colossians", "Col"],
    "1TH": ["1 Thessalonians", "1 Thess"], "2TH": ["2 Thessalonians", "2 Thess"],
    "1TI": ["1 Timothy", "1 Tim"], "2TI": ["2 Timothy", "2 Tim"], "TIT": ["Titus", "Tit"],
    "PHM": ["Philemon", "Philem"], "HEB": ["Hebrews", "Heb"], "JAS": ["James", "Jas"],
    "1PE": ["1 Peter", "1 Pet"], "2PE": ["2 Peter", "2 Pet"], "1JN": ["1 John"], "2JN": ["2 John"],
    "3JN": ["3 John"], "JUD": ["Jude"], "REV": ["Revelation", "Rev"],
}
_NAME_TO_ID = {}
for _id, _names in _BOOKS.items():
    for _n in _names:
        _NAME_TO_ID[_n.lower()] = _id
        _NAME_TO_ID[_n.lower().replace(" ", "")] = _id
        if _n[0] in "123":                       # "I John" / "First John" style ordinals
            roman = {"1": "I", "2": "II", "3": "III"}[_n[0]]
            _NAME_TO_ID[(roman + _n[1:]).lower()] = _id
_name_alt = "|".join(sorted((re.escape(n) for _id, ns in _BOOKS.items() for n in ns), key=len, reverse=True))
REF_RE = re.compile(rf"\b(?P<book>{_name_alt})\.?\s+(?P<ch>\d{{1,3}}):(?P<vs>\d{{1,3}}(?:\s*[-–,]\s*\d{{1,3}})*)")


def parse_refs(text: str) -> list[dict]:
    """[{book, chapter, verses:[...], span:(start,end), raw}] for every Scripture reference in ``text``."""
    out = []
    for m in REF_RE.finditer(text):
        book = _NAME_TO_ID.get(m.group("book").lower().rstrip("."))
        if not book:
            continue
        verses: list[int] = []
        for part in re.split(r"\s*,\s*", m.group("vs")):
            if re.search(r"[-–]", part):
                a, b = (int(x) for x in re.split(r"\s*[-–]\s*", part)[:2])
                if b >= a and b - a < 60:
                    verses += list(range(a, b + 1))
            elif part.strip().isdigit():
                verses.append(int(part))
        if verses:
            out.append({"book": book, "chapter": int(m.group("ch")), "verses": sorted(set(verses)),
                        "span": m.span(), "raw": m.group(0)})
    return out


class BibleLookup:
    """Verse text by (book, chapter, verse) from one or more cached text sets of a language."""

    def __init__(self, rows: list[dict]):
        self.v: dict[tuple[str, int, int], str] = {}
        for r in rows:
            for vn in range(int(r["verse_start"]), int(r["verse_end"]) + 1):
                self.v.setdefault((r["book"], int(r["chapter"]), vn), r["text"])

    @classmethod
    def for_language(cls, lang, paths) -> BibleLookup:
        from btp.bibleis import read_text_jsonl
        rows: list[dict] = []
        for t in dict.fromkeys(s.text for s in lang.text_sets):
            rows += read_text_jsonl(paths.text_set(t))
        return cls(rows)

    @classmethod
    def english(cls, paths) -> BibleLookup:
        from btp.bibleis import read_text_jsonl
        from btp.languages import english_sets
        en = english_sets()
        return cls(read_text_jsonl(paths.text_set(en["nt"])) + read_text_jsonl(paths.text_set(en["ot"])))

    def passage(self, ref: dict) -> str | None:
        texts = []
        for vn in ref["verses"]:
            t = self.v.get((ref["book"], ref["chapter"], vn))
            if t is None:
                return None                      # incomplete passage -> do not substitute
            if not texts or texts[-1] != t:      # combined verses share one text
                texts.append(t)
        return " ".join(texts) if texts else None

    def __len__(self) -> int:
        return len(self.v)


_QUOTE_RE = re.compile(r"[\"“](?P<q>[^\"“”]{12,})[\"”]")


def similarity(a: str, b: str) -> float:
    import sacrebleu
    return sacrebleu.sentence_chrf(a, [b]).score


def scripture_for_unit(text: str, en_bible: BibleLookup, tgt_bible: BibleLookup | None,
                       min_chrf: float = 30.0, whole_unit_share: float = 0.7) -> dict:
    """Scripture handling for one English translation unit.

    Returns {"refs": "John 3:16; ...", "target_verses": "...", "substitute": str | None}.
    ``substitute`` is set only when the unit is essentially one quoted passage whose text matches
    the referenced English verse: then the target-language verse replaces machine translation.
    """
    refs = parse_refs(text)
    if not refs:
        return {"refs": "", "target_verses": "", "substitute": None}
    tgt_texts = [tgt_bible.passage(r) if tgt_bible else None for r in refs]
    info = {"refs": "; ".join(r["raw"] for r in refs),
            "target_verses": " | ".join(f"{r['raw']}: {t}" for r, t in zip(refs, tgt_texts) if t),
            "substitute": None}
    quotes = list(_QUOTE_RE.finditer(text))
    if len(refs) == 1 and len(quotes) == 1 and tgt_texts[0]:
        q = quotes[0].group("q")
        en_verse = en_bible.passage(refs[0])
        if en_verse and len(q) >= whole_unit_share * len(text.replace(refs[0]["raw"], "").strip(" .()\"“”"))\
                and similarity(q, en_verse) >= min_chrf:
            info["substitute"] = f"“{tgt_texts[0]}” ({refs[0]['raw']})"
    return info


# ---------------------------------------------------------------------------
# Glossary
# ---------------------------------------------------------------------------
@lru_cache(maxsize=2)
def load_glossary(path: str | None = None) -> list[dict]:
    p = Path(path) if path else CONFIG_DIR / "glossary.yaml"
    if not p.is_file():
        return []
    data = yaml.safe_load(p.read_text(encoding="utf-8")) or {}
    return list(data.get("terms") or [])


def glossary_flags(source_en: str, translation: str, lang_code: str, terms: list[dict] | None = None) -> list[str]:
    """English glossary terms present in the source whose agreed target term is missing from the translation."""
    terms = load_glossary() if terms is None else terms
    flags = []
    src, tgt = source_en.lower(), translation.lower()
    for t in terms:
        target = (t.get(lang_code) or "").strip()
        if not target:
            continue
        if re.search(rf"\b{re.escape(t['en'].lower())}\b", src):
            alternatives = [a.strip().lower() for a in target.split("|") if a.strip()]
            if not any(a in tgt for a in alternatives):
                flags.append(f"{t['en']} -> {target}")
    return flags


# ---------------------------------------------------------------------------
# Review -> retrain loop
# ---------------------------------------------------------------------------
REVIEWED_FIELDS = ["ref", "book", "split", "en", "tgt"]


def _split_for(key: str, test_share: float = 0.1, dev_share: float = 0.05) -> str:
    h = int(hashlib.sha1(key.encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "test" if h < test_share else "dev" if h < test_share + dev_share else "train"


def harvest_reviews(lang, paths) -> dict:
    """Collect every human-reviewed sentence of every book review sheet into corpus/reviewed/<lang>.tsv.

    Rows are assigned to train/dev/test by a stable hash of (book, sentence), so a sentence keeps its
    split forever and the SoP test set only grows. Scripture-substituted and unchanged rows are kept
    out: they carry no new human translation.
    """
    out_rows, seen = [], set()
    for tsv in sorted((paths.books() / lang.code).glob("*.review.tsv")):
        book = tsv.name[: -len(".review.tsv")]
        with open(tsv, encoding="utf-8", newline="") as f:
            for r in csv.DictReader(f, delimiter="\t"):
                rev = (r.get("reviewed_translation") or "").strip()
                src = (r.get("source_en") or "").strip()
                if not rev or not src or (r.get("scripture_substituted") or "").strip().lower() == "true":
                    continue
                key = f"{book}:{src}"
                if key in seen:
                    continue
                seen.add(key)
                out_rows.append({"ref": f"sop:{book}:{r.get('unit', '')}", "book": book, "split": _split_for(key),
                                 "en": src, "tgt": rev})
    path = paths.ws / "corpus" / "reviewed" / f"{lang.code}.tsv"
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=REVIEWED_FIELDS, delimiter="\t")
        w.writeheader()
        w.writerows(out_rows)
    counts = {s: sum(r["split"] == s for r in out_rows) for s in ("train", "dev", "test")}
    log.info("%s: %d reviewed SoP sentences harvested %s", lang.code, len(out_rows), counts)
    return {"reviewed": len(out_rows), **counts}


def read_reviewed(paths, lang_code: str, split: str | None = None) -> list[dict]:
    p = paths.ws / "corpus" / "reviewed" / f"{lang_code}.tsv"
    if not p.is_file():
        return []
    with open(p, encoding="utf-8", newline="") as f:
        return [r for r in csv.DictReader(f, delimiter="\t") if split is None or r["split"] == split]

In [ ]:
%%writefile /tmp/btp/btp/speechdata.py
"""Select (audio, text) clips for ASR / TTS from aligned + pseudo-labelled shards.

Rows come from the parquet shards written by btp.align (source="aligned") and
btp.asr.pseudo_label (source="pseudo"). Splits follow the shared held-out books, so a
verse used to test MT is never heard in ASR/TTS training either.
"""
from __future__ import annotations

import logging
import random
from collections import defaultdict
from pathlib import Path

from btp.align import read_shards
from btp.corpus import split_of

log = logging.getLogger(__name__)

META_COLS = ["id", "lang", "audio_set", "style", "book", "chapter", "verse_start", "verse_end",
             "text", "duration", "score", "source"]


def shard_files(paths, lang: str, include_pseudo: bool = True, include_partial: bool = True) -> list[Path]:
    files = []
    roots = [paths.ws / "aligned" / lang, paths.ws / "human" / lang]
    if include_pseudo:
        roots.append(paths.ws / "pseudo" / lang)
    for root in roots:
        if root.is_dir():
            for p in sorted(root.glob("*/*.parquet")):
                if p.name.endswith(".partial.parquet") and not include_partial:
                    continue
                files.append(p)
    return files


def load_metadata(paths, lang: str, include_pseudo: bool = True, include_partial: bool = True) -> list[dict]:
    """Metadata of every clip of ``lang`` (audio bytes NOT loaded)."""
    return read_shards(shard_files(paths, lang, include_pseudo, include_partial), columns=META_COLS)


def summarize(rows: list[dict]) -> dict:
    out: dict = defaultdict(lambda: {"clips": 0, "hours": 0.0})
    for r in rows:
        k = f"{r['audio_set']}:{r['source']}"
        out[k]["clips"] += 1
        out[k]["hours"] += r["duration"] / 3600
    return {k: {"clips": v["clips"], "hours": round(v["hours"], 2)} for k, v in out.items()}


def select_clips(rows: list[dict], cfg: dict, *, task: str, hours_cap: float, max_sec: float,
                 min_score: float | None = None, min_conf: float | None = None,
                 voice_set: str | None = None, prefer_style: str | None = None,
                 seed: int = 1234) -> dict[str, list[dict]]:
    """Return {"train": [...], "dev": [...], "test": [...]} metadata rows.

    task="tts": keep ONE voice (a single audio set) for a consistent speaker. Defaults to
    the audio set with the most aligned hours in ``prefer_style`` (plain narration), or
    the largest set overall if no set of that style exists.
    task="asr": use every set (speaker variety helps ASR), capped at ``hours_cap``.
    """
    rng = random.Random(seed)
    min_score = cfg["align"]["min_score"] if min_score is None else min_score
    min_conf = cfg["pseudo_label"]["min_confidence"] if min_conf is None else min_conf

    def ok(r: dict) -> bool:
        if r["duration"] > max_sec or r["duration"] < cfg["align"]["min_clip_sec"]:
            return False
        if r["source"] == "human":
            return True
        if r["source"] == "aligned":
            return r["score"] >= min_score
        return r["score"] >= min_conf

    rows = [r for r in rows if ok(r)]
    if task == "tts":
        hours: dict[str, float] = defaultdict(float)
        style_of: dict[str, str] = {}
        for r in rows:
            hours[r["audio_set"]] += r["duration"] / 3600
            style_of[r["audio_set"]] = r["style"]
        if not hours:
            return {"train": [], "dev": [], "test": []}
        if voice_set is None:
            preferred = [s for s in hours if style_of[s] == prefer_style] if prefer_style else []
            pool = preferred or list(hours)
            voice_set = max(pool, key=lambda s: hours[s])
        rows = [r for r in rows if r["audio_set"] == voice_set]
        log.info("TTS voice: %s (%s, %.1f h usable)", voice_set, style_of.get(voice_set), hours.get(voice_set, 0))

    splits: dict[str, list[dict]] = {"train": [], "dev": [], "test": []}
    for r in rows:
        splits[split_of(r["book"], cfg)].append(r)
    rng.shuffle(splits["train"])
    capped, total = [], 0.0
    for r in splits["train"]:
        if total >= hours_cap * 3600:
            break
        capped.append(r)
        total += r["duration"]
    splits["train"] = capped
    for k in ("dev", "test"):
        splits[k].sort(key=lambda r: r["id"])
    return splits


def load_audio_for(paths, rows: list[dict]) -> dict[str, bytes]:
    """Fetch FLAC bytes for the given metadata rows (reads only the shards needed)."""
    import pyarrow.parquet as pq
    want: dict[tuple[str, str, str, str], set[str]] = defaultdict(set)
    for r in rows:
        want[(r["lang"], r["audio_set"], r["book"], r["source"])].add(r["id"])
    out: dict[str, bytes] = {}
    for (lang, aset, book, source), ids in want.items():
        base = paths.ws / source / lang / aset          # aligned | pseudo | human
        if source == "human":
            files = [base / "transcribed.parquet"]
        else:   # <BOOK>.parquet, <BOOK>.partial.parquet, <BOOK>.ob<NNN>.parquet (Open Bible import)
            files = sorted(base.glob(f"{book}.parquet")) + sorted(base.glob(f"{book}.partial.parquet")) \
                + sorted(base.glob(f"{book}.ob*.parquet"))
        files = [f for f in files if f.is_file()]
        if not files:
            log.warning("no shard for %s/%s/%s (%s)", lang, aset, book, source)
            continue
        t = [row for f in files for row in pq.read_table(f, columns=["id", "audio"]).to_pylist()]
        for row in t:
            if row["id"] in ids:
                out[row["id"]] = row["audio"]
    return out


def materialize(paths, rows: list[dict], out_path: Path) -> Path:
    """Write selected clips (metadata + audio) into one local parquet for fast training I/O."""
    import pyarrow as pa
    import pyarrow.parquet as pq
    audio = load_audio_for(paths, rows)
    table = pa.Table.from_pylist([{**{k: r[k] for k in META_COLS}, "audio": audio[r["id"]]}
                                  for r in rows if r["id"] in audio])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(table, out_path, compression="zstd")
    return out_path


def read_materialized(path: Path) -> list[dict]:
    import pyarrow.parquet as pq
    return pq.read_table(path).to_pylist() if Path(path).is_file() else []

In [ ]:
%%writefile /tmp/btp/btp/text.py
"""Text normalisation, sentence chunking and book loading (txt/md/docx/pdf/epub)."""
from __future__ import annotations

import re
import unicodedata
from functools import lru_cache
from pathlib import Path

# ---------------------------------------------------------------------------
# Bible text cleanup
# ---------------------------------------------------------------------------
_BRACKETED_NOTE = re.compile(r"\[[^\]]{0,80}\]")          # [footnote], [a]
_MARKUP = re.compile(r"[*_#`~|\\<>{}]")
_SPACES = re.compile(r"\s+")
# Hebrew letters: the acrostic stanza headings of Psalm 119 (e.g. "צ") in several Bibles. They are
# never spoken in our audio, and in a transcript they add bogus characters to the ASR vocabulary.
_FOREIGN_SCRIPT = re.compile(r"[֐-׿יִ-ﭏ]+")


def clean_bible_text(text: str) -> str:
    """Remove footnote brackets / markup artefacts that are not read aloud."""
    text = unicodedata.normalize("NFC", text or "")
    text = _BRACKETED_NOTE.sub(" ", text)
    text = _MARKUP.sub(" ", text)
    text = text.replace(" ", " ").replace("​", "")
    return _SPACES.sub(" ", text).strip()


# ---------------------------------------------------------------------------
# ASR / TTS normalisation
# ---------------------------------------------------------------------------
# Apostrophe-like characters are meaningful in many of these orthographies
# (e.g. Rendille, Somali, Oromo use ' as a glottal stop / length marker).
_APOSTROPHES = "'’‘ʼʻ`´"


def normalize_transcript(text: str) -> str:
    """Lowercase, unify apostrophes, drop punctuation/digits -> what ASR should predict."""
    text = unicodedata.normalize("NFC", text or "").lower()
    for a in _APOSTROPHES:
        text = text.replace(a, "'")
    text = _FOREIGN_SCRIPT.sub(" ", text)
    out = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat.startswith("M") or ch == "'":
            out.append(ch)
        else:
            out.append(" ")
    text = "".join(out)
    text = re.sub(r"(^|\s)'+|'+(\s|$)", " ", text)   # stray quote marks, keep word-internal '
    return _SPACES.sub(" ", text).strip()


def normalize_for_tts(text: str) -> str:
    """Keep punctuation (it drives prosody), drop markup, collapse whitespace."""
    text = unicodedata.normalize("NFC", text or "")
    for a in _APOSTROPHES[1:]:
        text = text.replace(a, "'")
    text = _FOREIGN_SCRIPT.sub(" ", text)
    text = _MARKUP.sub(" ", text)
    return _SPACES.sub(" ", text).strip()


@lru_cache(maxsize=1)
def _uroman():
    import uroman as ur
    return ur.Uroman()


def romanize(text: str) -> str:
    """Universal romanisation (uroman) -- used by the forced aligner and uroman TTS vocabularies."""
    return _uroman().romanize_string(text)


def char_inventory(texts) -> dict[str, int]:
    counts: dict[str, int] = {}
    for t in texts:
        for ch in t:
            counts[ch] = counts.get(ch, 0) + 1
    return dict(sorted(counts.items(), key=lambda kv: -kv[1]))


# ---------------------------------------------------------------------------
# Book chunking (English source books)
# ---------------------------------------------------------------------------
_SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?”\"'])\s+(?=[A-Z0-9“\"'(])")
_ABBREV = re.compile(r"\b(Mr|Mrs|Ms|Dr|St|Prof|Sr|Jr|vs|etc|e\.g|i\.e|No|Gen|Ex|Rev)\.$")


def split_into_paragraphs(text: str) -> list[str]:
    paras = re.split(r"\n\s*\n", text.replace("\r\n", "\n").strip())
    return [_SPACES.sub(" ", p).strip() for p in paras if p.strip()]


def split_into_sentences(paragraph: str) -> list[str]:
    """Regex sentence splitter for English that does not break after common abbreviations."""
    raw = _SENTENCE_BOUNDARY.split(paragraph.strip())
    out: list[str] = []
    for piece in raw:
        piece = piece.strip()
        if not piece:
            continue
        if out and _ABBREV.search(out[-1]):
            out[-1] = f"{out[-1]} {piece}"
        else:
            out.append(piece)
    return out


def _hard_wrap(sentence: str, max_chars: int) -> list[str]:
    parts = re.split(r"(?<=[,;:—])\s+", sentence)
    chunks: list[str] = []
    cur = ""
    for part in parts:
        while len(part) > max_chars:                       # no clause punctuation at all
            cut = part.rfind(" ", 0, max_chars)
            cut = cut if cut > 0 else max_chars
            if cur:
                chunks.append(cur)
                cur = ""
            chunks.append(part[:cut].strip())
            part = part[cut:].strip()
        cand = f"{cur} {part}".strip()
        if len(cand) > max_chars and cur:
            chunks.append(cur)
            cur = part
        else:
            cur = cand
    if cur:
        chunks.append(cur)
    return [c for c in chunks if c]


def chunk_book(text: str, max_chars_per_unit: int = 350) -> list[dict]:
    """Paragraph/sentence units small enough for NLLB and for natural TTS prosody."""
    units: list[dict] = []
    for p_id, para in enumerate(split_into_paragraphs(text)):
        flat: list[str] = []
        for s in split_into_sentences(para):
            flat.extend([s] if len(s) <= max_chars_per_unit else _hard_wrap(s, max_chars_per_unit))
        for s_id, sent in enumerate(flat):
            units.append({"paragraph_id": p_id, "sentence_id": s_id, "text": sent,
                          "is_paragraph_end": s_id == len(flat) - 1})
    return units


# ---------------------------------------------------------------------------
# Book loading
# ---------------------------------------------------------------------------
_PAGE_REF = re.compile(r"\{[A-Za-z0-9]{1,8} [0-9]{1,4}\.[0-9]{1,3}\}")      # EGW paragraph refs {GC 12.3}
_DIGITS = re.compile(r"\d+")
_ROMAN = re.compile(r"^[ivxlcdm]+$", re.I)
_PARA_END = re.compile("[.!?:”\"')]$")


def clean_pdf_pages(pages: list[str]) -> str:
    """Book text from PDF pages: drop running headers/footers and page numbers, rebuild paragraphs.

    Headers ("iv True Revival", "Contents v", "Preface vii") repeat with only the page number
    changing, so lines near the page edge whose digit/roman-stripped form recurs on many pages are
    removed. Lines are then re-joined into paragraphs: a paragraph ends at a line that finishes a
    sentence and is clearly shorter than a full line.
    """
    from collections import Counter

    def key(line: str) -> str:
        words = [w for w in _DIGITS.sub("#", line.strip().lower()).split() if not _ROMAN.match(w)]
        return " ".join(words)

    pages_lines = [[ln.strip() for ln in pg.splitlines() if ln.strip()] for pg in pages]
    edge: Counter = Counter()
    for lines in pages_lines:
        for ln in lines[:2] + lines[-2:]:
            edge[key(ln)] += 1
    min_repeat = max(3, len(pages) // 10)
    headers = {k for k, n in edge.items() if n >= min_repeat and len(k) < 60}
    body: list[str] = []
    for lines in pages_lines:
        for idx, ln in enumerate(lines):
            at_edge = idx < 2 or idx >= len(lines) - 2
            if at_edge and (key(ln) in headers or _DIGITS.fullmatch(ln) or _ROMAN.fullmatch(ln)):
                continue
            ln = _PAGE_REF.sub("", ln).strip()
            if ln:
                body.append(ln)
    if not body:
        return ""
    lengths = sorted(len(ln) for ln in body)
    typical = lengths[int(len(lengths) * 0.75)]
    paras: list[str] = []
    cur: list[str] = []
    for ln in body:
        if cur and cur[-1].endswith("-") and ln[:1].islower():
            cur[-1] = cur[-1][:-1] + ln                      # de-hyphenate across the line break
        else:
            cur.append(ln)
        if _PARA_END.search(ln) and len(ln) < 0.9 * typical:
            paras.append(" ".join(cur))
            cur = []
    if cur:
        paras.append(" ".join(cur))
    return "\n\n".join(_SPACES.sub(" ", p).strip() for p in paras if p.strip())


def load_book_text(path: str | Path) -> str:
    """Plain text from .txt/.md/.docx/.pdf/.epub (paragraphs separated by blank lines)."""
    p = Path(path)
    ext = p.suffix.lower()
    if ext in (".txt", ".md", ""):
        return p.read_text(encoding="utf-8")
    if ext == ".docx":
        import docx  # python-docx
        return "\n\n".join(par.text for par in docx.Document(str(p)).paragraphs if par.text.strip())
    if ext == ".pdf":
        from pypdf import PdfReader
        return clean_pdf_pages([pg.extract_text() or "" for pg in PdfReader(str(p)).pages])
    if ext == ".epub":
        import zipfile
        from html import unescape
        paras: list[str] = []
        with zipfile.ZipFile(p) as z:
            for name in sorted(n for n in z.namelist() if n.endswith((".xhtml", ".html", ".htm"))):
                html = z.read(name).decode("utf-8", "ignore")
                for block in re.findall(r"<p[^>]*>(.*?)</p>", html, flags=re.S | re.I):
                    t = unescape(re.sub(r"<[^>]+>", "", block)).strip()
                    if t:
                        paras.append(t)
        return "\n\n".join(paras)
    raise ValueError(f"Unsupported book format: {ext} (use .txt, .md, .docx, .pdf or .epub)")

In [ ]:
%%writefile /tmp/btp/btp/tracking.py
"""Experiment tracking (Weights & Biases) with safe fallbacks.

* Online when WANDB_API_KEY is set; offline into <workspace>/wandb otherwise
  (``wandb sync`` uploads those runs later -- see the notebook's Monitoring section).
* Tracking failures never stop a training run: every call is wrapped.
* Runs are grouped by stage (job_type) and language so the W&B workspace can be
  filtered per language / per stage for daily monitoring.
"""
from __future__ import annotations

import logging
import os
from pathlib import Path
from typing import Any

import numpy as np

log = logging.getLogger(__name__)


class Tracker:
    def __init__(self, cfg: dict, *, stage: str, name: str, lang: str | None = None,
                 config: dict | None = None, workspace: str | Path | None = None, tags: list[str] | None = None):
        self.run = None
        self.enabled = cfg.get("tracking", {}).get("backend", "wandb") == "wandb"
        if not self.enabled:
            return
        try:
            import wandb
        except ImportError:
            log.warning("wandb not installed -- tracking disabled")
            self.enabled = False
            return
        mode = "online"
        if not os.environ.get("WANDB_API_KEY"):
            if cfg["tracking"].get("offline_if_no_key", True):
                mode = "offline"
                log.warning("WANDB_API_KEY not set -- logging offline (sync later with `wandb sync`)")
            else:
                self.enabled = False
                return
        wdir = Path(workspace) if workspace else None
        if wdir is not None:
            wdir.mkdir(parents=True, exist_ok=True)
        try:
            self.run = wandb.init(
                project=os.environ.get("WANDB_PROJECT", cfg["tracking"]["project"]),
                name=name, group=lang or "all", job_type=stage, mode=mode,
                dir=str(wdir) if wdir else None, config=config or {},
                tags=[t for t in [stage, lang, *(tags or [])] if t], reinit=True,
                settings=wandb.Settings(start_method="thread", _disable_stats=False),
            )
            self._wandb = wandb
            log.info("W&B run: %s (%s)", getattr(self.run, "url", None) or name, mode)
        except Exception as e:  # noqa: BLE001
            log.warning("wandb.init failed (%s) -- tracking disabled for this run", e)
            self.run = None

    # ------------------------------------------------------------------
    @property
    def url(self) -> str | None:
        return getattr(self.run, "url", None) if self.run else None

    def log(self, data: dict[str, Any], step: int | None = None) -> None:
        if not self.run:
            return
        try:
            self.run.log(data, step=step)
        except Exception as e:  # noqa: BLE001
            log.warning("wandb.log failed: %s", e)

    def summary(self, data: dict[str, Any]) -> None:
        if not self.run:
            return
        try:
            for k, v in data.items():
                self.run.summary[k] = v
        except Exception as e:  # noqa: BLE001
            log.warning("wandb summary failed: %s", e)

    def audio(self, key: str, clips: list[np.ndarray], captions: list[str], sr: int, step: int | None = None) -> None:
        if not self.run:
            return
        try:
            self.run.log({key: [self._wandb.Audio(np.asarray(c, dtype=np.float32), sample_rate=sr, caption=cap[:200])
                                for c, cap in zip(clips, captions)]}, step=step)
        except Exception as e:  # noqa: BLE001
            log.warning("wandb audio failed: %s", e)

    def table(self, key: str, columns: list[str], rows: list[list[Any]], step: int | None = None) -> None:
        if not self.run:
            return
        try:
            self.run.log({key: self._wandb.Table(columns=columns, data=rows)}, step=step)
        except Exception as e:  # noqa: BLE001
            log.warning("wandb table failed: %s", e)

    def finish(self) -> None:
        if self.run:
            try:
                self.run.finish()
            except Exception as e:  # noqa: BLE001
                log.warning("wandb finish failed: %s", e)
            self.run = None

In [ ]:
%%writefile /tmp/btp/btp/tts.py
"""TTS: fine-tune MMS-TTS (VITS) voices on the aligned Bible narration.

Model code: the MIT-licensed training classes from ylacombe/finetune-hf-vits
(third_party/finetune_hf_vits). This module replaces that project's training script
with a loop that works with current transformers/datasets releases:
  * data comes straight from our parquet clips (no datasets.Audio decoding, which now
    needs torchcodec);
  * monotonic alignment search is btp.mas (no Cython build step);
  * checkpoints (generator + discriminator + optimisers) go to Drive and resume;
  * W&B gets losses plus audio of fixed held-out sentences every eval.

Languages without an MMS-TTS voice (Luo, Kamba, pipeline-B languages) start from their
bridge language's voice; characters the bridge vocabulary lacks are added to the text
embedding (initialised from the base letter of the same character, e.g. 'ĩ' <- 'i').
"""
from __future__ import annotations

import json
import logging
import shutil
import time
import unicodedata
from pathlib import Path

import numpy as np

from btp import audio as A
from btp.text import normalize_for_tts

log = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Losses (VITS; same definitions as the reference recipe)
# ---------------------------------------------------------------------------
def discriminator_loss(real_outputs, fake_outputs):
    import torch
    loss, real_l, fake_l = 0.0, 0.0, 0.0
    for dr, dg in zip(real_outputs, fake_outputs):
        r = torch.mean((1 - dr.float()) ** 2)
        g = torch.mean(dg.float() ** 2)
        loss, real_l, fake_l = loss + r + g, real_l + r, fake_l + g
    return loss, real_l, fake_l


def feature_loss(fmaps_real, fmaps_fake):
    import torch
    loss = 0.0
    for fr, fg in zip(fmaps_real, fmaps_fake):
        for r, g in zip(fr, fg):
            loss = loss + torch.mean(torch.abs(r.float().detach() - g.float()))
    return loss * 2


def generator_loss(disc_outputs):
    import torch
    return sum(torch.mean((1 - d.float()) ** 2) for d in disc_outputs)


def kl_loss(z_p, logs_q, m_p, logs_p, mask):
    import torch
    z_p, logs_q, m_p, logs_p = z_p.float(), logs_q.float(), m_p.float(), logs_p.float()
    kl = logs_p - logs_q - 0.5 + 0.5 * ((z_p - m_p) ** 2) * torch.exp(-2.0 * logs_p)
    return torch.sum(kl * mask) / torch.sum(mask)


# ---------------------------------------------------------------------------
# Checkpoint preparation
# ---------------------------------------------------------------------------
def _base_char(ch: str) -> str:
    d = unicodedata.normalize("NFD", ch)
    return d[0] if d else ch


def prepare_init_checkpoint(lang, texts: list[str], out_dir: Path) -> dict:
    """facebook/mms-tts-<start> + its discriminator -> trainable checkpoint in ``out_dir``.

    Adds characters used in ``texts`` that the start vocabulary lacks.
    """
    import torch
    from transformers import VitsTokenizer

    from third_party.finetune_hf_vits import VitsModelForPreTraining
    from third_party.finetune_hf_vits.convert_discriminator import convert_checkpoint

    start = lang.tts_start
    if not start:
        raise RuntimeError(f"{lang.code}: no MMS-TTS voice or bridge configured")
    if (out_dir / "btp_tts_init.json").is_file():
        return json.loads((out_dir / "btp_tts_init.json").read_text(encoding="utf-8"))
    out_dir.mkdir(parents=True, exist_ok=True)
    convert_checkpoint(language_code=start, pytorch_dump_folder_path=str(out_dir))

    tok = VitsTokenizer.from_pretrained(str(out_dir))
    vocab = dict(tok.get_vocab())
    used = set()
    for t in texts:
        s = t.lower() if getattr(tok, "normalize", True) else t
        used.update(ch for ch in s if not ch.isspace())
    if tok.is_uroman:
        missing = []          # uroman output is plain ASCII -> always covered
    else:
        missing = sorted(ch for ch in used if ch not in vocab and (ch.isalpha() or ch == "'"))
    info = {"lang": lang.code, "start": start, "bootstrap": lang.tts_needs_bootstrap, "added_chars": missing,
            "is_uroman": bool(tok.is_uroman)}
    if missing:
        model = VitsModelForPreTraining.from_pretrained(str(out_dir))
        emb = model.text_encoder.embed_tokens
        old_n = emb.weight.shape[0]
        new_vocab = dict(vocab)
        for ch in missing:
            new_vocab[ch] = max(new_vocab.values()) + 1
        new_emb = torch.nn.Embedding(max(new_vocab.values()) + 1, emb.weight.shape[1])
        with torch.no_grad():
            new_emb.weight[:old_n] = emb.weight
            for ch in missing:
                base = _base_char(ch)
                src = vocab.get(base)
                new_emb.weight[new_vocab[ch]] = emb.weight[src] if src is not None else emb.weight.mean(0)
        model.text_encoder.embed_tokens = new_emb
        model.config.vocab_size = new_emb.weight.shape[0]
        model.save_pretrained(str(out_dir))
        with open(out_dir / "vocab.json", "w", encoding="utf-8") as f:
            json.dump(new_vocab, f, ensure_ascii=False)
        log.info("%s TTS: added %d characters to the %s vocabulary: %s", lang.code, len(missing), start,
                 "".join(missing))
    (out_dir / "btp_tts_init.json").write_text(json.dumps(info, indent=1, ensure_ascii=False), encoding="utf-8")
    return info


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
class TTSDataset:
    def __init__(self, rows: list[dict], tokenizer, fe, max_sec: float, max_tokens: int = 500):
        self.rows, self.tok, self.fe = rows, tokenizer, fe
        self.max_samples = int(max_sec * A.SR)
        self.max_tokens = max_tokens

    def __len__(self) -> int:
        return len(self.rows)

    def text_ids(self, text: str) -> list[int]:
        from btp.text import romanize
        s = normalize_for_tts(text)
        if self.tok.is_uroman:
            s = romanize(s)
        return self.tok(s).input_ids[: self.max_tokens]

    def __getitem__(self, i: int) -> dict:
        r = self.rows[i]
        wav = A.decode_audio_bytes(r["audio"])[: self.max_samples]
        f = self.fe(wav, sampling_rate=A.SR, return_attention_mask=False, do_normalize=False)
        return {"input_ids": self.text_ids(r["text"]), "waveform": wav,
                "labels": np.asarray(f["input_features"][0], dtype=np.float32),
                "mel": np.asarray(f["mel_scaled_input_features"][0], dtype=np.float32)}


def _feat_time_major(x: np.ndarray, n_feat: int) -> np.ndarray:
    """Return features as [T, n_feat] whatever orientation the extractor produced."""
    x = np.squeeze(x)
    if x.ndim != 2:
        raise ValueError(f"unexpected feature shape {x.shape}")
    return x if x.shape[1] == n_feat else x.T


def collate_tts(batch: list[dict], pad_id: int, n_freq: int, n_mel: int) -> dict:
    """Pad to the layouts VitsModelForPreTraining expects:
    input_ids [B,L], attention_mask [B,L], labels [B,n_freq,T], labels_attention_mask [B,T],
    mel_scaled_input_features [B,n_mel,T], waveform [B,1,S]."""
    import torch
    B = len(batch)
    L = max(len(b["input_ids"]) for b in batch)
    lab = [_feat_time_major(b["labels"], n_freq) for b in batch]
    mel = [_feat_time_major(b["mel"], n_mel) for b in batch]
    T = max(x.shape[0] for x in lab)
    S = max(len(b["waveform"]) for b in batch)
    ids = torch.full((B, L), pad_id, dtype=torch.long)
    am = torch.zeros((B, L), dtype=torch.long)
    labels = torch.zeros((B, n_freq, T))
    lam = torch.zeros((B, T), dtype=torch.long)
    mels = torch.zeros((B, n_mel, T))
    wav = torch.zeros((B, 1, S))
    for i, b in enumerate(batch):
        ids[i, : len(b["input_ids"])] = torch.tensor(b["input_ids"])
        am[i, : len(b["input_ids"])] = 1
        labels[i, :, : lab[i].shape[0]] = torch.from_numpy(lab[i].T)
        lam[i, : lab[i].shape[0]] = 1
        mels[i, :, : mel[i].shape[0]] = torch.from_numpy(mel[i].T)
        wav[i, 0, : len(b["waveform"])] = torch.from_numpy(b["waveform"])
    return {"input_ids": ids, "attention_mask": am, "labels": labels, "labels_attention_mask": lam,
            "mel_scaled_input_features": mels, "waveform": wav}


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def _apply_generator_weight_norm(model) -> None:
    import torch
    model.decoder.apply_weight_norm()
    for flow in model.flow.flows:
        torch.nn.utils.weight_norm(flow.conv_pre)
        torch.nn.utils.weight_norm(flow.conv_post)


def _apply_weight_norm(model, discriminator) -> None:
    _apply_generator_weight_norm(model)
    for d in discriminator.discriminators:
        d.apply_weight_norm()


def _remove_weight_norm(model) -> None:
    import torch
    model.decoder.remove_weight_norm()
    for flow in model.flow.flows:
        torch.nn.utils.remove_weight_norm(flow.conv_pre)
        torch.nn.utils.remove_weight_norm(flow.conv_post)


def export_inference_model(model, out_dir: Path, init_dir: Path, meta: dict | None = None) -> None:
    """Save a plain transformers VitsModel (+ tokenizer) that loads with VitsModel.from_pretrained."""
    from transformers import VitsConfig, VitsModel

    from third_party.finetune_hf_vits import VitsModelForPreTraining

    # Never deepcopy the live training model: after a training forward pass its weight-norm
    # weights are non-leaf tensors and deepcopy raises (this crashed exports at arbitrary steps,
    # e.g. the Kaggle deadline save). Rebuild a fresh generator from detached weights instead.
    weights = {k: v.detach().cpu().float().clone() for k, v in model.state_dict().items()}
    gen = VitsModelForPreTraining(model.config)
    if hasattr(gen, "discriminator"):
        del gen.discriminator
    _apply_generator_weight_norm(gen)
    missing_gen, _ = gen.load_state_dict(weights, strict=False)
    missing_gen = [k for k in missing_gen if not k.endswith("num_batches_tracked")]
    if missing_gen:
        raise RuntimeError(f"export: training weights missing for {missing_gen[:5]}")
    _remove_weight_norm(gen)
    cfg = VitsConfig.from_pretrained(str(init_dir))
    cfg.vocab_size = gen.text_encoder.embed_tokens.weight.shape[0]
    inf = VitsModel(cfg)
    sd = {k: v for k, v in gen.state_dict().items() if not k.startswith("discriminator.")}
    missing, unexpected = inf.load_state_dict(sd, strict=False)
    real_missing = [k for k in missing if not k.endswith("num_batches_tracked")]
    if real_missing:
        raise RuntimeError(f"export: generator weights missing for {real_missing[:5]}")
    from btp.utils import atomic_dir
    with atomic_dir(out_dir) as tmp:
        inf.save_pretrained(str(tmp))
        for f in ("vocab.json", "tokenizer_config.json", "special_tokens_map.json", "preprocessor_config.json"):
            if (init_dir / f).is_file():
                shutil.copy(init_dir / f, tmp / f)
        if meta is not None:
            (tmp / "btp_tts.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")


def held_out_sentences(rows: list[dict], n: int) -> list[str]:
    seen, out = set(), []
    for r in sorted(rows, key=lambda r: r["id"]):
        t = normalize_for_tts(r["text"])
        if 20 <= len(t) <= 160 and t not in seen:
            seen.add(t)
            out.append(t)
        if len(out) >= n:
            break
    return out


def train_tts(lang, cfg: dict, paths, tracker=None, device: str | None = None, max_steps: int | None = None,
              resume: bool = True) -> dict:
    import torch
    from transformers import VitsTokenizer

    from btp.mas import maximum_path
    from btp.speechdata import load_metadata, materialize, read_materialized, select_clips
    from btp.utils import atomic_torch_save, deadline_passed, remove_checkpoint, safe_torch_load, seed_everything
    from btp.utils import device as auto_device
    from third_party.finetune_hf_vits import (
        VitsDiscriminator,
        VitsFeatureExtractor,
        VitsModelForPreTraining,
        slice_segments,
    )

    seed_everything()
    device = device or auto_device()
    c = cfg["tts"]
    max_steps = max_steps or c["max_steps"]
    out_dir = paths.tts_dir(lang.code)
    init_dir = out_dir / "init"
    ckpt = out_dir / "checkpoint.pt"
    from btp.utils import mark_training_done, too_late_to_start, training_done
    if resume and (prev := training_done(paths, f"tts_{lang.code}", max_steps)) is not None:
        log.info("%s TTS already trained to %d steps -- skipped", lang.code, max_steps)
        return prev
    if too_late_to_start(c.get("min_minutes_to_start", 25)):
        log.warning("%s TTS: too little time before the deadline -- deferred to the next run", lang.code)
        return {"lang": lang.code, "deferred": "deadline"}

    # data ---------------------------------------------------------------
    data_dir = paths.cache / "tts" / lang.code
    if not (data_dir / "train.parquet").is_file():
        meta = load_metadata(paths, lang.code)
        splits = select_clips(meta, cfg, task="tts", hours_cap=c["hours_per_lang"], max_sec=c["max_clip_sec"],
                              voice_set=c.get("voice_set"), prefer_style=c.get("prefer_style"))
        if not splits["train"]:
            raise RuntimeError(f"No TTS clips for {lang.code} -- run align/pseudo stages first")
        for k, rows in splits.items():
            materialize(paths, rows if k == "train" else rows[:200], data_dir / f"{k}.parquet")
    train_rows = read_materialized(data_dir / "train.parquet")
    dev_rows = read_materialized(data_dir / "dev.parquet")
    if not train_rows:
        raise RuntimeError(f"No TTS clips for {lang.code} -- run align/pseudo stages first")
    log.info("%s TTS data: %d clips (%.2f h) from %s", lang.code, len(train_rows),
             sum(r["duration"] for r in train_rows) / 3600, sorted({r['audio_set'] for r in train_rows}))

    info = prepare_init_checkpoint(lang, [r["text"] for r in train_rows], init_dir)
    tok = VitsTokenizer.from_pretrained(str(init_dir))
    fe = VitsFeatureExtractor(sampling_rate=A.SR, feature_size=80)
    model = VitsModelForPreTraining.from_pretrained(str(init_dir))
    # Train the discriminator as a separate module with its own optimiser. (The reference
    # recipe round-trips it through save_pretrained, which transformers 5 rejects for a
    # sub-module that never ran post_init -- and the round trip is not needed.)
    disc = model.discriminator
    del model.discriminator
    assert isinstance(disc, VitsDiscriminator)
    warm_step = 0
    if resume and not ckpt.exists() and not ckpt.with_name(ckpt.name + ".prev").exists():
        # Training finished earlier and its checkpoint was pruned; max_steps has since been raised.
        from transformers import VitsModel

        from btp.utils import read_json, resolve_dir
        rec = read_json(paths.status() / f"train_tts_{lang.code}.json", None)
        fin = resolve_dir(out_dir / "final")
        if rec and fin is not None and (fin / "config.json").is_file():
            missing, _ = model.load_state_dict(VitsModel.from_pretrained(str(fin)).state_dict(), strict=False)
            warm_step = int(rec.get("steps", 0))
            log.info("%s TTS checkpoint was pruned -- continuing from final/ at step %d (%d params not in final)",
                     lang.code, warm_step, len(missing))
    _apply_weight_norm(model, disc)
    model.to(device)
    disc.to(device)

    betas = (0.8, 0.99)
    g_opt = torch.optim.AdamW(model.parameters(), c["lr"], betas=betas, eps=1e-8)
    d_opt = torch.optim.AdamW(disc.parameters(), c["lr"], betas=betas, eps=1e-8)
    use_amp = c.get("fp16", True) and device.startswith("cuda")
    g_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    d_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    step = warm_step
    st, _src = safe_torch_load(ckpt) if resume else (None, None)
    if st is not None:
        model.load_state_dict(st["model"])
        disc.load_state_dict(st["disc"])
        g_opt.load_state_dict(st["g_opt"])
        d_opt.load_state_dict(st["d_opt"])
        step = st["step"]
        log.info("resumed TTS %s from step %d", lang.code, step)

    ds = TTSDataset(train_rows, tok, fe, c["max_clip_sec"])
    n_freq = fe.n_fft // 2 + 1 if hasattr(fe, "n_fft") else 513
    n_mel = fe.feature_size
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
    loader = torch.utils.data.DataLoader(ds, batch_size=c["per_device_batch"], shuffle=True, num_workers=2,
                                         drop_last=True, collate_fn=lambda b: collate_tts(b, pad_id, n_freq, n_mel),
                                         generator=torch.Generator().manual_seed(step))
    w = {"disc": 3.0, "gen": 1.0, "fmaps": 1.0, "mel": 35.0, "kl": 1.5, "duration": 1.0}
    w.update(c.get("loss_weights", {}))
    eval_texts = held_out_sentences(dev_rows, c["eval_sentences"])
    seg = model.segment_size
    cfg_seg = model.config.segment_size
    hop = fe.hop_length
    t0 = time.time()
    last = {}
    stopped_for_deadline = False
    while step < max_steps and not stopped_for_deadline:
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            model.train()
            disc.train()
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                            labels=batch["labels"], labels_attention_mask=batch["labels_attention_mask"],
                            speaker_id=None, return_dict=True, monotonic_alignment_function=maximum_path)
            mel_target = slice_segments(batch["mel_scaled_input_features"], out.ids_slice, seg)
            mel_gen = fe._torch_extract_fbank_features(out.waveform.squeeze(1).float())[1]
            wav_target = slice_segments(batch["waveform"], out.ids_slice * hop, cfg_seg)

            # discriminator step
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                d_real, _ = disc(wav_target)
                d_fake, _ = disc(out.waveform.detach())
            loss_d, loss_dr, loss_df = discriminator_loss(d_real, d_fake)
            d_scaler.scale(loss_d * w["disc"]).backward()
            d_scaler.unscale_(d_opt)
            torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
            d_scaler.step(d_opt)
            d_scaler.update()
            d_opt.zero_grad(set_to_none=True)

            # generator step
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                _, fm_real = disc(wav_target)
                d_fake2, fm_fake = disc(out.waveform)
            loss_dur = torch.sum(out.log_duration.float())
            loss_mel = torch.nn.functional.l1_loss(mel_target.float(), mel_gen.float())
            loss_kl = kl_loss(out.prior_latents, out.posterior_log_variances, out.prior_means,
                              out.prior_log_variances, out.labels_padding_mask)
            loss_fm = feature_loss(fm_real, fm_fake)
            loss_g = generator_loss(d_fake2)
            total = (loss_dur * w["duration"] + loss_mel * w["mel"] + loss_kl * w["kl"] + loss_fm * w["fmaps"]
                     + loss_g * w["gen"])
            if not torch.isfinite(total):
                log.warning("non-finite generator loss at step %d -- skipped", step)
                g_opt.zero_grad(set_to_none=True)
                continue
            g_scaler.scale(total).backward()
            g_scaler.unscale_(g_opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            g_scaler.step(g_opt)
            g_scaler.update()
            g_opt.zero_grad(set_to_none=True)
            step += 1
            last = {"train/loss_total": float(total), "train/loss_mel": float(loss_mel), "train/loss_kl": float(loss_kl),
                    "train/loss_duration": float(loss_dur), "train/loss_fmaps": float(loss_fm),
                    "train/loss_gen": float(loss_g), "train/loss_disc": float(loss_d),
                    "train/loss_disc_real": float(loss_dr), "train/loss_disc_fake": float(loss_df)}
            if tracker and step % 10 == 0:
                tracker.log({**last, "train/sec_per_step": (time.time() - t0) / 10}, step=step)
                t0 = time.time()
            if step % c["eval_steps"] == 0 or step == max_steps:
                clips = synthesize_with(model, tok, eval_texts[:4], device)
                if tracker and clips:
                    tracker.audio("dev/samples", clips, eval_texts[:4], A.SR, step=step)
            out_of_time = deadline_passed(margin_sec=900)
            if step % c["save_steps"] == 0 or step == max_steps or out_of_time:
                atomic_torch_save({"model": model.state_dict(), "disc": disc.state_dict(),
                                   "g_opt": g_opt.state_dict(), "d_opt": d_opt.state_dict(), "step": step}, ckpt)
                export_inference_model(model, out_dir / "final", init_dir)
                log.info("%s TTS step %d saved (mel L1 %.3f)", lang.code, step, float(loss_mel))
            if out_of_time and step < max_steps:
                log.warning("%s: deadline near -- TTS stopped at step %d (resume next run)", lang.code, step)
                stopped_for_deadline = True
            if step >= max_steps or stopped_for_deadline:
                break
    export_inference_model(model, out_dir / "final", init_dir,
                           meta={**info, "steps": step, "last": last})
    result = {"lang": lang.code, "steps": step, **{k: round(v, 4) for k, v in last.items()}, "init": info}
    if step >= max_steps and not stopped_for_deadline:
        mark_training_done(paths, f"tts_{lang.code}", step, result)   # recorded BEFORE pruning
        if c.get("prune_checkpoint_when_done") and not deadline_passed():
            remove_checkpoint(ckpt)                      # finished: keep only final/ (storage)
            log.info("%s TTS finished -- training checkpoint removed to save storage (final/ kept)", lang.code)
    return result


def synthesize_with(model, tok, texts: list[str], device: str, noise_scale: float = 0.667,
                    speaking_rate: float = 1.0) -> list[np.ndarray]:
    """Synthesise with a training model (VitsModelForPreTraining) or an exported VitsModel."""
    import torch

    from btp.text import romanize
    out = []
    was_training = model.training
    model.eval()
    for t in texts:
        s = normalize_for_tts(t)
        if tok.is_uroman:
            s = romanize(s)
        ids = tok(s, return_tensors="pt").input_ids.to(device)
        if ids.shape[1] == 0:
            out.append(np.zeros(1, dtype=np.float32))
            continue
        with torch.inference_mode():
            if hasattr(model, "infer"):
                wav = model.infer(input_ids=ids, noise_scale=noise_scale, speaking_rate=speaking_rate).waveform
            else:
                model.noise_scale, model.speaking_rate = noise_scale, speaking_rate
                wav = model(input_ids=ids).waveform
        out.append(wav.squeeze().float().cpu().numpy())
    if was_training:
        model.train()
    return out


def load_tts(model_dir: str | Path, device: str = "cpu"):
    from transformers import VitsModel, VitsTokenizer
    return VitsModel.from_pretrained(str(model_dir)).to(device).eval(), VitsTokenizer.from_pretrained(str(model_dir))


def tts_model_dir(paths, lang) -> str:
    """Fine-tuned voice if it exists, else the off-the-shelf MMS voice (or bridge voice)."""
    from btp.utils import resolve_dir
    final = resolve_dir(paths.tts_dir(lang.code) / "final")
    if final is not None and (final / "config.json").is_file():
        return str(final)
    if lang.tts_start:
        return f"facebook/mms-tts-{lang.tts_start}"
    raise RuntimeError(f"{lang.code}: no TTS voice available")

In [ ]:
%%writefile /tmp/btp/btp/utils.py
"""Small shared helpers: logging to Drive, resumable status files, retries, timing."""
from __future__ import annotations

import json
import logging
import os
import sys
import time
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, TypeVar

T = TypeVar("T")
log = logging.getLogger("btp")


def setup_logging(log_dir: str | Path | None = None, name: str = "btp") -> Path | None:
    """Log to stdout and (optionally) to <log_dir>/<name>-<UTC timestamp>.log.

    Writing logs into the Drive workspace is what lets a run be inspected after the
    Colab session is gone (and lets tools/pull_logs.py fetch them remotely).
    """
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s", "%H:%M:%S")
    root = logging.getLogger()
    root.setLevel(logging.INFO)
    for h in list(root.handlers):
        root.removeHandler(h)
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    root.addHandler(sh)
    path = None
    if log_dir:
        Path(log_dir).mkdir(parents=True, exist_ok=True)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
        path = Path(log_dir) / f"{name}-{stamp}.log"
        fh = logging.FileHandler(path, encoding="utf-8")
        fh.setFormatter(fmt)
        root.addHandler(fh)
    for noisy in ("urllib3", "httpx", "filelock", "fsspec", "datasets", "huggingface_hub"):
        logging.getLogger(noisy).setLevel(logging.WARNING)
    return path


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def read_json(path: str | Path, default: Any = None) -> Any:
    p = Path(path)
    if not p.is_file():
        return default
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def write_json(path: str | Path, obj: Any) -> None:
    """Atomic JSON write (temp file + rename) so a killed session never leaves half a file."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix(p.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, p)


class Status:
    """Resumable per-task progress record stored as JSON (e.g. done chapters)."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.data = read_json(self.path, default={}) or {}
        self.data.setdefault("done", [])
        self.data.setdefault("failed", {})
        self._done = set(self.data["done"])

    def is_done(self, key: str) -> bool:
        return key in self._done

    def mark_done(self, key: str, **info: Any) -> None:
        if key not in self._done:
            self._done.add(key)
            self.data["done"].append(key)
        self.data["failed"].pop(key, None)
        if info:
            self.data.setdefault("info", {})[key] = info
        self.data["updated"] = now_iso()
        write_json(self.path, self.data)

    def mark_failed(self, key: str, reason: str) -> None:
        self.data["failed"][key] = reason[:500]
        self.data["updated"] = now_iso()
        write_json(self.path, self.data)

    def set(self, key: str, value: Any) -> None:
        self.data[key] = value
        self.data["updated"] = now_iso()
        write_json(self.path, self.data)


def retry(fn: Callable[[], T], what: str, tries: int = 5, base_wait: float = 2.0) -> T:
    for i in range(tries):
        try:
            return fn()
        except Exception as e:  # noqa: BLE001 - network/IO errors of many kinds
            if i == tries - 1:
                raise
            wait = min(60.0, base_wait * 2 ** i)
            log.warning("%s failed (%s: %s); retrying in %.0fs", what, type(e).__name__, str(e)[:200], wait)
            time.sleep(wait)
    raise RuntimeError("unreachable")


@contextmanager
def timed(what: str) -> Iterator[None]:
    t0 = time.time()
    yield
    log.info("%s took %.1fs", what, time.time() - t0)


def atomic_torch_save(obj: Any, path: str | Path, keep_prev: bool | None = None) -> None:
    """torch.save that can never leave a half-written checkpoint.

    Writes <path>.tmp, then renames. The previous checkpoint is kept as <path>.prev so that even a
    corrupted latest file (e.g. a storage hiccup) still leaves something to resume from.
    $BTP_KEEP_PREV=0 (Kaggle: 20 GB output limit) drops the .prev copy -- the tmp+rename alone
    still guarantees a complete file.
    """
    import torch
    if keep_prev is None:
        keep_prev = os.environ.get("BTP_KEEP_PREV", "1") != "0"
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + ".tmp")
    torch.save(obj, tmp)
    prev = p.with_name(p.name + ".prev")
    if keep_prev and p.exists():
        os.replace(p, prev)
    elif not keep_prev and prev.exists():
        prev.unlink()
    os.replace(tmp, p)


def safe_torch_load(path: str | Path) -> tuple[Any, Path | None]:
    """Load <path>, falling back to <path>.prev if the latest checkpoint is missing or unreadable."""
    import torch
    p = Path(path)
    for cand in (p, p.with_name(p.name + ".prev")):
        if cand.is_file():
            try:
                return torch.load(cand, map_location="cpu", weights_only=False), cand
            except Exception as e:  # noqa: BLE001
                log.warning("checkpoint %s unreadable (%s) -- trying the previous one", cand, e)
    return None, None


def remove_checkpoint(path: str | Path) -> None:
    for cand in (Path(path), Path(str(path) + ".prev"), Path(str(path) + ".tmp")):
        if cand.is_file():
            cand.unlink()


def resolve_dir(path: str | Path) -> Path | None:
    """The directory written by atomic_dir, or its .prev if a crash hit between the two renames."""
    p = Path(path)
    if p.is_dir():
        return p
    prev = p.with_name(p.name + ".prev")
    return prev if prev.is_dir() else None


@contextmanager
def atomic_dir(path: str | Path) -> Iterator[Path]:
    """Build a directory (e.g. a saved model) beside its destination, then swap it in.

    Readers either see the complete old directory or the complete new one; if a crash hits
    between the two renames, resolve_dir() finds the old copy under <path>.prev.
    """
    import shutil
    dst = Path(path)
    tmp = dst.with_name(dst.name + ".tmp")
    prev = dst.with_name(dst.name + ".prev")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    yield tmp
    if prev.exists():
        shutil.rmtree(prev)
    if dst.exists():
        os.replace(dst, prev)
    os.replace(tmp, dst)
    shutil.rmtree(prev, ignore_errors=True)


def training_done(paths, name: str, max_steps: int) -> dict | None:
    """Result of a finished training job if it already reached ``max_steps`` (else None).

    Re-running a stage after its training finished must not retrain -- on Kaggle the checkpoint
    may even have been pruned, which would otherwise restart from step 0 and burn GPU hours.
    Raising max_steps later continues training instead of skipping.
    """
    rec = read_json(Path(paths.status()) / f"train_{name}.json", None)
    if rec and rec.get("finished") and int(rec.get("steps", 0)) >= int(max_steps):
        return rec.get("result")
    return None


def mark_training_done(paths, name: str, steps: int, result: dict) -> None:
    write_json(Path(paths.status()) / f"train_{name}.json",
               {"finished": True, "steps": int(steps), "result": result, "updated": now_iso()})


def deadline_passed(margin_sec: float = 0.0) -> bool:
    """True once the wall-clock deadline in $BTP_DEADLINE (unix seconds) is near.

    Kaggle kills a run at 12 h and then saves NO output, so the Kaggle notebook sets a deadline
    (~11 h) and every long loop checks it: save a checkpoint, stop cleanly, resume next run.
    """
    d = os.environ.get("BTP_DEADLINE")
    return bool(d) and time.time() + margin_sec >= float(d)


def too_late_to_start(minutes: float = 20.0) -> bool:
    """True if less than ``minutes`` remain before the deadline: a stage should not even load a model
    for its next language (it would train ~0 steps and only burn the session's remaining time)."""
    return deadline_passed(margin_sec=minutes * 60)


def device() -> str:
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed: int = 1234) -> None:
    import random

    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
%%writefile /tmp/btp/config/glossary.yaml
# Agreed translations of key Spirit of Prophecy terms.
#
# Fill in the target term for each language code (swh, kik, luo, kam, som, orm, saq, spy, enb, rel,
# teo, dig, dug, nyf, pkb, lsm, sxb, ...). Use "|" to list accepted alternatives, e.g. "Sabato|Sabbato".
# Empty = not checked. Review sheets then get a `glossary_flags` column listing rows whose machine
# translation does not contain the agreed term, so reviewers see terminology slips at a glance.
#
# Translations are deliberately left blank: established Adventist terminology differs per language
# and must come from fluent church readers, not from a model.

terms:
  - {en: Sabbath}
  - {en: sanctuary}
  - {en: Most Holy Place}
  - {en: Holy Place}
  - {en: investigative judgment}
  - {en: second coming}
  - {en: Spirit of Prophecy}
  - {en: great controversy}
  - {en: three angels}
  - {en: remnant}
  - {en: Holy Spirit}
  - {en: atonement}
  - {en: justification}
  - {en: sanctification}
  - {en: righteousness by faith}
  - {en: health reform}
  - {en: law of God}
  - {en: Ten Commandments}
  - {en: mark of the beast}
  - {en: seal of God}
  - {en: Satan}
  - {en: Redeemer}
  - {en: salvation}
  - {en: grace}
  - {en: repentance}
  - {en: baptism}
  - {en: prophecy}
  - {en: Scriptures}

In [ ]:
%%writefile /tmp/btp/config/languages.yaml
# languages.yaml -- master registry for the book -> translation -> audio pipelines.
#
# Every field below was checked against the live sources on 2026-11-18:
#   * audio/text set ids: live.bible.is API (audio already copied to Drive under
#     adventproclaimer_training_data/<drive_folder>/<AUDIO_SET> (<style> <part>)/)
#   * mms_asr: adapter exists in facebook/mms-1b-all  (adapter.<code>.safetensors)
#   * mms_tts: facebook/mms-tts-<code> exists AND its discriminator exists in
#     facebook/mms-tts/full_models/<code>/D_100000.pth (needed for VITS fine-tuning)
#   * nllb: the language is one of NLLB-200's 202 codes
#
# Field reference
#   name          human-readable name
#   drive_folder  folder under the raw-audio root on Drive
#   mms_asr       facebook/mms-1b-all adapter code to start ASR from (null = new adapter)
#   mms_tts       facebook/mms-tts-<code> to start TTS from        (null = bootstrap from tts_bridge)
#   tts_bridge    MMS-TTS code to bootstrap from when mms_tts is null
#   asr_bridge    MMS ASR adapter to bootstrap from when mms_asr is null
#   nllb          existing NLLB-200 language code               (null = add a new language token)
#   nllb_init     NLLB code whose embedding seeds the new token (used when nllb is null)
#   sets          audio set -> matching text set. text: null means no matching text exists;
#                 those sets are only usable through ASR pseudo-labelling.
#                 style: plain = single narrator (best for TTS); drama = multi-voice + music.

# ---------------------------------------------------------------------------
# Pipeline A: the 17 target languages (all have MMS ASR; 15 have MMS TTS)
# ---------------------------------------------------------------------------
pipeline_a:
  swh:
    name: Swahili
    drive_folder: Kiswahili
    mms_asr: swh
    mms_tts: swh
    nllb: swh_Latn
    openbible: Swahili          # verse-level audio+text, CC BY-SA (btp.openbible)
    extra_texts: ["ebible:swhulb", "ebible:swhonmm"]   # open versions: translation pairs only
    sets:
      - {audio: SWABIBN1DA, text: SWABIBN_ET, style: plain}
      - {audio: SWABIBO1DA, text: SWABIBO_ET, style: plain}
      - {audio: SWAHNVN2DA, text: SWAHNVN_ET, style: drama}
      - {audio: SWHBTKO1DA, text: null, style: plain}
  kik:
    name: Kikuyu (Gikuyu)
    drive_folder: Kikuyu
    mms_asr: kik
    mms_tts: kik
    nllb: kik_Latn
    openbible: Kikuyu
    extra_texts: ["ebible:kik"]   # Biblica Kikuyu (differs from the KIKKGV recordings): pairs only
    sets:
      - {audio: KIKKGVN1DA, text: KIKKGVN_ET, style: plain}
      - {audio: KIKKGVO1DA, text: KIKKGVO_ET, style: plain}
      - {audio: KIUBSKN2DA, text: KIUBSK, style: drama}
      - {audio: KIUDPIO1DA, text: null, style: plain}
  luo:
    name: Dholuo (Luo)
    drive_folder: Luo
    mms_asr: luo
    mms_tts: null
    tts_bridge: ach          # Acholi: closest Western Nilotic (Lwoo) language with an MMS-TTS voice
    nllb: luo_Latn
    openbible: Dholuo           # Open.Bible verse audio+text (Biblica Open New Luo Translation, CC BY-SA)
    licence: "Text and Open.Bible audio: (c) Biblica, CC BY-SA 4.0 -- attribute; share derivatives alike."
    note: "eBible 'luo' = FCBH LUOBIB; verified 2026-11-19 by alignment (Mark 2: 28/28 verses, median -0.045,
      planted wrong verses -1.1..-2.3)."
    sets:
      - {audio: LUOBIBN1DA, text: "ebible:luo", style: plain}
      - {audio: LUOBIBO1DA, text: "ebible:luo", style: plain}
      - {audio: LUODPIO1DA, text: null, style: plain}
      - {audio: LUOBSKN2DA, text: null, style: drama}
  kam:
    name: Kamba
    drive_folder: Kikamba
    mms_asr: kam
    mms_tts: null
    tts_bridge: kik          # Central Kenya Bantu, closest MMS-TTS voice
    nllb: kam_Latn
    note: "No Bible.is text matches any Kamba recording -> ASR/TTS data comes from MMS pseudo-labels."
    sets:
      - {audio: KAMDPIN1DA, text: null, style: plain}
      - {audio: KAMDPIO1DA, text: null, style: plain}
      - {audio: KIKBSKN2DA, text: null, style: drama}
  som:
    name: Somali
    drive_folder: Somali
    mms_asr: som
    mms_tts: som
    nllb: som_Latn
    sets:
      - {audio: SOMTVIN2DA, text: SOMTVIN_ET, style: drama}
      - {audio: SOMSIMN2DA, text: SOMSIM, style: drama}
      - {audio: SOMSIMO2DA, text: SOMSIM, style: drama}
  orm:
    name: Orma (Oromo)
    drive_folder: Orma
    mms_asr: orm
    mms_tts: orm
    nllb: gaz_Latn
    openbible: Oromo            # West Central Oromo: extra ASR data (not an Orma voice)
    extra_texts: ["ebible:gaz", "ebible:gaze"]
    note: "Recordings are Orma (ISO orc), an Oromo variety; MMS/NLLB only have generic/West Central Oromo."
    sets:
      - {audio: ORCBTLN1DA, text: ORCBTL, style: plain}
      - {audio: ORCBTLN2DA, text: ORCBTL, style: drama}
  saq:
    name: Samburu
    drive_folder: Samburu
    mms_asr: saq
    mms_tts: saq
    nllb: null
    nllb_init: swh_Latn
    sets:
      - {audio: SAQBTLN1DA, text: SAQBTLN_ET, style: plain}
      - {audio: SAQBTLN2DA, text: SAQBTLN_ET, style: drama}
      - {audio: SAQSIVN1DA, text: SAQSIVN_ET, style: plain, variety: Ilchamus, drive_folder: Ilchamus}
      - {audio: SAQSIVN2DA, text: SAQSIVN_ET, style: drama, variety: Ilchamus, drive_folder: Ilchamus}
  spy:
    name: Sabaot
    drive_folder: Sabaot
    mms_asr: spy
    mms_tts: spy
    nllb: null
    nllb_init: luo_Latn
    sets:
      - {audio: SPYWBTN2DA, text: SPYWBTN_ET, style: drama}
  enb:
    name: Marakwet
    drive_folder: Marakwet
    mms_asr: enb
    mms_tts: enb
    nllb: null
    nllb_init: luo_Latn
    sets:
      - {audio: ENBBTLN2DA, text: ENBBTL, style: drama}
  rel:
    name: Rendille
    drive_folder: Rendille
    mms_asr: rel
    mms_tts: rel
    nllb: null
    nllb_init: som_Latn
    sets:
      - {audio: RELBTLN1DA, text: RELBTLN_ET, style: plain}
      - {audio: RELBTLN2DA, text: RELBTLN_ET, style: drama}
  teo:
    name: Teso (Ateso)
    drive_folder: Ateso
    mms_asr: teo
    mms_tts: teo
    nllb: null
    nllb_init: luo_Latn
    sets:
      - {audio: TEOBSUN2DA, text: TEOBSU, style: drama}
      - {audio: TEODPIN1DA, text: null, style: plain}
      - {audio: TEODPIO1DA, text: null, style: plain}
  dig:
    name: Digo
    drive_folder: Digo
    mms_asr: dig
    mms_tts: dig
    nllb: null
    nllb_init: swh_Latn
    sets:
      - {audio: DIGTBLN1DA, text: DIGTBLN_ET, style: plain}
  dug:
    name: Duruma
    drive_folder: Duruma
    mms_asr: dug
    mms_tts: dug
    nllb: null
    nllb_init: swh_Latn
    sets:
      - {audio: DUGBTLN2DA, text: DUGBTLN_ET, style: drama}
  nyf:
    name: Giryama
    drive_folder: Giryama
    mms_asr: nyf
    mms_tts: nyf
    nllb: null
    nllb_init: swh_Latn
    sets:
      - {audio: NYFBTLN2DA, text: NYFBTL, style: drama}
      - {audio: NYFBTRO1DA, text: null, style: plain}
  pkb:
    name: Pokomo
    drive_folder: "Pokomo, Lower"
    mms_asr: pkb
    mms_tts: pkb
    nllb: null
    nllb_init: swh_Latn
    sets:
      - {audio: PKBBTLN2DA, text: PKBBTL, style: drama, variety: Lower Pokomo}
      - {audio: PKBTBLN1DA, text: null, style: plain, variety: Upper Pokomo, drive_folder: "Pokomo, Upper"}
      - {audio: PKBTBLN2DA, text: null, style: drama, variety: Upper Pokomo, drive_folder: "Pokomo, Upper"}
  lsm:
    name: Saamia (Luhya)
    drive_folder: Lusamya-Lugwe
    mms_asr: lsm
    mms_tts: lsm
    nllb: null
    nllb_init: lug_Latn
    sets:
      - {audio: LSMBSUN2DA, text: LSMBSUN_ET, style: drama}
      - {audio: LSMDPIN1DA, text: null, style: plain}
      - {audio: LSMDPIO1DA, text: null, style: plain}
  sxb:
    name: Suba
    drive_folder: Suba
    mms_asr: sxb
    mms_tts: sxb
    nllb: null
    nllb_init: lug_Latn
    sets:
      - {audio: SXBBTLN1DA, text: SXBBTLN_ET, style: plain}
      - {audio: SXBBTLN2DA, text: SXBBTLN_ET, style: drama}

# ---------------------------------------------------------------------------
# Pipeline B: languages not in the 17 above. Audio for all of these is on Drive.
# Grouped by what data actually exists (checked 2026-11-18):
#   tier "text"       matching Bible text exists -> full align/ASR/TTS/MT path
#   tier "audio_only" no matching text anywhere on Bible.is -> speech corpus +
#                     bridge pseudo-labels + human transcription queue
# ---------------------------------------------------------------------------
pipeline_b:
  dsh:
    name: Daasanach
    tier: text
    drive_folder: Daasanach
    mms_asr: dsh          # MMS happens to cover it
    mms_tts: dsh
    nllb: null
    nllb_init: som_Latn
    sets:
      - {audio: DSHBTLN1DA, text: DSHBTL, style: plain}
      - {audio: DSHBTLN2DA, text: DSHBTL, style: drama}
  mlk:
    name: Ilwana
    tier: text
    drive_folder: Ilwana
    mms_asr: null
    asr_bridge: pkb
    mms_tts: null
    tts_bridge: pkb       # Tana River Bantu neighbour
    nllb: null
    nllb_init: swh_Latn
    sets:
      - {audio: MLKBTLN1DA, text: MLKBTLN_ET, style: plain}
      - {audio: MLKBTLN2DA, text: MLKBTLN_ET, style: drama}
  mer:
    name: Meru (Kimeru)
    tier: text
    drive_folder: Kimeru
    mms_asr: null
    asr_bridge: kik
    mms_tts: null
    tts_bridge: kik
    nllb: null
    nllb_init: kik_Latn
    note: "Text set MERUBS covers only parts of the NT/OT; unmatched chapters are skipped automatically."
    sets:
      - {audio: MERUBSN2DA, text: MERUBS, style: drama}
      - {audio: MERDPIO1DA, text: null, style: plain}
  # ---- audio only -------------------------------------------------------
  guz:
    name: Ekegusii
    tier: text
    drive_folder: Ekegusii
    asr_bridge: kik
    tts_bridge: kik
    nllb_init: kik_Latn
    licence: "Text: Ebibilia Enchenu (c) 2021 Bible Society of Kenya, NOT openly licensed -- private training
      only; obtain the Bible Society of Kenya's permission before distributing models, translations or audio."
    note: "eBible 'guz' is FCBH GUZBSK = the GUZBSKN2DA recordings (drama NT)."
    sets:
      - {audio: GUZBSKN2DA, text: "ebible:guz", style: drama}
      - {audio: GUZDPIN1DA, style: plain}
      - {audio: GUZDPIO1DA, style: plain}
  mas: {name: Maasai, tier: audio_only, drive_folder: Maasai, asr_bridge: saq, tts_bridge: saq, nllb_init: swh_Latn,
        sets: [{audio: MASDPIN1DA, style: plain}, {audio: MASDPIO1DA, style: plain}, {audio: METBSVN2DA, style: drama}]}
  ebu: {name: Embu (Kiembu), tier: audio_only, drive_folder: Kiembu, asr_bridge: kik, tts_bridge: kik, nllb_init: kik_Latn,
        sets: [{audio: EBUDPIO1DA, style: plain}, {audio: EBUBSKN2DA, style: drama}]}
  tuv: {name: Turkana, tier: audio_only, drive_folder: Turkana, asr_bridge: teo, tts_bridge: teo, nllb_init: luo_Latn,
        sets: [{audio: TUVDPIN1DA, style: plain}, {audio: TUVDPIO1DA, style: plain}, {audio: TUVBSKN2DA, style: drama}]}
  kuj: {name: Kuria, tier: audio_only, drive_folder: Kuria, asr_bridge: sxb, tts_bridge: sxb, nllb_init: lug_Latn,
        sets: [{audio: KUJDPIN1DA, style: plain}, {audio: KUJDPIO1DA, style: plain}]}
  bxk: {name: Bukusu (Luhya), tier: audio_only, drive_folder: Lubukusu, asr_bridge: lsm, tts_bridge: lsm, nllb_init: lug_Latn,
        sets: [{audio: BXKBSKN1DA, style: plain}, {audio: BXKDPIO1DA, style: plain}, {audio: BXKBSKN2DA, style: drama}]}
  rag: {name: Logooli (Luhya), tier: audio_only, drive_folder: Logooli, asr_bridge: lsm, tts_bridge: lsm, nllb_init: lug_Latn,
        sets: [{audio: RAGBSKN1DA, style: plain}, {audio: RAGDPIO1DA, style: plain}]}
  lwg: {name: Wanga (Luhya), tier: audio_only, drive_folder: Oluwanga, asr_bridge: lsm, tts_bridge: lsm, nllb_init: lug_Latn,
        sets: [{audio: LWGDPIO1DA, style: plain}]}
  nyd: {name: Nyore (Luhya), tier: audio_only, drive_folder: Lunyore, asr_bridge: lsm, tts_bridge: lsm, nllb_init: lug_Latn,
        sets: [{audio: NYDBSKN1DA, style: plain}, {audio: NYDDPIO1DA, style: plain}, {audio: NYDBSKN2DA, style: drama}]}
  lto: {name: Tsotso (Luhya), tier: audio_only, drive_folder: Tsotso, asr_bridge: lsm, tts_bridge: lsm, nllb_init: lug_Latn,
        sets: [{audio: LTODPIO1DA, style: plain}]}
  luy: {name: Luhya (Oluluyia), tier: audio_only, drive_folder: Oluluyia, asr_bridge: lsm, tts_bridge: lsm, nllb_init: lug_Latn,
        sets: [{audio: LUYBSKN2DA, style: drama}]}
  dav: {name: Taita (Kidaw'ida), tier: audio_only, drive_folder: Taita, asr_bridge: dig, tts_bridge: dig, nllb_init: swh_Latn,
        sets: [{audio: DAVBSKN1DA, style: plain}, {audio: DAVDPIO1DA, style: plain}, {audio: DAVBSKN2DA, style: drama}]}
  tvs: {name: Taveta, tier: audio_only, drive_folder: Taveta, asr_bridge: dig, tts_bridge: dig, nllb_init: swh_Latn,
        sets: [{audio: TVSBTLN1DA, style: plain}, {audio: TVSBTLN2DA, style: drama}]}
  tga: {name: Sagalla, tier: audio_only, drive_folder: Sagalla, asr_bridge: dig, tts_bridge: dig, nllb_init: swh_Latn,
        sets: [{audio: TGADPIN1DA, style: plain}]}
  coh: {name: Chonyi, tier: audio_only, drive_folder: Chonyi, asr_bridge: nyf, tts_bridge: nyf, nllb_init: swh_Latn,
        sets: [{audio: COHBTLN1DA, style: plain}, {audio: COHBTLN2DA, style: drama}]}
  cuh: {name: Chuka (Gichuka), tier: audio_only, drive_folder: Gichuka, asr_bridge: kik, tts_bridge: kik, nllb_init: kik_Latn,
        sets: [{audio: CUHTBLN1DA, style: plain}, {audio: CUHTBLN2DA, style: drama}]}
  kln: {name: Kalenjin, tier: audio_only, drive_folder: Kalenjin, asr_bridge: enb, tts_bridge: enb, nllb_init: luo_Latn,
        sets: [{audio: KLNRKVN1DA, style: plain}, {audio: KLNDPIO1DA, style: plain}, {audio: KLNRKVN2DA, style: drama}]}
  tuy: {name: Tugen (Kalenjin), tier: audio_only, drive_folder: Tugen, asr_bridge: enb, tts_bridge: enb, nllb_init: luo_Latn,
        sets: [{audio: TUYBTLN1DA, style: plain}, {audio: TUYBTLN2DA, style: drama}]}
  pko: {name: Pokot, tier: audio_only, drive_folder: Pokoot, asr_bridge: enb, tts_bridge: enb, nllb_init: luo_Latn,
        sets: [{audio: PKODPIO1DA, style: plain}, {audio: PKOBSKN2DA, style: drama}]}
  ssn: {name: Waata, tier: audio_only, drive_folder: Waata, asr_bridge: orm, tts_bridge: orm, nllb_init: gaz_Latn,
        sets: [{audio: SSNBTLN1DA, style: plain}, {audio: SSNBTLN2DA, style: drama}]}
  nle: {name: East Nyala (Luhya), tier: audio_only, drive_folder: Nyala, asr_bridge: lsm, tts_bridge: lsm, nllb_init: lug_Latn,
        sets: [{audio: NLEBSKN1DA, style: plain}, {audio: NLEBSKN2DA, style: drama}]}
  teo_nyakutio: {name: Ateso (Nyakutio), tier: audio_only, drive_folder: Nyakutio, asr_bridge: teo, tts_bridge: teo, nllb_init: luo_Latn,
        sets: [{audio: TEOBSKN1DA, style: plain}, {audio: TEOBSKN2DA, style: drama}]}

# English side of every parallel corpus: World English Bible (public domain).
english:
  nt: ENGWEBN_ET
  ot: ENGWEBO_ET

In [ ]:
%%writefile /tmp/btp/config/pipeline.yaml
# pipeline.yaml -- paths and hyper-parameters. Every value can be overridden from
# the CLI with  --set key.subkey=value  (e.g. --set mt.max_steps=200).

paths:
  # Raw chapter MP3s copied from Bible.is (read-only for this project).
  raw_audio_root: /content/drive/MyDrive/adventproclaimer_training_data
  # Everything this project writes (corpora, aligned clips, checkpoints, logs).
  workspace: /content/drive/MyDrive/btp_workspace
  # Fast local scratch on the Colab VM (copied from / to Drive as needed).
  local_cache: /content/btp_cache
  # Where chapter MP3s come from: "drive" = the copies under raw_audio_root (Colab);
  # "bibleis" = streamed from Bible.is with fresh signed links (Kaggle / anywhere without Drive).
  audio_source: drive

bibleis:
  api: https://live.bible.is/api
  request_delay_sec: 0.25       # be polite to the public API
  timeout_sec: 60

splits:
  # Held-out books, shared by MT, ASR and TTS so scores are comparable across tasks
  # and no test verse is ever seen in training. NT books first (every language has NT).
  dev_books: [COL, 2TH]
  test_books: [PHP, 1JN]
  # Extra held-out OT books for languages that also have OT text.
  dev_books_ot: [RUT]
  test_books_ot: [JON]

align:
  model: MMS_FA                 # torchaudio.pipelines.MMS_FA (language-agnostic, uroman)
  chunk_sec: 900                # chapters longer than this are aligned in windows
  min_clip_sec: 1.0
  max_clip_sec: 20.0
  # Verse confidence = mean over the verse's frames of log p(forced label) - max log p.
  # Measured on real Samburu audio: correct verses median -0.06 (worst -0.82); verses
  # paired with the wrong text -1.5 .. -5.4. -1.0 separates the two.
  min_score: -1.0
  pad_sec: 0.08
  # Aligning every text-matched hour (~700 h) would take 30+ T4-hours. TTS uses <= 6 h of one voice
  # and ASR <= 10 h, so stop each language at this many aligned hours (held-out books always first,
  # plain narration sets before drama). Raise it later to grow the ASR data.
  max_hours_per_lang: 15

pseudo_label:                   # for audio sets with no matching text (e.g. Luo, Kamba)
  segment_min_sec: 3.0
  segment_max_sec: 15.0
  silence_top_db: 35
  min_confidence: 0.80          # mean max-softmax of the CTC frames that emit a token
  max_hours_per_set: 12

mt:
  base_model: facebook/nllb-200-distilled-600M
  directions: [en2x, x2en]      # en2x is what book translation needs; x2en helps transfer
  max_source_len: 128
  max_target_len: 128
  # T4 (15 GB): NLLB's 256k-token vocabulary makes logits + AdamW states overflow at batch 16
  # (measured OOM on Colab). Adafactor + batch 8 x accum 4 keeps the same 32-pair updates.
  optimizer: adafactor
  per_device_batch: 8
  grad_accum: 4
  lr: 1.0e-4                    # Adafactor (no relative step) tolerates a higher LR than AdamW
  replay_pairs: 2000
  warmup_steps: 500
  max_steps: 20000              # resumable; a T4 session typically gets 6-8k steps
  eval_steps: 1000
  save_steps: 1000
  eval_samples_per_lang: 200    # dev subset decoded during training (full test at eval)
  num_beams: 4
  gradient_checkpointing: true
  fp16: true
  prune_checkpoint_when_done: false   # true on Kaggle (20 GB output): keep only final/ once training ends
  temperature_sampling: 0.3     # T for sampling languages ~ n^T (upsamples small languages)

asr:
  base_model: facebook/mms-1b-all
  max_audio_sec: 15
  hours_per_lang: 10            # cap of training audio per language
  per_device_batch: 8
  grad_accum: 2
  lr: 1.0e-3                    # adapters only -> high LR is the MMS recipe
  warmup_steps: 100
  max_steps: 2000
  eval_steps: 250
  save_steps: 250
  fp16: true
  eval_samples: 300
  # Build on MMS: it was trained on Bible recordings, and fine-tuning on the same kind of audio did
  # not help where it is already good (Swahili dev CER 2.41% zero-shot -> 2.50% after 2000 steps).
  accept_zero_shot_cer: 0.03    # dev CER of the released adapter at/below this -> adopted, no training
  early_stop_evals: 2           # stop after 2 evaluations (500 steps) without a real dev CER gain...
  early_stop_min_rel_improvement: 0.03   # ...of at least 3% relative over the best so far (MMS included)
  min_minutes_to_start: 20      # don't load the 1B model for another language this close to the deadline

tts:
  recipe_repo: https://github.com/ylacombe/finetune-hf-vits.git
  recipe_commit: main
  hours_per_lang: 6             # VITS fine-tuning needs little data; plain narration preferred
  prefer_style: plain
  per_device_batch: 16
  grad_accum: 1
  lr: 2.0e-5
  max_steps: 4000
  eval_steps: 500
  save_steps: 500
  fp16: true
  max_clip_sec: 15
  eval_sentences: 20
  min_minutes_to_start: 25      # don't start another voice this close to the deadline
  prune_checkpoint_when_done: false   # true on Kaggle: drop the ~1.3 GB training checkpoint once a voice is done

book:
  max_chars_per_unit: 350
  sentence_pause_sec: 0.35
  paragraph_pause_sec: 0.9
  target_dbfs: -20.0

tracking:
  backend: wandb                # wandb | none
  project: kenyan-speech-mt
  offline_if_no_key: true       # log to <workspace>/wandb and `wandb sync` later

In [ ]:
%%writefile /tmp/btp/third_party/__init__.py
"""Third-party code vendored with its licence (see each sub-package README)."""

In [ ]:
%%writefile /tmp/btp/third_party/finetune_hf_vits/__init__.py
"""MIT-licensed modules from ylacombe/finetune-hf-vits (see LICENSE, README.md)."""
from .configuration_vits import VitsConfig
from .feature_extraction_vits import VitsFeatureExtractor
from .modeling_vits_training import VitsDiscriminator, VitsModelForPreTraining, slice_segments

In [ ]:
%%writefile /tmp/btp/third_party/finetune_hf_vits/configuration_vits.py
""" VITS model configuration"""


from transformers.configuration_utils import PretrainedConfig
from transformers.utils import logging


logger = logging.get_logger(__name__)

VITS_PRETRAINED_CONFIG_ARCHIVE_MAP = {
    "facebook/mms-tts-eng": "https://huggingface.co/facebook/mms-tts-eng/resolve/main/config.json",
}


class VitsConfig(PretrainedConfig):
    r"""
    This is the configuration class to store the configuration of a [`VitsModel`]. It is used to instantiate a VITS
    model according to the specified arguments, defining the model architecture. Instantiating a configuration with the
    defaults will yield a similar configuration to that of the VITS
    [facebook/mms-tts-eng](https://huggingface.co/facebook/mms-tts-eng) architecture.

    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the
    documentation from [`PretrainedConfig`] for more information.

    Args:
        vocab_size (`int`, *optional*, defaults to 38):
            Vocabulary size of the VITS model. Defines the number of different tokens that can be represented by the
            `inputs_ids` passed to the forward method of [`VitsModel`].
        hidden_size (`int`, *optional*, defaults to 192):
            Dimensionality of the text encoder layers.
        num_hidden_layers (`int`, *optional*, defaults to 6):
            Number of hidden layers in the Transformer encoder.
        num_attention_heads (`int`, *optional*, defaults to 2):
            Number of attention heads for each attention layer in the Transformer encoder.
        window_size (`int`, *optional*, defaults to 4):
            Window size for the relative positional embeddings in the attention layers of the Transformer encoder.
        use_bias (`bool`, *optional*, defaults to `True`):
            Whether to use bias in the key, query, value projection layers in the Transformer encoder.
        ffn_dim (`int`, *optional*, defaults to 768):
            Dimensionality of the "intermediate" (i.e., feed-forward) layer in the Transformer encoder.
        layerdrop (`float`, *optional*, defaults to 0.1):
            The LayerDrop probability for the encoder. See the [LayerDrop paper](see https://arxiv.org/abs/1909.11556)
            for more details.
        ffn_kernel_size (`int`, *optional*, defaults to 3):
            Kernel size of the 1D convolution layers used by the feed-forward network in the Transformer encoder.
        flow_size (`int`, *optional*, defaults to 192):
            Dimensionality of the flow layers.
        spectrogram_bins (`int`, *optional*, defaults to 513):
            Number of frequency bins in the target spectrogram.
        hidden_act (`str` or `function`, *optional*, defaults to `"relu"`):
            The non-linear activation function (function or string) in the encoder and pooler. If string, `"gelu"`,
            `"relu"`, `"selu"` and `"gelu_new"` are supported.
        hidden_dropout (`float`, *optional*, defaults to 0.1):
            The dropout probability for all fully connected layers in the embeddings and encoder.
        attention_dropout (`float`, *optional*, defaults to 0.1):
            The dropout ratio for the attention probabilities.
        activation_dropout (`float`, *optional*, defaults to 0.1):
            The dropout ratio for activations inside the fully connected layer.
        initializer_range (`float`, *optional*, defaults to 0.02):
            The standard deviation of the truncated_normal_initializer for initializing all weight matrices.
        layer_norm_eps (`float`, *optional*, defaults to 1e-05):
            The epsilon used by the layer normalization layers.
        use_stochastic_duration_prediction (`bool`, *optional*, defaults to `True`):
            Whether to use the stochastic duration prediction module or the regular duration predictor.
        num_speakers (`int`, *optional*, defaults to 1):
            Number of speakers if this is a multi-speaker model.
        speaker_embedding_size (`int`, *optional*, defaults to 0):
            Number of channels used by the speaker embeddings. Is zero for single-speaker models.
        upsample_initial_channel (`int`, *optional*, defaults to 512):
            The number of input channels into the HiFi-GAN upsampling network.
        upsample_rates (`Tuple[int]` or `List[int]`, *optional*, defaults to `[8, 8, 2, 2]`):
            A tuple of integers defining the stride of each 1D convolutional layer in the HiFi-GAN upsampling network.
            The length of `upsample_rates` defines the number of convolutional layers and has to match the length of
            `upsample_kernel_sizes`.
        upsample_kernel_sizes (`Tuple[int]` or `List[int]`, *optional*, defaults to `[16, 16, 4, 4]`):
            A tuple of integers defining the kernel size of each 1D convolutional layer in the HiFi-GAN upsampling
            network. The length of `upsample_kernel_sizes` defines the number of convolutional layers and has to match
            the length of `upsample_rates`.
        resblock_kernel_sizes (`Tuple[int]` or `List[int]`, *optional*, defaults to `[3, 7, 11]`):
            A tuple of integers defining the kernel sizes of the 1D convolutional layers in the HiFi-GAN
            multi-receptive field fusion (MRF) module.
        resblock_dilation_sizes (`Tuple[Tuple[int]]` or `List[List[int]]`, *optional*, defaults to `[[1, 3, 5], [1, 3, 5], [1, 3, 5]]`):
            A nested tuple of integers defining the dilation rates of the dilated 1D convolutional layers in the
            HiFi-GAN multi-receptive field fusion (MRF) module.
        leaky_relu_slope (`float`, *optional*, defaults to 0.1):
            The angle of the negative slope used by the leaky ReLU activation.
        depth_separable_channels (`int`, *optional*, defaults to 2):
            Number of channels to use in each depth-separable block.
        depth_separable_num_layers (`int`, *optional*, defaults to 3):
            Number of convolutional layers to use in each depth-separable block.
        duration_predictor_flow_bins (`int`, *optional*, defaults to 10):
            Number of channels to map using the unonstrained rational spline in the duration predictor model.
        duration_predictor_tail_bound (`float`, *optional*, defaults to 5.0):
            Value of the tail bin boundary when computing the unconstrained rational spline in the duration predictor
            model.
        duration_predictor_kernel_size (`int`, *optional*, defaults to 3):
            Kernel size of the 1D convolution layers used in the duration predictor model.
        duration_predictor_dropout (`float`, *optional*, defaults to 0.5):
            The dropout ratio for the duration predictor model.
        duration_predictor_num_flows (`int`, *optional*, defaults to 4):
            Number of flow stages used by the duration predictor model.
        duration_predictor_filter_channels (`int`, *optional*, defaults to 256):
            Number of channels for the convolution layers used in the duration predictor model.
        prior_encoder_num_flows (`int`, *optional*, defaults to 4):
            Number of flow stages used by the prior encoder flow model.
        prior_encoder_num_wavenet_layers (`int`, *optional*, defaults to 4):
            Number of WaveNet layers used by the prior encoder flow model.
        posterior_encoder_num_wavenet_layers (`int`, *optional*, defaults to 16):
            Number of WaveNet layers used by the posterior encoder model.
        wavenet_kernel_size (`int`, *optional*, defaults to 5):
            Kernel size of the 1D convolution layers used in the WaveNet model.
        wavenet_dilation_rate (`int`, *optional*, defaults to 1):
            Dilation rates of the dilated 1D convolutional layers used in the WaveNet model.
        wavenet_dropout (`float`, *optional*, defaults to 0.0):
            The dropout ratio for the WaveNet layers.
        speaking_rate (`float`, *optional*, defaults to 1.0):
            Speaking rate. Larger values give faster synthesised speech.
        noise_scale (`float`, *optional*, defaults to 0.667):
            How random the speech prediction is. Larger values create more variation in the predicted speech.
        noise_scale_duration (`float`, *optional*, defaults to 0.8):
            How random the duration prediction is. Larger values create more variation in the predicted durations.
        sampling_rate (`int`, *optional*, defaults to 16000):
            The sampling rate at which the output audio waveform is digitalized expressed in hertz (Hz).
        discriminator_kernel_size (`int`, *optional*, defaults to 5):
            Kernel size of the first channel of the 2D convolution layers used in the discriminator model.
        discriminator_stride (`int`, *optional*, defaults to 3):
            Stride of the first channel of the 2D convolution layers used in the discriminator model.
        discriminator_periods (`Tuple[int]` or `List[int]`, *optional*, defaults to `[2, 3, 5, 7, 11]`):
            A tuple of integers defining the period of each sub-discriminator of the discriminator model.
        discriminator_period_channels (`Tuple[int]` or `List[int]`, *optional*, defaults to `[1, 32, 128, 512, 1024]`):
            A tuple of integers defining the number of convolutional layers and their channel sizes of each period
            sub-discriminator of the discriminator model. Must have more 2 elements or more and be made of increasing
            multiple of 2.
        discriminator_scale_channels (`Tuple[int]` or `List[int]`, *optional*, defaults to `[1, 16, 64, 256, 1024]`):
            A tuple of integers defining the number of convolutional layers and their channel sizes of the scale
            sub-discriminator the discriminator model. Must have more 2 elements or more and be made of increasing
            multiple of 4.
        segment_size (`int`, *optional*, defaults to 8192):
            Segment size of the spectrogam used to compute the recontruction loss during training.
        hop_length (`int`, *optional*, defaults to 256):
            Length of the overlaping windows for the STFT used to obtain the Mel Frequency coefficients. Used during
            training.
    Example:

    ```python
    >>> from transformers import VitsModel, VitsConfig

    >>> # Initializing a "facebook/mms-tts-eng" style configuration
    >>> configuration = VitsConfig()

    >>> # Initializing a model (with random weights) from the "facebook/mms-tts-eng" style configuration
    >>> model = VitsModel(configuration)

    >>> # Accessing the model configuration
    >>> configuration = model.config
    ```"""

    model_type = "vits"

    def __init__(
        self,
        vocab_size=38,
        hidden_size=192,
        num_hidden_layers=6,
        num_attention_heads=2,
        window_size=4,
        use_bias=True,
        ffn_dim=768,
        layerdrop=0.1,
        ffn_kernel_size=3,
        flow_size=192,
        spectrogram_bins=513,
        hidden_act="relu",
        hidden_dropout=0.1,
        attention_dropout=0.1,
        activation_dropout=0.1,
        initializer_range=0.02,
        layer_norm_eps=1e-5,
        use_stochastic_duration_prediction=True,
        num_speakers=1,
        speaker_embedding_size=0,
        upsample_initial_channel=512,
        upsample_rates=[8, 8, 2, 2],
        upsample_kernel_sizes=[16, 16, 4, 4],
        resblock_kernel_sizes=[3, 7, 11],
        resblock_dilation_sizes=[[1, 3, 5], [1, 3, 5], [1, 3, 5]],
        leaky_relu_slope=0.1,
        depth_separable_channels=2,
        depth_separable_num_layers=3,
        duration_predictor_flow_bins=10,
        duration_predictor_tail_bound=5.0,
        duration_predictor_kernel_size=3,
        duration_predictor_dropout=0.5,
        duration_predictor_num_flows=4,
        duration_predictor_filter_channels=256,
        prior_encoder_num_flows=4,
        prior_encoder_num_wavenet_layers=4,
        posterior_encoder_num_wavenet_layers=16,
        wavenet_kernel_size=5,
        wavenet_dilation_rate=1,
        wavenet_dropout=0.0,
        speaking_rate=1.0,
        noise_scale=0.667,
        noise_scale_duration=0.8,
        sampling_rate=16_000,
        discriminator_kernel_size=5,
        discriminator_stride=3,
        discriminator_periods=[2, 3, 5, 7, 11],
        discriminator_period_channels=[1, 32, 128, 512, 1024],
        discriminator_scale_channels=[1, 16, 64, 256, 1024],
        segment_size=8192,
        hop_length=256,
        pad_token_id=0,
        **kwargs,
    ):
        # btp patch: transformers 5 no longer defines pad_token_id on PretrainedConfig, but the
        # text encoder's nn.Embedding(padding_idx=config.pad_token_id) needs it.
        self.pad_token_id = pad_token_id
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.window_size = window_size
        self.use_bias = use_bias
        self.ffn_dim = ffn_dim
        self.layerdrop = layerdrop
        self.ffn_kernel_size = ffn_kernel_size
        self.flow_size = flow_size
        self.spectrogram_bins = spectrogram_bins
        self.hidden_act = hidden_act
        self.hidden_dropout = hidden_dropout
        self.attention_dropout = attention_dropout
        self.activation_dropout = activation_dropout
        self.initializer_range = initializer_range
        self.layer_norm_eps = layer_norm_eps
        self.use_stochastic_duration_prediction = use_stochastic_duration_prediction
        self.num_speakers = num_speakers
        self.speaker_embedding_size = speaker_embedding_size
        self.upsample_initial_channel = upsample_initial_channel
        self.upsample_rates = upsample_rates
        self.upsample_kernel_sizes = upsample_kernel_sizes
        self.resblock_kernel_sizes = resblock_kernel_sizes
        self.resblock_dilation_sizes = resblock_dilation_sizes
        self.leaky_relu_slope = leaky_relu_slope
        self.depth_separable_channels = depth_separable_channels
        self.depth_separable_num_layers = depth_separable_num_layers
        self.duration_predictor_flow_bins = duration_predictor_flow_bins
        self.duration_predictor_tail_bound = duration_predictor_tail_bound
        self.duration_predictor_kernel_size = duration_predictor_kernel_size
        self.duration_predictor_dropout = duration_predictor_dropout
        self.duration_predictor_num_flows = duration_predictor_num_flows
        self.duration_predictor_filter_channels = duration_predictor_filter_channels
        self.prior_encoder_num_flows = prior_encoder_num_flows
        self.prior_encoder_num_wavenet_layers = prior_encoder_num_wavenet_layers
        self.posterior_encoder_num_wavenet_layers = posterior_encoder_num_wavenet_layers
        self.wavenet_kernel_size = wavenet_kernel_size
        self.wavenet_dilation_rate = wavenet_dilation_rate
        self.wavenet_dropout = wavenet_dropout
        self.speaking_rate = speaking_rate
        self.noise_scale = noise_scale
        self.noise_scale_duration = noise_scale_duration
        self.sampling_rate = sampling_rate

        # used for training
        self.discriminator_kernel_size = discriminator_kernel_size
        self.discriminator_stride = discriminator_stride
        self.discriminator_periods = discriminator_periods
        self.discriminator_period_channels = discriminator_period_channels
        self.discriminator_scale_channels = discriminator_scale_channels
        self.segment_size = segment_size
        self.hop_length = hop_length

        if len(upsample_kernel_sizes) != len(upsample_rates):
            raise ValueError(
                f"The length of `upsample_kernel_sizes` ({len(upsample_kernel_sizes)}) must match the length of "
                f"`upsample_rates` ({len(upsample_rates)})"
            )

        super().__init__(**kwargs)
        if getattr(self, "pad_token_id", None) is None:   # btp patch (see above)
            self.pad_token_id = pad_token_id

In [ ]:
%%writefile /tmp/btp/third_party/finetune_hf_vits/convert_discriminator.py
"""Convert VITS discriminator checkpoint and add it to an already converted VITS checkpoint."""

import argparse
import torch

from transformers.models.vits.modeling_vits import VitsModel
from transformers.models.vits.tokenization_vits import VitsTokenizer

from huggingface_hub import hf_hub_download

from .feature_extraction_vits import VitsFeatureExtractor
from .configuration_vits import VitsConfig, logging
from .modeling_vits_training import VitsDiscriminator, VitsModelForPreTraining

logging.set_verbosity_info()
logger = logging.get_logger("transformers.models.vits")


MAPPING = {
    "conv_post": "final_conv",
}
TOP_LEVEL_KEYS = []
IGNORE_KEYS = []


@torch.no_grad()
def convert_checkpoint(
    language_code,
    pytorch_dump_folder_path,
    checkpoint_path=None,
    generator_checkpoint_path=None,
    repo_id=None,
):
    """
    Copy/paste/tweak model's weights to transformers design.
    """
    if language_code is not None:
        checkpoint_path = hf_hub_download(repo_id="facebook/mms-tts", subfolder=f"full_models/{language_code}", filename="D_100000.pth")
        generator_checkpoint_path = f"facebook/mms-tts-{language_code}"
    
    config = VitsConfig.from_pretrained(generator_checkpoint_path)
    generator = VitsModel.from_pretrained(generator_checkpoint_path)

    discriminator = VitsDiscriminator(config)

    for disc in discriminator.discriminators:
        disc.apply_weight_norm()

    checkpoint = torch.load(checkpoint_path, map_location=torch.device("cpu"))

    # load weights

    state_dict = checkpoint["model"]

    for k, v in list(state_dict.items()):
        for old_layer_name in MAPPING:
            new_k = k.replace(old_layer_name, MAPPING[old_layer_name])

        state_dict[new_k] = state_dict.pop(k)

    extra_keys = set(state_dict.keys()) - set(discriminator.state_dict().keys())
    extra_keys = {k for k in extra_keys if not k.endswith(".attn.bias")}
    missing_keys = set(discriminator.state_dict().keys()) - set(state_dict.keys())
    missing_keys = {k for k in missing_keys if not k.endswith(".attn.bias")}
    if len(extra_keys) != 0:
        raise ValueError(f"extra keys found: {extra_keys}")
    if len(missing_keys) != 0:
        raise ValueError(f"missing keys: {missing_keys}")
    discriminator.load_state_dict(state_dict, strict=False)
    n_params = discriminator.num_parameters(exclude_embeddings=True)
    logger.info(f"model loaded: {round(n_params/1e6,1)}M params")

    for disc in discriminator.discriminators:
        disc.remove_weight_norm()

    model = VitsModelForPreTraining(config)

    # load weights
    model.text_encoder = generator.text_encoder
    model.flow = generator.flow
    model.decoder = generator.decoder
    model.duration_predictor = generator.duration_predictor
    model.posterior_encoder = generator.posterior_encoder

    if config.num_speakers > 1:
        model.embed_speaker = generator.embed_speaker

    model.discriminator = discriminator
    tokenizer = VitsTokenizer.from_pretrained(generator_checkpoint_path, verbose=False)
    feature_extractor = VitsFeatureExtractor(sampling_rate=model.config.sampling_rate, feature_size=80)

    model.save_pretrained(pytorch_dump_folder_path)
    tokenizer.save_pretrained(pytorch_dump_folder_path)
    feature_extractor.save_pretrained(pytorch_dump_folder_path)

    if repo_id:
        print("Pushing to the hub...")
        model.push_to_hub(repo_id)
        tokenizer.push_to_hub(repo_id)
        feature_extractor.push_to_hub(repo_id)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    
    parser.add_argument("--language_code", default=None, type=str, help="""If set, indicates the language code of the MMS checkpoint to convert. 
                        In that case, it will automatically creates the right MMS checkpoint in the HF required format.
                        If used, `checkpoint_path` and `generator_checkpoint_path` are ignored.""")
    
    parser.add_argument(
        "--checkpoint_path", default=None, type=str, help="Local path to original discriminator checkpoint. Ignored if `language_code` is used."
    )
    parser.add_argument(
        "--generator_checkpoint_path", default=None, type=str, help="Path to the 🤗 generator (VitsModel). Ignored if `language_code` is used."
    )
    parser.add_argument(
        "--pytorch_dump_folder_path", required=True, default=None, type=str, help="Path to the output PyTorch model."
    )
    parser.add_argument(
        "--push_to_hub", default=None, type=str, help="Where to upload the converted model on the 🤗 hub."
    )
    
    args = parser.parse_args()
    convert_checkpoint(
        args.language_code,
        args.pytorch_dump_folder_path,
        args.checkpoint_path,
        args.generator_checkpoint_path,
        args.push_to_hub,
    )

In [ ]:
%%writefile /tmp/btp/third_party/finetune_hf_vits/feature_extraction_vits.py
"""
Feature extractor class for Vits
"""
import copy
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np

from transformers import is_torch_available
from transformers.audio_utils import mel_filter_bank
from transformers.feature_extraction_sequence_utils import SequenceFeatureExtractor
from transformers.feature_extraction_utils import BatchFeature
from transformers.utils import TensorType, logging


MAX_WAV_VALUE = 32768.0

if is_torch_available():
    import torch

logger = logging.get_logger(__name__)


class VitsFeatureExtractor(SequenceFeatureExtractor):
    r"""
    Constructs a Vits feature extractor.

    This feature extractor inherits from [`~feature_extraction_sequence_utils.SequenceFeatureExtractor`] which contains
    most of the main methods. Users should refer to this superclass for more information regarding those methods.

    This class extracts `Short Time Fourier Transform` from raw speech using a custom numpy implementation which should
    match pytorch's `torch.stft`.

    Args:
        feature_size (`int`, defaults to 80):
            The feature dimension of the extracted features.
        sampling_rate (`int`, defaults to 22050):
            The sampling rate at which the audio files should be digitalized expressed in hertz (Hz).
        hop_length (`int`, defaults to 256):
            Length of the overlaping windows for the STFT used to obtain the Mel Frequency coefficients.
        n_fft (`int`, defaults to 1024):
            Size of the Fourier transform.
        padding_value (`float`, *optional*, defaults to 0.0):
            Padding value used to pad the audio. Should correspond to silences.
        return_attention_mask (`bool`, *optional*, defaults to `False`):
            Whether to return the attention mask.

            [What are attention masks?](../glossary#attention-mask)

            <Tip>

            For Vits finetuning, `attention_mask` should always be passed for batched inference, to avoid subtle bugs.

            </Tip>

        max_wav_value (`float`, defaults to 32768.0):
            Maximum wav value. Used to normalize the input waveforms if `do_normalize=True` in the forward pass of this
            feature extractor.
    """

    model_input_names = ["input_features"]

    def __init__(
        self,
        feature_size=80,
        sampling_rate=22050,
        hop_length=256,
        n_fft=1024,
        padding_value=0.0,
        return_attention_mask=False,  # pad inputs to max length with silence token (zero) and no attention mask,
        max_wav_value=32768.0,
        **kwargs,
    ):
        super().__init__(
            feature_size=feature_size,
            sampling_rate=sampling_rate,
            padding_value=padding_value,
            return_attention_mask=return_attention_mask,
            **kwargs,
        )
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.sampling_rate = sampling_rate
        self.mel_filters = mel_filter_bank(
            num_frequency_bins=1 + n_fft // 2,
            num_mel_filters=feature_size,
            min_frequency=0.0,
            max_frequency=sampling_rate // 2,
            sampling_rate=sampling_rate,
            norm="slaney",
            mel_scale="slaney",
        )
        self.max_wav_value = max_wav_value

    def _torch_extract_fbank_features(self, waveform: np.array) -> Tuple[torch.Tensor]:
        """
        Compute the log-mel spectrogram of the provided audio using the PyTorch STFT implementation.
        """
        if len(waveform.shape) == 1:
            waveform = waveform.unsqueeze(0)

        waveform = torch.nn.functional.pad(
            waveform,
            (int((self.n_fft - self.hop_length) / 2), int((self.n_fft - self.hop_length) / 2)),
            mode="reflect",
        )

        window = torch.hann_window(self.n_fft).to(waveform.device)
        stft = torch.stft(
            waveform,
            self.n_fft,
            hop_length=self.hop_length,
            win_length=self.n_fft,
            window=window,
            center=False,
            pad_mode="reflect",
            normalized=False,
            onesided=True,
            return_complex=False,
        )
        magnitudes = torch.sqrt(stft.pow(2).sum(-1) + 1e-6)

        mel_filters = torch.from_numpy(self.mel_filters).type(torch.float32).to(waveform.device)
        mel_spec = mel_filters.T @ magnitudes

        log_spec = torch.clamp(mel_spec, min=1e-5).log()
        return magnitudes, log_spec

    def __call__(
        self,
        raw_speech: Union[np.ndarray, List[float], List[np.ndarray], List[List[float]]],
        truncation: bool = False,
        pad_to_multiple_of: Optional[int] = None,
        return_tensors: Optional[Union[str, TensorType]] = None,
        return_attention_mask: Optional[bool] = True,
        padding: Optional[str] = True,
        max_length: Optional[int] = None,
        sampling_rate: Optional[int] = None,
        do_normalize: Optional[bool] = None,
        **kwargs,
    ) -> BatchFeature:
        """
        Main method to featurize and prepare for the model one or several sequence(s).

        Args:
            raw_speech (`np.ndarray`, `List[float]`, `List[np.ndarray]`, `List[List[float]]`):
                The sequence or batch of sequences to be padded. Each sequence can be a numpy array, a list of float
                values, a list of numpy arrays or a list of list of float values. Must be mono channel audio, not
                stereo, i.e. single float per timestep.
            truncation (`bool`, *optional*, default to `False`):
                Activates truncation to cut input sequences longer than *max_length* to *max_length*.
            pad_to_multiple_of (`int`, *optional*, defaults to None):
                If set will pad the sequence to a multiple of the provided value.

                This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability
                `>= 7.5` (Volta), or on TPUs which benefit from having sequence lengths be a multiple of 128.
            return_tensors (`str` or [`~utils.TensorType`], *optional*):
                If set, will return tensors instead of list of python integers. Acceptable values are:

                - `'tf'`: Return TensorFlow `tf.constant` objects.
                - `'pt'`: Return PyTorch `torch.Tensor` objects.
                - `'np'`: Return Numpy `np.ndarray` objects.
            return_attention_mask (`bool`, *optional*, defaults to `True`):
                Whether to return the attention mask. If left to the default, will return the attention mask according
                to the specific feature_extractor's default.

                [What are attention masks?](../glossary#attention-mask)

                <Tip>

                For Vits finetuning, `attention_mask` should always be passed for batched inference, to avoid subtle
                bugs.

                </Tip>

            padding (`bool`, `str` or [`~utils.PaddingStrategy`], *optional*, defaults to `True`):
                Select a strategy to pad the returned sequences (according to the model's padding side and padding
                index) among:

                - `True` or `'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
                  sequence if provided).
                - `'max_length'`: Pad to a maximum length specified with the argument `max_length` or to the maximum
                  acceptable input length for the model if that argument is not provided.
                - `False` or `'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of different
                  lengths).
            max_length (`int`, *optional*):
                Maximum length of the returned list and optionally padding length (see above).
            sampling_rate (`int`, *optional*):
                The sampling rate at which the `raw_speech` input was sampled. It is strongly recommended to pass
                `sampling_rate` at the forward call to prevent silent errors and allow automatic speech recognition
                pipeline.
            do_normalize (`bool`, *optional*):
                Whether or not to divide the input waveform by `self.max_wav_value`.
        """

        if sampling_rate is not None:
            if sampling_rate != self.sampling_rate:
                raise ValueError(
                    f"The model corresponding to this feature extractor: {self.__class__.__name__} was trained using a"
                    f" sampling rate of {self.sampling_rate}. Please make sure that the provided `raw_speech` input"
                    f" was sampled with {self.sampling_rate} and not {sampling_rate}."
                )
        else:
            logger.warning(
                "It is strongly recommended to pass the `sampling_rate` argument to this function. "
                "Failing to do so can result in silent errors that might be hard to debug."
            )

        is_batched_numpy = isinstance(raw_speech, np.ndarray) and len(raw_speech.shape) > 1
        if is_batched_numpy and len(raw_speech.shape) > 2:
            raise ValueError(f"Only mono-channel audio is supported for input to {self}")
        is_batched = is_batched_numpy or (
            isinstance(raw_speech, (list, tuple)) and (isinstance(raw_speech[0], (np.ndarray, tuple, list)))
        )

        if is_batched:
            raw_speech = [np.asarray([speech], dtype=np.float32).T for speech in raw_speech]
        elif not is_batched and not isinstance(raw_speech, np.ndarray):
            raw_speech = np.asarray(raw_speech, dtype=np.float32)
        elif isinstance(raw_speech, np.ndarray) and raw_speech.dtype is np.dtype(np.float64):
            raw_speech = raw_speech.astype(np.float32)

        # always return batch
        if not is_batched:
            raw_speech = [np.asarray([raw_speech]).T]

        if self.max_wav_value is not None and do_normalize:
            raw_speech = [
                speech if self.max_wav_value is None else speech / self.max_wav_value for speech in raw_speech
            ]

        batched_speech = BatchFeature({"input_features": raw_speech})

        # convert into correct format for padding
        padded_inputs = self.pad(
            batched_speech,
            padding=padding,
            max_length=max_length,
            truncation=truncation,
            pad_to_multiple_of=pad_to_multiple_of,
            return_attention_mask=return_attention_mask or do_normalize,
            return_tensors="pt",
        )

        # make sure list is in array format
        if isinstance(padded_inputs.get("input_features"),list):
            input_features = torch.tensor(padded_inputs.get("input_features")).transpose(1, 2).transpose(0, 1)
        else:
            input_features = padded_inputs.get("input_features").clone().detach().transpose(1, 2).transpose(0, 1)


        input_features = self._torch_extract_fbank_features(input_features[0])

        mel_scaled_input_features = input_features[1]
        input_features = input_features[0]

        padded_inputs["input_features"] = input_features
        padded_inputs["mel_scaled_input_features"] = mel_scaled_input_features

        if return_attention_mask:
            # rescale from sample (48000) to feature (3000)
            padded_inputs["attention_mask"] = padded_inputs["attention_mask"][:, :: self.hop_length]

        if return_tensors is not None:
            padded_inputs = padded_inputs.convert_to_tensors(return_tensors)

        return padded_inputs

    def to_dict(self) -> Dict[str, Any]:
        """
        Serializes this instance to a Python dictionary.

        Returns:
            `Dict[str, Any]`: Dictionary of all the attributes that make up this configuration instance.
        """
        output = copy.deepcopy(self.__dict__)
        output["feature_extractor_type"] = self.__class__.__name__
        if "mel_filters" in output:
            del output["mel_filters"]
        return output

In [ ]:
%%writefile /tmp/btp/third_party/finetune_hf_vits/modeling_vits_training.py
# coding=utf-8
# Copyright 2023 The Kakao Enterprise Authors and the HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
""" PyTorch VITS model."""

import math
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any, Optional, Tuple, Union

import numpy as np
import torch
import torch.utils.checkpoint
from torch import nn

from transformers.activations import ACT2FN
from transformers.integrations.deepspeed import is_deepspeed_zero3_enabled
from transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask
from transformers.modeling_outputs import (
    BaseModelOutput,
    ModelOutput,
)
from transformers.modeling_utils import PreTrainedModel
from transformers.utils import add_start_docstrings, add_start_docstrings_to_model_forward, logging, replace_return_docstrings
from .configuration_vits import VitsConfig


logger = logging.get_logger(__name__)


# General docstring
_CONFIG_FOR_DOC = "VitsConfig"


VITS_PRETRAINED_MODEL_ARCHIVE_LIST = [
    "facebook/mms-tts-eng",
    # See all VITS models at https://huggingface.co/models?filter=vits
    # and all MMS models at https://huggingface.co/models?sort=trending&search=facebook%2Fmms-tts
]


@dataclass
class VitsModelOutput(ModelOutput):
    """
    Describes the outputs for the VITS model, with potential hidden states and attentions.

    Args:
        waveform (`torch.FloatTensor` of shape `(batch_size, sequence_length)`):
            The final audio waveform predicted by the model.
        sequence_lengths  (`torch.FloatTensor` of shape `(batch_size,)`):
            The length in samples of each element in the `waveform` batch.
        spectrogram (`torch.FloatTensor` of shape `(batch_size, sequence_length, num_bins)`):
            The log-mel spectrogram predicted at the output of the flow model. This spectrogram is passed to the Hi-Fi
            GAN decoder model to obtain the final audio waveform.
        hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):
            Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +
            one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.

            Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.
        attentions (`tuple(torch.FloatTensor)`, *optional*, returned when `output_attentions=True` is passed or when `config.output_attentions=True`):
            Tuple of `torch.FloatTensor` (one for each layer) of shape `(batch_size, num_heads, sequence_length,
            sequence_length)`.

            Attention weights after the attention softmax, used to compute the weighted average in the self-attention
            heads.
    """

    waveform: torch.FloatTensor = None
    sequence_lengths: torch.FloatTensor = None
    spectrogram: Optional[Tuple[torch.FloatTensor]] = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None


@dataclass
class VitsTextEncoderOutput(ModelOutput):
    """
    Describes the outputs for the VITS text encoder model, with potential hidden states and attentions.

    Args:
        last_hidden_state (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):
            Sequence of hidden-states at the output of the last layer of the model.
        prior_means (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):
            The predicted mean values of the prior distribution for the latent text variables.
        prior_log_variances (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):
            The predicted log-variance values of the prior distribution for the latent text variables.
        hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):
            Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +
            one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.

            Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.
        attentions (`tuple(torch.FloatTensor)`, *optional*, returned when `output_attentions=True` is passed or when `config.output_attentions=True`):
            Tuple of `torch.FloatTensor` (one for each layer) of shape `(batch_size, num_heads, sequence_length,
            sequence_length)`.

            Attention weights after the attention softmax, used to compute the weighted average in the self-attention
            heads.
    """

    last_hidden_state: torch.FloatTensor = None
    prior_means: torch.FloatTensor = None
    prior_log_variances: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None


@dataclass
class VitsTrainingOutput(ModelOutput):
    """
    Describes the training outputs for the VITS model, used to compute losses when training.

    Args:
        waveform (`torch.FloatTensor` of shape `(batch_size, 1, segment_length)`):
            A sliced-segment of the audio waveform predicted by the model.
        log_duration (`torch.FloatTensor` of shape `(batch_size,)`):
            Log duration of each element of the latent tensor.
        attn (`torch.FloatTensor` of shape `(batch_size, 1, audio_length, latent_length)`):
            Attention mask. Corresponds to the most likely alignment between text and speech?
        ids_slice (`torch.FloatTensor` of shape `(batch_size,)`):
            A 1D tensor containing the starting indices for each segment of the batch.
        input_padding_mask (`torch.FloatTensor` of shape `(batch_size, 1, latent_length)`):
            Input padding mask.
        labels_padding_mask (`torch.FloatTensor` of shape `(batch_size, 1, audio_length)`):
            Labels padding mask.
        latents (`torch.FloatTensor` of shape `(batch_size, hidden_size, audio_length)`):
            Latent variable.
        prior_latents (`torch.FloatTensor` of shape `(batch_size, hidden_size, audio_length)`):
            Prior latent variable.
        prior_means (`torch.FloatTensor` of shape `(batch_size, hidden_size, audio_length)`):
            Prior means.
        prior_log_variances (`torch.FloatTensor` of shape `(batch_size, hidden_size, audio_length)`):
            Prior log variance.
        posterior_means (`torch.FloatTensor` of shape `(batch_size, hidden_size, audio_length)`):
            Posterior means.
        posterior_log_variances (`torch.FloatTensor` of shape `(batch_size, hidden_size, audio_length)`):
            Posterior log variance.
    """

    waveform: torch.FloatTensor = None
    log_duration: torch.FloatTensor = None
    attn: torch.FloatTensor = None
    ids_slice: torch.FloatTensor = None
    input_padding_mask: torch.FloatTensor = None
    labels_padding_mask: torch.FloatTensor = None
    latents: torch.FloatTensor = None
    prior_latents: torch.FloatTensor = None
    prior_means: torch.FloatTensor = None
    prior_log_variances: torch.FloatTensor = None
    posterior_means: torch.FloatTensor = None
    posterior_log_variances: torch.FloatTensor = None


@torch.jit.script
def fused_add_tanh_sigmoid_multiply(input_a, input_b, num_channels):
    in_act = input_a + input_b
    t_act = torch.tanh(in_act[:, :num_channels, :])
    s_act = torch.sigmoid(in_act[:, num_channels:, :])
    acts = t_act * s_act
    return acts


def slice_segments(hidden_states, ids_str, segment_size=4):
    """
    This method slices segments of the same size from the input tensor.

    Args:
        hidden_states (`torch.Tensor` of shape `(batch_size, channels, seq_len)`):
            The input tensor to be sliced alongside the `seq_len` dimension.
        ids_str (`torch.Tensor` of shape `(batch_size,)`):
            A 1D tensor containing the starting indices for each segment.
        segment_size (`int`, defaults to 4):
            The size of each segment to slice
    Returns:
        output (`torch.Tensor` of shape `(batch_size, channels, segment_size)`): A tensor containing the sliced
        segments.
    """

    batch_size, channels, _ = hidden_states.shape
    # 1d tensor containing the indices to keep
    indices = torch.arange(segment_size).to(ids_str.device)
    # extend the indices to match the shape of hidden_states
    indices = indices.view(1, 1, -1).expand(batch_size, channels, -1)
    # offset indices with ids_str
    indices = indices + ids_str.view(-1, 1, 1)

    # gather indices
    output = torch.gather(hidden_states, dim=2, index=indices)

    return output


def rand_slice_segments(hidden_states, sample_lengths=None, segment_size=4):
    """
    This method randomly slices segments of the same size from the input tensor.

    Args:
        hidden_states (`torch.Tensor` of shape `(batch_size, channels, seq_len)`):
            The input tensor to be sliced alongside the `seq_len` dimension.
        sample_lengths (`torch.Tensor` of shape `(batch_size,)`, *optional*):
            A 1D tensor containing the sequence length of each sample of the batch. Will defaults to `seq_len` if not
            provided.
        segment_size (`int`, defaults to 4):
            The size of each segment to slice
    Returns:
        output (`torch.Tensor` of shape `(batch_size, channels, segment_size)`): A tensor containing the sliced
        segments.
    """
    batch_size, _, seq_len = hidden_states.size()
    if sample_lengths is None:
        sample_lengths = seq_len
    ids_str_max = sample_lengths - segment_size + 1
    ids_str = (torch.rand([batch_size]).to(device=hidden_states.device) * ids_str_max).to(dtype=torch.long)
    ret = slice_segments(hidden_states, ids_str, segment_size)
    return ret, ids_str


def monotonic_align_max_path(log_likelihoods, mask):
    """
    Returns the most likely alignment between text and speech for the given log-likelihood matrix.

    Args:
        log_likelihoods (`torch.Tensor` of shape `(batch_size, text_length, latent_variable_length)`):
            The log-likelihood matrix for each sample of the batch.
        mask (`torch.Tensor` of shape `(batch_size, text_length, latent_variable_length)`, *optional*):
            The padding mask.
    Returns:
        path (`torch.Tensor` of shape `(batch_size, text_length, latent_variable_length)`): The most likely alignement.
    """
    # used for training - awfully slow
    # an alternative is proposed in examples/pytorch/text-to-speech/run_vits_finetuning.py
    path = torch.zeros_like(log_likelihoods)

    text_length_maxs = mask.sum(1)[:, 0]
    latent_length_maxs = mask.sum(2)[:, 0]

    indexes = latent_length_maxs - 1

    max_neg_val = -1e9

    for batch_id in range(len(path)):
        index = int(indexes[batch_id].item())
        text_length_max = int(text_length_maxs[batch_id].item())
        latent_length_max = int(latent_length_maxs[batch_id].item())

        for y in range(text_length_max):
            for x in range(max(0, latent_length_max + y - text_length_max), min(latent_length_max, y + 1)):
                if x == y:
                    v_cur = max_neg_val
                else:
                    v_cur = log_likelihoods[batch_id, y - 1, x]
                if x == 0:
                    if y == 0:
                        v_prev = 0.0
                    else:
                        v_prev = max_neg_val
                else:
                    v_prev = log_likelihoods[batch_id, y - 1, x - 1]
                log_likelihoods[batch_id, y, x] += max(v_prev, v_cur)

        for y in range(text_length_max - 1, -1, -1):
            path[batch_id, y, index] = 1
            if index != 0 and (
                index == y or log_likelihoods[batch_id, y - 1, index] < log_likelihoods[batch_id, y - 1, index - 1]
            ):
                index = index - 1
    return path


def _unconstrained_rational_quadratic_spline(
    inputs,
    unnormalized_widths,
    unnormalized_heights,
    unnormalized_derivatives,
    reverse=False,
    tail_bound=5.0,
    min_bin_width=1e-3,
    min_bin_height=1e-3,
    min_derivative=1e-3,
):
    """
    This transformation represents a monotonically increasing piecewise rational quadratic function. Outside of the
    `tail_bound`, the transform behaves as an identity function.

    Args:
        inputs (`torch.FloatTensor` of shape `(batch_size, channels, seq_len)`:
            Second half of the hidden-states input to the Vits convolutional flow module.
        unnormalized_widths (`torch.FloatTensor` of shape `(batch_size, channels, seq_len, duration_predictor_flow_bins)`):
            First `duration_predictor_flow_bins` of the hidden-states from the output of the convolution projection
            layer in the convolutional flow module
        unnormalized_heights (`torch.FloatTensor` of shape `(batch_size, channels, seq_len, duration_predictor_flow_bins)`):
            Second `duration_predictor_flow_bins` of the hidden-states from the output of the convolution projection
            layer in the convolutional flow module
        unnormalized_derivatives (`torch.FloatTensor` of shape `(batch_size, channels, seq_len, duration_predictor_flow_bins)`):
            Third `duration_predictor_flow_bins` of the hidden-states from the output of the convolution projection
            layer in the convolutional flow module
        reverse (`bool`, *optional*, defaults to `False`):
            Whether the model is being run in reverse mode.
        tail_bound (`float`, *optional* defaults to 5):
            Upper and lower limit bound for the rational quadratic function. Outside of this `tail_bound`, the
            transform behaves as an identity function.
        min_bin_width (`float`, *optional*, defaults to 1e-3):
            Minimum bin value across the width dimension for the piecewise rational quadratic function.
        min_bin_height (`float`, *optional*, defaults to 1e-3):
            Minimum bin value across the height dimension for the piecewise rational quadratic function.
        min_derivative (`float`, *optional*, defaults to 1e-3):
            Minimum bin value across the derivatives for the piecewise rational quadratic function.
    Returns:
        outputs (`torch.FloatTensor` of shape `(batch_size, channels, seq_len)`:
            Hidden-states as transformed by the piecewise rational quadratic function with the `tail_bound` limits
            applied.
        log_abs_det (`torch.FloatTensor` of shape `(batch_size, channels, seq_len)`:
            Logarithm of the absolute value of the determinants corresponding to the `outputs` with the `tail_bound`
            limits applied.
    """
    inside_interval_mask = (inputs >= -tail_bound) & (inputs <= tail_bound)
    outside_interval_mask = ~inside_interval_mask

    outputs = torch.zeros_like(inputs)
    log_abs_det = torch.zeros_like(inputs)
    constant = np.log(np.exp(1 - min_derivative) - 1)

    unnormalized_derivatives = nn.functional.pad(unnormalized_derivatives, pad=(1, 1))
    unnormalized_derivatives[..., 0] = constant
    unnormalized_derivatives[..., -1] = constant

    outputs[outside_interval_mask] = inputs[outside_interval_mask]
    log_abs_det[outside_interval_mask] = 0.0

    outputs[inside_interval_mask], log_abs_det[inside_interval_mask] = _rational_quadratic_spline(
        inputs=inputs[inside_interval_mask],
        unnormalized_widths=unnormalized_widths[inside_interval_mask, :],
        unnormalized_heights=unnormalized_heights[inside_interval_mask, :],
        unnormalized_derivatives=unnormalized_derivatives[inside_interval_mask, :],
        reverse=reverse,
        tail_bound=tail_bound,
        min_bin_width=min_bin_width,
        min_bin_height=min_bin_height,
        min_derivative=min_derivative,
    )
    return outputs, log_abs_det


def _rational_quadratic_spline(
    inputs,
    unnormalized_widths,
    unnormalized_heights,
    unnormalized_derivatives,
    reverse,
    tail_bound,
    min_bin_width,
    min_bin_height,
    min_derivative,
):
    """
    This transformation represents a monotonically increasing piecewise rational quadratic function. Unlike the
    function `_unconstrained_rational_quadratic_spline`, the function behaves the same across the `tail_bound`.

    Args:
        inputs (`torch.FloatTensor` of shape `(batch_size, channels, seq_len)`:
            Second half of the hidden-states input to the Vits convolutional flow module.
        unnormalized_widths (`torch.FloatTensor` of shape `(batch_size, channels, seq_len, duration_predictor_flow_bins)`):
            First `duration_predictor_flow_bins` of the hidden-states from the output of the convolution projection
            layer in the convolutional flow module
        unnormalized_heights (`torch.FloatTensor` of shape `(batch_size, channels, seq_len, duration_predictor_flow_bins)`):
            Second `duration_predictor_flow_bins` of the hidden-states from the output of the convolution projection
            layer in the convolutional flow module
        unnormalized_derivatives (`torch.FloatTensor` of shape `(batch_size, channels, seq_len, duration_predictor_flow_bins)`):
            Third `duration_predictor_flow_bins` of the hidden-states from the output of the convolution projection
            layer in the convolutional flow module
        reverse (`bool`):
            Whether the model is being run in reverse mode.
        tail_bound (`float`):
            Upper and lower limit bound for the rational quadratic function. Outside of this `tail_bound`, the
            transform behaves as an identity function.
        min_bin_width (`float`):
            Minimum bin value across the width dimension for the piecewise rational quadratic function.
        min_bin_height (`float`):
            Minimum bin value across the height dimension for the piecewise rational quadratic function.
        min_derivative (`float`):
            Minimum bin value across the derivatives for the piecewise rational quadratic function.
    Returns:
        outputs (`torch.FloatTensor` of shape `(batch_size, channels, seq_len)`:
            Hidden-states as transformed by the piecewise rational quadratic function.
        log_abs_det (`torch.FloatTensor` of shape `(batch_size, channels, seq_len)`:
            Logarithm of the absolute value of the determinants corresponding to the `outputs`.
    """
    upper_bound = tail_bound
    lower_bound = -tail_bound
    if torch.min(inputs) < lower_bound or torch.max(inputs) > upper_bound:
        raise ValueError("Input to a transform is not within its domain")

    num_bins = unnormalized_widths.shape[-1]

    if min_bin_width * num_bins > 1.0:
        raise ValueError(f"Minimal bin width {min_bin_width} too large for the number of bins {num_bins}")
    if min_bin_height * num_bins > 1.0:
        raise ValueError(f"Minimal bin height {min_bin_height} too large for the number of bins {num_bins}")

    widths = nn.functional.softmax(unnormalized_widths, dim=-1)
    widths = min_bin_width + (1 - min_bin_width * num_bins) * widths
    cumwidths = torch.cumsum(widths, dim=-1)
    cumwidths = nn.functional.pad(cumwidths, pad=(1, 0), mode="constant", value=0.0)
    cumwidths = (upper_bound - lower_bound) * cumwidths + lower_bound
    cumwidths[..., 0] = lower_bound
    cumwidths[..., -1] = upper_bound
    widths = cumwidths[..., 1:] - cumwidths[..., :-1]

    derivatives = min_derivative + nn.functional.softplus(unnormalized_derivatives)

    heights = nn.functional.softmax(unnormalized_heights, dim=-1)
    heights = min_bin_height + (1 - min_bin_height * num_bins) * heights
    cumheights = torch.cumsum(heights, dim=-1)
    cumheights = nn.functional.pad(cumheights, pad=(1, 0), mode="constant", value=0.0)
    cumheights = (upper_bound - lower_bound) * cumheights + lower_bound
    cumheights[..., 0] = lower_bound
    cumheights[..., -1] = upper_bound
    heights = cumheights[..., 1:] - cumheights[..., :-1]

    bin_locations = cumheights if reverse else cumwidths
    bin_locations[..., -1] += 1e-6
    bin_idx = torch.sum(inputs[..., None] >= bin_locations, dim=-1) - 1
    bin_idx = bin_idx[..., None]

    input_cumwidths = cumwidths.gather(-1, bin_idx)[..., 0]
    input_bin_widths = widths.gather(-1, bin_idx)[..., 0]

    input_cumheights = cumheights.gather(-1, bin_idx)[..., 0]
    delta = heights / widths
    input_delta = delta.gather(-1, bin_idx)[..., 0]

    input_derivatives = derivatives.gather(-1, bin_idx)[..., 0]
    input_derivatives_plus_one = derivatives[..., 1:].gather(-1, bin_idx)[..., 0]

    input_heights = heights.gather(-1, bin_idx)[..., 0]

    intermediate1 = input_derivatives + input_derivatives_plus_one - 2 * input_delta
    if not reverse:
        theta = (inputs - input_cumwidths) / input_bin_widths
        theta_one_minus_theta = theta * (1 - theta)

        numerator = input_heights * (input_delta * theta.pow(2) + input_derivatives * theta_one_minus_theta)
        denominator = input_delta + intermediate1 * theta_one_minus_theta
        outputs = input_cumheights + numerator / denominator

        derivative_numerator = input_delta.pow(2) * (
            input_derivatives_plus_one * theta.pow(2)
            + 2 * input_delta * theta_one_minus_theta
            + input_derivatives * (1 - theta).pow(2)
        )
        log_abs_det = torch.log(derivative_numerator) - 2 * torch.log(denominator)
        return outputs, log_abs_det
    else:
        # find the roots of a quadratic equation
        intermediate2 = inputs - input_cumheights
        intermediate3 = intermediate2 * intermediate1
        a = input_heights * (input_delta - input_derivatives) + intermediate3
        b = input_heights * input_derivatives - intermediate3
        c = -input_delta * intermediate2

        discriminant = b.pow(2) - 4 * a * c
        if not (discriminant >= 0).all():
            raise RuntimeError(f"invalid discriminant {discriminant}")

        root = (2 * c) / (-b - torch.sqrt(discriminant))
        outputs = root * input_bin_widths + input_cumwidths

        theta_one_minus_theta = root * (1 - root)
        denominator = input_delta + intermediate1 * theta_one_minus_theta
        derivative_numerator = input_delta.pow(2) * (
            input_derivatives_plus_one * root.pow(2)
            + 2 * input_delta * theta_one_minus_theta
            + input_derivatives * (1 - root).pow(2)
        )
        log_abs_det = torch.log(derivative_numerator) - 2 * torch.log(denominator)
        return outputs, -log_abs_det


class VitsWaveNet(torch.nn.Module):
    def __init__(self, config: VitsConfig, num_layers: int):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_layers = num_layers
        self.speaker_embedding_size = config.speaker_embedding_size

        self.in_layers = torch.nn.ModuleList()
        self.res_skip_layers = torch.nn.ModuleList()
        self.dropout = nn.Dropout(config.wavenet_dropout)

        if hasattr(nn.utils.parametrizations, "weight_norm"):
            weight_norm = nn.utils.parametrizations.weight_norm
        else:
            weight_norm = nn.utils.weight_norm

        if config.speaker_embedding_size != 0:
            cond_layer = torch.nn.Conv1d(config.speaker_embedding_size, 2 * config.hidden_size * num_layers, 1)
            self.cond_layer = weight_norm(cond_layer, name="weight")

        for i in range(num_layers):
            dilation = config.wavenet_dilation_rate**i
            padding = (config.wavenet_kernel_size * dilation - dilation) // 2
            in_layer = torch.nn.Conv1d(
                in_channels=config.hidden_size,
                out_channels=2 * config.hidden_size,
                kernel_size=config.wavenet_kernel_size,
                dilation=dilation,
                padding=padding,
            )
            in_layer = weight_norm(in_layer, name="weight")
            self.in_layers.append(in_layer)

            # last one is not necessary
            if i < num_layers - 1:
                res_skip_channels = 2 * config.hidden_size
            else:
                res_skip_channels = config.hidden_size

            res_skip_layer = torch.nn.Conv1d(config.hidden_size, res_skip_channels, 1)
            res_skip_layer = weight_norm(res_skip_layer, name="weight")
            self.res_skip_layers.append(res_skip_layer)

    def forward(self, inputs, padding_mask, global_conditioning=None):
        outputs = torch.zeros_like(inputs)
        num_channels_tensor = torch.IntTensor([self.hidden_size])

        if global_conditioning is not None:
            global_conditioning = self.cond_layer(global_conditioning)

        for i in range(self.num_layers):
            hidden_states = self.in_layers[i](inputs)

            if global_conditioning is not None:
                cond_offset = i * 2 * self.hidden_size
                global_states = global_conditioning[:, cond_offset : cond_offset + 2 * self.hidden_size, :]
            else:
                global_states = torch.zeros_like(hidden_states)

            acts = fused_add_tanh_sigmoid_multiply(hidden_states, global_states, num_channels_tensor[0])
            acts = self.dropout(acts)

            res_skip_acts = self.res_skip_layers[i](acts)
            if i < self.num_layers - 1:
                res_acts = res_skip_acts[:, : self.hidden_size, :]
                inputs = (inputs + res_acts) * padding_mask
                outputs = outputs + res_skip_acts[:, self.hidden_size :, :]
            else:
                outputs = outputs + res_skip_acts

        return outputs * padding_mask

    def remove_weight_norm(self):
        if self.speaker_embedding_size != 0:
            torch.nn.utils.remove_weight_norm(self.cond_layer)
        for layer in self.in_layers:
            torch.nn.utils.remove_weight_norm(layer)
        for layer in self.res_skip_layers:
            torch.nn.utils.remove_weight_norm(layer)

    def apply_weight_norm(self):
        if hasattr(nn.utils.parametrizations, "weight_norm"):
            weight_norm = nn.utils.parametrizations.weight_norm
        else:
            weight_norm = nn.utils.weight_norm

        if self.speaker_embedding_size != 0:
            weight_norm(self.cond_layer)
        for layer in self.in_layers:
            weight_norm(layer)
        for layer in self.res_skip_layers:
            weight_norm(layer)


class VitsPosteriorEncoder(nn.Module):
    def __init__(self, config: VitsConfig):
        super().__init__()
        self.out_channels = config.flow_size

        self.conv_pre = nn.Conv1d(config.spectrogram_bins, config.hidden_size, 1)
        self.wavenet = VitsWaveNet(config, num_layers=config.posterior_encoder_num_wavenet_layers)
        self.conv_proj = nn.Conv1d(config.hidden_size, self.out_channels * 2, 1)

    def forward(self, inputs, padding_mask, global_conditioning=None):
        inputs = self.conv_pre(inputs) * padding_mask
        inputs = self.wavenet(inputs, padding_mask, global_conditioning)
        stats = self.conv_proj(inputs) * padding_mask
        mean, log_stddev = torch.split(stats, self.out_channels, dim=1)
        sampled = (mean + torch.randn_like(mean) * torch.exp(log_stddev)) * padding_mask
        return sampled, mean, log_stddev

    def apply_weight_norm(self):
        self.wavenet.apply_weight_norm()

    def remove_weight_norm(self):
        self.wavenet.remove_weight_norm()

    def resize_speaker_embeddings(self, speaker_embedding_size: Optional[int] = None):
        self.wavenet.speaker_embedding_size = speaker_embedding_size
        hidden_size = self.wavenet.hidden_size
        num_layers = self.wavenet.num_layers

        cond_layer = torch.nn.Conv1d(speaker_embedding_size, 2 * hidden_size * num_layers, 1)
        self.wavenet.cond_layer = nn.utils.weight_norm(cond_layer, name="weight")
        nn.init.kaiming_normal_(self.wavenet.cond_layer.weight)
        if self.wavenet.cond_layer.bias is not None:
            k = math.sqrt(
                self.wavenet.cond_layer.groups
                / (self.wavenet.cond_layer.in_channels * self.wavenet.cond_layer.kernel_size[0])
            )
            nn.init.uniform_(self.wavenet.cond_layer.bias, a=-k, b=k)


class VitsHifiGanDiscriminatorScaleResidualBlock(nn.Module):
    def __init__(self, discriminator_scale_channels, leaky_relu_slope=0.1):
        super().__init__()
        self.leaky_relu_slope = leaky_relu_slope

        in_channels, out_channels = discriminator_scale_channels[:2]
        self.convs = nn.ModuleList([nn.Conv1d(in_channels, out_channels, 15, 1, padding=7)])

        groups = 4
        for in_channels, out_channels in zip(discriminator_scale_channels[1:-1], discriminator_scale_channels[2:]):
            self.convs.append(nn.Conv1d(in_channels, out_channels, 41, 4, groups=groups, padding=20))
            groups = groups * 4

        channel_size = discriminator_scale_channels[-1]
        self.convs.append(nn.Conv1d(channel_size, channel_size, 41, 4, groups=groups, padding=20))
        self.convs.append(nn.Conv1d(channel_size, channel_size, 5, 1, padding=2))
        self.final_conv = nn.Conv1d(channel_size, 1, 3, 1, padding=1)

    def apply_weight_norm(self):
        for layer in self.convs:
            nn.utils.weight_norm(layer)
        nn.utils.weight_norm(self.final_conv)

    def remove_weight_norm(self):
        for layer in self.convs:
            nn.utils.remove_weight_norm(layer)
        nn.utils.remove_weight_norm(self.final_conv)

    def forward(self, hidden_states):
        fmap = []

        for conv in self.convs:
            hidden_states = conv(hidden_states)
            hidden_states = nn.functional.leaky_relu(hidden_states, self.leaky_relu_slope)
            fmap.append(hidden_states)

        hidden_states = self.final_conv(hidden_states)
        fmap.append(hidden_states)
        hidden_states = torch.flatten(hidden_states, 1, -1)

        return hidden_states, fmap


class VitsHifiGanDiscriminatorPeriodResidualBlock(nn.Module):
    def __init__(self, discriminator_period_channels, period, kernel_size=5, stride=3, leaky_relu_slope=0.1):
        super().__init__()
        self.leaky_relu_slope = leaky_relu_slope
        self.period = period

        self.convs = nn.ModuleList()
        for in_channels, out_channels in zip(discriminator_period_channels[:-1], discriminator_period_channels[1:]):
            self.convs.append(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    (kernel_size, 1),
                    (stride, 1),
                    padding=(self.get_padding(kernel_size, 1), 0),
                )
            )

        channel_size = discriminator_period_channels[-1]
        self.convs.append(
            nn.Conv2d(channel_size, channel_size, (kernel_size, 1), 1, padding=(self.get_padding(kernel_size, 1), 0))
        )
        self.final_conv = nn.Conv2d(channel_size, 1, (3, 1), 1, padding=(1, 0))

    def get_padding(self, kernel_size, dilation=1):
        return (kernel_size * dilation - dilation) // 2

    def apply_weight_norm(self):
        for layer in self.convs:
            nn.utils.weight_norm(layer)
        nn.utils.weight_norm(self.final_conv)

    def remove_weight_norm(self):
        for layer in self.convs:
            nn.utils.remove_weight_norm(layer)
        nn.utils.remove_weight_norm(self.final_conv)

    def forward(self, hidden_states):
        fmap = []

        # from 1D to 2D
        batch_size, channels, length = hidden_states.shape
        if length % self.period != 0:
            # pad first
            n_pad = self.period - (length % self.period)
            hidden_states = nn.functional.pad(hidden_states, (0, n_pad), "reflect")
            length = length + n_pad
        hidden_states = hidden_states.view(batch_size, channels, length // self.period, self.period)

        for conv in self.convs:
            hidden_states = conv(hidden_states)
            hidden_states = nn.functional.leaky_relu(hidden_states, self.leaky_relu_slope)
            fmap.append(hidden_states)

        hidden_states = self.final_conv(hidden_states)
        fmap.append(hidden_states)
        hidden_states = torch.flatten(hidden_states, 1, -1)

        return hidden_states, fmap


# Copied from transformers.models.speecht5.modeling_speecht5.HifiGanResidualBlock
class HifiGanResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilation=(1, 3, 5), leaky_relu_slope=0.1):
        super().__init__()
        self.leaky_relu_slope = leaky_relu_slope

        self.convs1 = nn.ModuleList(
            [
                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    stride=1,
                    dilation=dilation[i],
                    padding=self.get_padding(kernel_size, dilation[i]),
                )
                for i in range(len(dilation))
            ]
        )
        self.convs2 = nn.ModuleList(
            [
                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    stride=1,
                    dilation=1,
                    padding=self.get_padding(kernel_size, 1),
                )
                for _ in range(len(dilation))
            ]
        )

    def get_padding(self, kernel_size, dilation=1):
        return (kernel_size * dilation - dilation) // 2

    def apply_weight_norm(self):
        for layer in self.convs1:
            nn.utils.weight_norm(layer)
        for layer in self.convs2:
            nn.utils.weight_norm(layer)

    def remove_weight_norm(self):
        for layer in self.convs1:
            nn.utils.remove_weight_norm(layer)
        for layer in self.convs2:
            nn.utils.remove_weight_norm(layer)

    def forward(self, hidden_states):
        for conv1, conv2 in zip(self.convs1, self.convs2):
            residual = hidden_states
            hidden_states = nn.functional.leaky_relu(hidden_states, self.leaky_relu_slope)
            hidden_states = conv1(hidden_states)
            hidden_states = nn.functional.leaky_relu(hidden_states, self.leaky_relu_slope)
            hidden_states = conv2(hidden_states)
            hidden_states = hidden_states + residual
        return hidden_states


class VitsHifiGan(nn.Module):
    def __init__(self, config: VitsConfig):
        super().__init__()
        self.config = config
        self.num_kernels = len(config.resblock_kernel_sizes)
        self.num_upsamples = len(config.upsample_rates)
        self.conv_pre = nn.Conv1d(
            config.flow_size,
            config.upsample_initial_channel,
            kernel_size=7,
            stride=1,
            padding=3,
        )

        self.upsampler = nn.ModuleList()
        for i, (upsample_rate, kernel_size) in enumerate(zip(config.upsample_rates, config.upsample_kernel_sizes)):
            self.upsampler.append(
                nn.ConvTranspose1d(
                    config.upsample_initial_channel // (2**i),
                    config.upsample_initial_channel // (2 ** (i + 1)),
                    kernel_size=kernel_size,
                    stride=upsample_rate,
                    padding=(kernel_size - upsample_rate) // 2,
                )
            )

        self.resblocks = nn.ModuleList()
        for i in range(len(self.upsampler)):
            channels = config.upsample_initial_channel // (2 ** (i + 1))
            for kernel_size, dilation in zip(config.resblock_kernel_sizes, config.resblock_dilation_sizes):
                self.resblocks.append(HifiGanResidualBlock(channels, kernel_size, dilation, config.leaky_relu_slope))

        self.conv_post = nn.Conv1d(channels, 1, kernel_size=7, stride=1, padding=3, bias=False)

        if config.speaker_embedding_size != 0:
            self.cond = nn.Conv1d(config.speaker_embedding_size, config.upsample_initial_channel, 1)

    def resize_speaker_embedding(self, speaker_embedding_size):
        self.config.speaker_embedding_size = speaker_embedding_size
        self.cond = nn.Conv1d(speaker_embedding_size, self.config.upsample_initial_channel, 1)
        nn.init.kaiming_normal_(self.cond.weight)
        if self.cond.bias is not None:
            k = math.sqrt(self.cond.groups / (self.cond.in_channels * self.cond.kernel_size[0]))
            nn.init.uniform_(self.cond.bias, a=-k, b=k)

    def apply_weight_norm(self):
        for layer in self.upsampler:
            nn.utils.weight_norm(layer)
        for layer in self.resblocks:
            layer.apply_weight_norm()

    def remove_weight_norm(self):
        for layer in self.upsampler:
            nn.utils.remove_weight_norm(layer)
        for layer in self.resblocks:
            layer.remove_weight_norm()

    def forward(
        self, spectrogram: torch.FloatTensor, global_conditioning: Optional[torch.FloatTensor] = None
    ) -> torch.FloatTensor:
        r"""
        Converts a spectrogram into a speech waveform.

        Args:
            spectrogram (`torch.FloatTensor` of shape `(batch_size, config.spectrogram_bins, sequence_length)`):
                Tensor containing the spectrograms.
            global_conditioning (`torch.FloatTensor` of shape `(batch_size, config.speaker_embedding_size, 1)`, *optional*):
                Tensor containing speaker embeddings, for multispeaker models.

        Returns:
            `torch.FloatTensor`: Tensor of shape shape `(batch_size, 1, num_frames)` containing the speech waveform.
        """
        hidden_states = self.conv_pre(spectrogram)

        if global_conditioning is not None:
            hidden_states = hidden_states + self.cond(global_conditioning)

        for i in range(self.num_upsamples):
            hidden_states = nn.functional.leaky_relu(hidden_states, self.config.leaky_relu_slope)
            hidden_states = self.upsampler[i](hidden_states)

            res_state = self.resblocks[i * self.num_kernels](hidden_states)
            for j in range(1, self.num_kernels):
                res_state += self.resblocks[i * self.num_kernels + j](hidden_states)
            hidden_states = res_state / self.num_kernels

        hidden_states = nn.functional.leaky_relu(hidden_states)
        hidden_states = self.conv_post(hidden_states)
        waveform = torch.tanh(hidden_states)
        return waveform


class VitsResidualCouplingLayer(nn.Module):
    def __init__(self, config: VitsConfig):
        super().__init__()
        self.half_channels = config.flow_size // 2

        self.conv_pre = nn.Conv1d(self.half_channels, config.hidden_size, 1)
        self.wavenet = VitsWaveNet(config, num_layers=config.prior_encoder_num_wavenet_layers)
        self.conv_post = nn.Conv1d(config.hidden_size, self.half_channels, 1)

    def forward(self, inputs, padding_mask, global_conditioning=None, reverse=False):
        first_half, second_half = torch.split(inputs, [self.half_channels] * 2, dim=1)
        hidden_states = self.conv_pre(first_half) * padding_mask
        hidden_states = self.wavenet(hidden_states, padding_mask, global_conditioning)
        mean = self.conv_post(hidden_states) * padding_mask
        log_stddev = torch.zeros_like(mean)

        if not reverse:
            second_half = mean + second_half * torch.exp(log_stddev) * padding_mask
            outputs = torch.cat([first_half, second_half], dim=1)
            log_determinant = torch.sum(log_stddev, [1, 2])
            return outputs, log_determinant
        else:
            second_half = (second_half - mean) * torch.exp(-log_stddev) * padding_mask
            outputs = torch.cat([first_half, second_half], dim=1)
            return outputs, None

    def apply_weight_norm(self):
        nn.utils.weight_norm(self.conv_pre)
        self.wavenet.apply_weight_norm()
        nn.utils.weight_norm(self.conv_post)

    def remove_weight_norm(self):
        nn.utils.remove_weight_norm(self.conv_pre)
        self.wavenet.remove_weight_norm()
        nn.utils.remove_weight_norm(self.conv_post)


class VitsResidualCouplingBlock(nn.Module):
    def __init__(self, config: VitsConfig):
        super().__init__()
        self.flows = nn.ModuleList()
        for _ in range(config.prior_encoder_num_flows):
            self.flows.append(VitsResidualCouplingLayer(config))

    def forward(self, inputs, padding_mask, global_conditioning=None, reverse=False):
        if not reverse:
            for flow in self.flows:
                inputs, _ = flow(inputs, padding_mask, global_conditioning)
                inputs = torch.flip(inputs, [1])
        else:
            for flow in reversed(self.flows):
                inputs = torch.flip(inputs, [1])
                inputs, _ = flow(inputs, padding_mask, global_conditioning, reverse=True)
        return inputs

    def apply_weight_norm(self):
        for flow in self.flows:
            flow.apply_weight_norm()

    def remove_weight_norm(self):
        for flow in self.flows:
            flow.remove_weight_norm()

    def resize_speaker_embeddings(self, speaker_embedding_size: Optional[int] = None):
        for flow in self.flows:
            flow.wavenet.speaker_embedding_size = speaker_embedding_size
            hidden_size = flow.wavenet.hidden_size
            num_layers = flow.wavenet.num_layers

            cond_layer = torch.nn.Conv1d(speaker_embedding_size, 2 * hidden_size * num_layers, 1)
            flow.wavenet.cond_layer = nn.utils.weight_norm(cond_layer, name="weight")


class VitsDilatedDepthSeparableConv(nn.Module):
    def __init__(self, config: VitsConfig, dropout_rate=0.0):
        super().__init__()
        kernel_size = config.duration_predictor_kernel_size
        channels = config.hidden_size
        self.num_layers = config.depth_separable_num_layers

        self.dropout = nn.Dropout(dropout_rate)
        self.convs_dilated = nn.ModuleList()
        self.convs_pointwise = nn.ModuleList()
        self.norms_1 = nn.ModuleList()
        self.norms_2 = nn.ModuleList()
        for i in range(self.num_layers):
            dilation = kernel_size**i
            padding = (kernel_size * dilation - dilation) // 2
            self.convs_dilated.append(
                nn.Conv1d(
                    in_channels=channels,
                    out_channels=channels,
                    kernel_size=kernel_size,
                    groups=channels,
                    dilation=dilation,
                    padding=padding,
                )
            )
            self.convs_pointwise.append(nn.Conv1d(channels, channels, 1))
            self.norms_1.append(nn.LayerNorm(channels))
            self.norms_2.append(nn.LayerNorm(channels))

    def forward(self, inputs, padding_mask, global_conditioning=None):
        if global_conditioning is not None:
            inputs = inputs + global_conditioning

        for i in range(self.num_layers):
            hidden_states = self.convs_dilated[i](inputs * padding_mask)
            hidden_states = self.norms_1[i](hidden_states.transpose(1, -1)).transpose(1, -1)
            hidden_states = nn.functional.gelu(hidden_states)
            hidden_states = self.convs_pointwise[i](hidden_states)
            hidden_states = self.norms_2[i](hidden_states.transpose(1, -1)).transpose(1, -1)
            hidden_states = nn.functional.gelu(hidden_states)
            hidden_states = self.dropout(hidden_states)
            inputs = inputs + hidden_states

        return inputs * padding_mask


class VitsConvFlow(nn.Module):
    def __init__(self, config: VitsConfig):
        super().__init__()
        self.filter_channels = config.hidden_size
        self.half_channels = config.depth_separable_channels // 2
        self.num_bins = config.duration_predictor_flow_bins
        self.tail_bound = config.duration_predictor_tail_bound

        self.conv_pre = nn.Conv1d(self.half_channels, self.filter_channels, 1)
        self.conv_dds = VitsDilatedDepthSeparableConv(config)
        self.conv_proj = nn.Conv1d(self.filter_channels, self.half_channels * (self.num_bins * 3 - 1), 1)

    def forward(self, inputs, padding_mask, global_conditioning=None, reverse=False):
        first_half, second_half = torch.split(inputs, [self.half_channels] * 2, dim=1)

        hidden_states = self.conv_pre(first_half)
        hidden_states = self.conv_dds(hidden_states, padding_mask, global_conditioning)
        hidden_states = self.conv_proj(hidden_states) * padding_mask

        batch_size, channels, length = first_half.shape
        hidden_states = hidden_states.reshape(batch_size, channels, -1, length).permute(0, 1, 3, 2)

        unnormalized_widths = hidden_states[..., : self.num_bins] / math.sqrt(self.filter_channels)
        unnormalized_heights = hidden_states[..., self.num_bins : 2 * self.num_bins] / math.sqrt(self.filter_channels)
        unnormalized_derivatives = hidden_states[..., 2 * self.num_bins :]

        second_half, log_abs_det = _unconstrained_rational_quadratic_spline(
            second_half,
            unnormalized_widths,
            unnormalized_heights,
            unnormalized_derivatives,
            reverse=reverse,
            tail_bound=self.tail_bound,
        )

        outputs = torch.cat([first_half, second_half], dim=1) * padding_mask
        if not reverse:
            log_determinant = torch.sum(log_abs_det * padding_mask, [1, 2])
            return outputs, log_determinant
        else:
            return outputs, None


class VitsElementwiseAffine(nn.Module):
    def __init__(self, config: VitsConfig):
        super().__init__()
        self.channels = config.depth_separable_channels
        self.translate = nn.Parameter(torch.zeros(self.channels, 1))
        self.log_scale = nn.Parameter(torch.zeros(self.channels, 1))

    def forward(self, inputs, padding_mask, global_conditioning=None, reverse=False):
        if not reverse:
            outputs = self.translate + torch.exp(self.log_scale) * inputs
            outputs = outputs * padding_mask
            log_determinant = torch.sum(self.log_scale * padding_mask, [1, 2])
            return outputs, log_determinant
        else:
            outputs = (inputs - self.translate) * torch.exp(-self.log_scale) * padding_mask
            return outputs, None


class VitsStochasticDurationPredictor(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.speaker_embedding_size
        filter_channels = config.hidden_size

        self.conv_pre = nn.Conv1d(filter_channels, filter_channels, 1)
        self.conv_proj = nn.Conv1d(filter_channels, filter_channels, 1)
        self.conv_dds = VitsDilatedDepthSeparableConv(
            config,
            dropout_rate=config.duration_predictor_dropout,
        )

        if embed_dim != 0:
            self.cond = nn.Conv1d(embed_dim, filter_channels, 1)

        self.flows = nn.ModuleList()
        self.flows.append(VitsElementwiseAffine(config))
        for _ in range(config.duration_predictor_num_flows):
            self.flows.append(VitsConvFlow(config))

        self.post_conv_pre = nn.Conv1d(1, filter_channels, 1)
        self.post_conv_proj = nn.Conv1d(filter_channels, filter_channels, 1)
        self.post_conv_dds = VitsDilatedDepthSeparableConv(
            config,
            dropout_rate=config.duration_predictor_dropout,
        )

        self.post_flows = nn.ModuleList()
        self.post_flows.append(VitsElementwiseAffine(config))
        for _ in range(config.duration_predictor_num_flows):
            self.post_flows.append(VitsConvFlow(config))

        self.filter_channels = filter_channels

    def resize_speaker_embeddings(self, speaker_embedding_size):
        self.cond = nn.Conv1d(speaker_embedding_size, self.filter_channels, 1)

    def forward(self, inputs, padding_mask, global_conditioning=None, durations=None, reverse=False, noise_scale=1.0):
        inputs = torch.detach(inputs)
        inputs = self.conv_pre(inputs)

        if global_conditioning is not None:
            global_conditioning = torch.detach(global_conditioning)
            inputs = inputs + self.cond(global_conditioning)

        inputs = self.conv_dds(inputs, padding_mask)
        inputs = self.conv_proj(inputs) * padding_mask

        if not reverse:
            hidden_states = self.post_conv_pre(durations)
            hidden_states = self.post_conv_dds(hidden_states, padding_mask)
            hidden_states = self.post_conv_proj(hidden_states) * padding_mask

            random_posterior = (
                torch.randn(durations.size(0), 2, durations.size(2)).to(device=inputs.device, dtype=inputs.dtype)
                * padding_mask
            )
            latents_posterior = random_posterior

            latents_posterior, log_determinant = self.post_flows[0](
                latents_posterior, padding_mask, global_conditioning=inputs + hidden_states
            )
            log_determinant_posterior_sum = log_determinant

            for flow in self.post_flows[1:]:
                latents_posterior, log_determinant = flow(
                    latents_posterior, padding_mask, global_conditioning=inputs + hidden_states
                )
                latents_posterior = torch.flip(latents_posterior, [1])
                log_determinant_posterior_sum += log_determinant

            first_half, second_half = torch.split(latents_posterior, [1, 1], dim=1)

            log_determinant_posterior_sum += torch.sum(
                (nn.functional.logsigmoid(first_half) + nn.functional.logsigmoid(-first_half)) * padding_mask, [1, 2]
            )
            logq = (
                torch.sum(-0.5 * (math.log(2 * math.pi) + (random_posterior**2)) * padding_mask, [1, 2])
                - log_determinant_posterior_sum
            )

            first_half = (durations - torch.sigmoid(first_half)) * padding_mask
            first_half = torch.log(torch.clamp_min(first_half, 1e-5)) * padding_mask
            log_determinant_sum = torch.sum(-first_half, [1, 2])

            latents = torch.cat([first_half, second_half], dim=1)
            latents, log_determinant = self.flows[0](latents, padding_mask, global_conditioning=inputs)

            log_determinant_sum += log_determinant
            for flow in self.flows[1:]:
                latents, log_determinant = flow(latents, padding_mask, global_conditioning=inputs)
                latents = torch.flip(latents, [1])
                log_determinant_sum += log_determinant

            nll = torch.sum(0.5 * (math.log(2 * math.pi) + (latents**2)) * padding_mask, [1, 2]) - log_determinant_sum
            return nll + logq
        else:
            flows = list(reversed(self.flows))
            flows = flows[:-2] + [flows[-1]]  # remove a useless vflow

            latents = (
                torch.randn(inputs.size(0), 2, inputs.size(2)).to(device=inputs.device, dtype=inputs.dtype)
                * noise_scale
            )
            for flow in flows:
                latents = torch.flip(latents, [1])
                latents, _ = flow(latents, padding_mask, global_conditioning=inputs, reverse=True)

            log_duration, _ = torch.split(latents, [1, 1], dim=1)
            return log_duration


class VitsDurationPredictor(nn.Module):
    def __init__(self, config):
        super().__init__()
        kernel_size = config.duration_predictor_kernel_size
        filter_channels = config.duration_predictor_filter_channels

        self.dropout = nn.Dropout(config.duration_predictor_dropout)
        self.conv_1 = nn.Conv1d(config.hidden_size, filter_channels, kernel_size, padding=kernel_size // 2)
        self.norm_1 = nn.LayerNorm(filter_channels, eps=config.layer_norm_eps)
        self.conv_2 = nn.Conv1d(filter_channels, filter_channels, kernel_size, padding=kernel_size // 2)
        self.norm_2 = nn.LayerNorm(filter_channels, eps=config.layer_norm_eps)
        self.proj = nn.Conv1d(filter_channels, 1, 1)

        if config.speaker_embedding_size != 0:
            self.cond = nn.Conv1d(config.speaker_embedding_size, config.hidden_size, 1)

        self.hidden_size = config.hidden_size

    def resize_speaker_embeddings(self, speaker_embedding_size):
        self.cond = nn.Conv1d(speaker_embedding_size, self.hidden_size, 1)

    def forward(self, inputs, padding_mask, global_conditioning=None):
        inputs = torch.detach(inputs)

        if global_conditioning is not None:
            global_conditioning = torch.detach(global_conditioning)
            inputs = inputs + self.cond(global_conditioning)

        inputs = self.conv_1(inputs * padding_mask)
        inputs = torch.relu(inputs)
        inputs = self.norm_1(inputs.transpose(1, -1)).transpose(1, -1)
        inputs = self.dropout(inputs)

        inputs = self.conv_2(inputs * padding_mask)
        inputs = torch.relu(inputs)
        inputs = self.norm_2(inputs.transpose(1, -1)).transpose(1, -1)
        inputs = self.dropout(inputs)

        inputs = self.proj(inputs * padding_mask)
        return inputs * padding_mask


class VitsAttention(nn.Module):
    """Multi-headed attention with relative positional representation."""

    def __init__(self, config: VitsConfig):
        super().__init__()
        self.embed_dim = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.dropout = config.attention_dropout
        self.window_size = config.window_size

        self.head_dim = self.embed_dim // self.num_heads
        self.scaling = self.head_dim**-0.5

        if (self.head_dim * self.num_heads) != self.embed_dim:
            raise ValueError(
                f"hidden_size must be divisible by num_attention_heads (got `hidden_size`: {self.embed_dim}"
                f" and `num_attention_heads`: {self.num_heads})."
            )

        self.k_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=config.use_bias)
        self.v_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=config.use_bias)
        self.q_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=config.use_bias)
        self.out_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=config.use_bias)

        if self.window_size:
            self.emb_rel_k = nn.Parameter(torch.randn(1, self.window_size * 2 + 1, self.head_dim) * self.scaling)
            self.emb_rel_v = nn.Parameter(torch.randn(1, self.window_size * 2 + 1, self.head_dim) * self.scaling)

    def _shape(self, tensor: torch.Tensor, seq_len: int, bsz: int):
        return tensor.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2).contiguous()

    def forward(
        self,
        hidden_states: torch.Tensor,
        key_value_states: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        layer_head_mask: Optional[torch.Tensor] = None,
        output_attentions: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Input shape: Batch x Time x Channel"""

        # if key_value_states are provided this layer is used as a cross-attention layer
        # for the decoder

        bsz, tgt_len, _ = hidden_states.size()

        # get query proj
        query_states = self.q_proj(hidden_states) * self.scaling

        # self_attention
        key_states = self._shape(self.k_proj(hidden_states), -1, bsz)
        value_states = self._shape(self.v_proj(hidden_states), -1, bsz)

        proj_shape = (bsz * self.num_heads, -1, self.head_dim)
        query_states = self._shape(query_states, tgt_len, bsz).view(*proj_shape)
        key_states = key_states.view(*proj_shape)
        value_states = value_states.view(*proj_shape)

        src_len = key_states.size(1)
        attn_weights = torch.bmm(query_states, key_states.transpose(1, 2))

        if attn_weights.size() != (bsz * self.num_heads, tgt_len, src_len):
            raise ValueError(
                f"Attention weights should be of size {(bsz * self.num_heads, tgt_len, src_len)}, but is"
                f" {attn_weights.size()}"
            )

        if self.window_size is not None:
            key_relative_embeddings = self._get_relative_embeddings(self.emb_rel_k, src_len)
            relative_logits = torch.matmul(query_states, key_relative_embeddings.transpose(-2, -1))
            rel_pos_bias = self._relative_position_to_absolute_position(relative_logits)
            attn_weights += rel_pos_bias

        if attention_mask is not None:
            if attention_mask.size() != (bsz, 1, tgt_len, src_len):
                raise ValueError(
                    f"Attention mask should be of size {(bsz, 1, tgt_len, src_len)}, but is {attention_mask.size()}"
                )
            attn_weights = attn_weights.view(bsz, self.num_heads, tgt_len, src_len) + attention_mask
            attn_weights = attn_weights.view(bsz * self.num_heads, tgt_len, src_len)

        attn_weights = nn.functional.softmax(attn_weights, dim=-1)

        if layer_head_mask is not None:
            if layer_head_mask.size() != (self.num_heads,):
                raise ValueError(
                    f"Head mask for a single layer should be of size {(self.num_heads,)}, but is"
                    f" {layer_head_mask.size()}"
                )
            attn_weights = layer_head_mask.view(1, -1, 1, 1) * attn_weights.view(bsz, self.num_heads, tgt_len, src_len)
            attn_weights = attn_weights.view(bsz * self.num_heads, tgt_len, src_len)

        if output_attentions:
            # this operation is a bit awkward, but it's required to
            # make sure that attn_weights keeps its gradient.
            # In order to do so, attn_weights have to be reshaped
            # twice and have to be reused in the following
            attn_weights_reshaped = attn_weights.view(bsz, self.num_heads, tgt_len, src_len)
            attn_weights = attn_weights_reshaped.view(bsz * self.num_heads, tgt_len, src_len)
        else:
            attn_weights_reshaped = None

        attn_probs = nn.functional.dropout(attn_weights, p=self.dropout, training=self.training)

        attn_output = torch.bmm(attn_probs, value_states)

        if attn_output.size() != (bsz * self.num_heads, tgt_len, self.head_dim):
            raise ValueError(
                f"`attn_output` should be of size {(bsz, self.num_heads, tgt_len, self.head_dim)}, but is"
                f" {attn_output.size()}"
            )

        if self.window_size is not None:
            value_relative_embeddings = self._get_relative_embeddings(self.emb_rel_v, src_len)
            relative_weights = self._absolute_position_to_relative_position(attn_probs)
            rel_pos_bias = torch.matmul(relative_weights, value_relative_embeddings)
            attn_output += rel_pos_bias

        attn_output = attn_output.view(bsz, self.num_heads, tgt_len, self.head_dim)
        attn_output = attn_output.transpose(1, 2)

        # Use the `embed_dim` from the config (stored in the class) rather than `hidden_state` because `attn_output` can be
        # partitioned aross GPUs when using tensor-parallelism.
        attn_output = attn_output.reshape(bsz, tgt_len, self.embed_dim)

        attn_output = self.out_proj(attn_output)

        return attn_output, attn_weights_reshaped

    def _get_relative_embeddings(self, relative_embeddings, length):
        pad_length = max(length - (self.window_size + 1), 0)
        if pad_length > 0:
            relative_embeddings = nn.functional.pad(relative_embeddings, [0, 0, pad_length, pad_length, 0, 0])

        slice_start_position = max((self.window_size + 1) - length, 0)
        slice_end_position = slice_start_position + 2 * length - 1
        return relative_embeddings[:, slice_start_position:slice_end_position]

    def _relative_position_to_absolute_position(self, x):
        batch_heads, length, _ = x.size()

        # Concat columns of pad to shift from relative to absolute indexing.
        x = nn.functional.pad(x, [0, 1, 0, 0, 0, 0])

        # Concat extra elements so to add up to shape (len+1, 2*len-1).
        x_flat = x.view([batch_heads, length * 2 * length])
        x_flat = nn.functional.pad(x_flat, [0, length - 1, 0, 0])

        # Reshape and slice out the padded elements.
        x_final = x_flat.view([batch_heads, length + 1, 2 * length - 1])
        x_final = x_final[:, :length, length - 1 :]
        return x_final

    def _absolute_position_to_relative_position(self, x):
        batch_heads, length, _ = x.size()

        # Pad along column
        x = nn.functional.pad(x, [0, length - 1, 0, 0, 0, 0])
        x_flat = x.view([batch_heads, length**2 + length * (length - 1)])

        # Add 0's in the beginning that will skew the elements after reshape
        x_flat = nn.functional.pad(x_flat, [length, 0, 0, 0])
        x_final = x_flat.view([batch_heads, length, 2 * length])[:, :, 1:]
        return x_final


class VitsFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.conv_1 = nn.Conv1d(config.hidden_size, config.ffn_dim, config.ffn_kernel_size)
        self.conv_2 = nn.Conv1d(config.ffn_dim, config.hidden_size, config.ffn_kernel_size)
        self.dropout = nn.Dropout(config.activation_dropout)

        if isinstance(config.hidden_act, str):
            self.act_fn = ACT2FN[config.hidden_act]
        else:
            self.act_fn = config.hidden_act

        if config.ffn_kernel_size > 1:
            pad_left = (config.ffn_kernel_size - 1) // 2
            pad_right = config.ffn_kernel_size // 2
            self.padding = [pad_left, pad_right, 0, 0, 0, 0]
        else:
            self.padding = None

    def forward(self, hidden_states, padding_mask):
        hidden_states = hidden_states.permute(0, 2, 1)
        padding_mask = padding_mask.permute(0, 2, 1)

        hidden_states = hidden_states * padding_mask
        if self.padding is not None:
            hidden_states = nn.functional.pad(hidden_states, self.padding)

        hidden_states = self.conv_1(hidden_states)
        hidden_states = self.act_fn(hidden_states)
        hidden_states = self.dropout(hidden_states)

        hidden_states = hidden_states * padding_mask
        if self.padding is not None:
            hidden_states = nn.functional.pad(hidden_states, self.padding)

        hidden_states = self.conv_2(hidden_states)
        hidden_states = hidden_states * padding_mask

        hidden_states = hidden_states.permute(0, 2, 1)
        return hidden_states


class VitsEncoderLayer(nn.Module):
    def __init__(self, config: VitsConfig):
        super().__init__()
        self.attention = VitsAttention(config)
        self.dropout = nn.Dropout(config.hidden_dropout)
        self.layer_norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.feed_forward = VitsFeedForward(config)
        self.final_layer_norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        padding_mask: torch.FloatTensor,
        attention_mask: Optional[torch.Tensor] = None,
        output_attentions: bool = False,
    ):
        residual = hidden_states
        hidden_states, attn_weights = self.attention(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
        )

        hidden_states = self.dropout(hidden_states)
        hidden_states = self.layer_norm(residual + hidden_states)

        residual = hidden_states
        hidden_states = self.feed_forward(hidden_states, padding_mask)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.final_layer_norm(residual + hidden_states)

        outputs = (hidden_states,)

        if output_attentions:
            outputs += (attn_weights,)

        return outputs


class VitsEncoder(nn.Module):
    def __init__(self, config: VitsConfig):
        super().__init__()
        self.config = config
        self.layers = nn.ModuleList([VitsEncoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.gradient_checkpointing = False
        self.layerdrop = config.layerdrop

    def forward(
        self,
        hidden_states: torch.FloatTensor,
        padding_mask: torch.FloatTensor,
        attention_mask: Optional[torch.Tensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple, BaseModelOutput]:
        all_hidden_states = () if output_hidden_states else None
        all_self_attentions = () if output_attentions else None

        # expand attention_mask
        if attention_mask is not None:
            # [bsz, seq_len] -> [bsz, 1, tgt_seq_len, src_seq_len]
            attention_mask = _prepare_4d_attention_mask(attention_mask, hidden_states.dtype)

        hidden_states = hidden_states * padding_mask

        deepspeed_zero3_is_enabled = is_deepspeed_zero3_enabled()

        for encoder_layer in self.layers:
            if output_hidden_states:
                all_hidden_states = all_hidden_states + (hidden_states,)

            # add LayerDrop (see https://arxiv.org/abs/1909.11556 for description)
            dropout_probability = np.random.uniform(0, 1)

            skip_the_layer = self.training and (dropout_probability < self.layerdrop)
            if not skip_the_layer or deepspeed_zero3_is_enabled:
                # under deepspeed zero3 all gpus must run in sync
                if self.gradient_checkpointing and self.training:
                    layer_outputs = self._gradient_checkpointing_func(
                        encoder_layer.__call__,
                        hidden_states,
                        padding_mask,
                        attention_mask,
                        output_attentions,
                    )
                else:
                    layer_outputs = encoder_layer(
                        hidden_states,
                        attention_mask=attention_mask,
                        padding_mask=padding_mask,
                        output_attentions=output_attentions,
                    )
                hidden_states = layer_outputs[0]

            if skip_the_layer:
                layer_outputs = (None, None)

            if output_attentions:
                all_self_attentions = all_self_attentions + (layer_outputs[1],)

        hidden_states = hidden_states * padding_mask

        if output_hidden_states:
            all_hidden_states = all_hidden_states + (hidden_states,)

        if not return_dict:
            return tuple(v for v in [hidden_states, all_hidden_states, all_self_attentions] if v is not None)

        return BaseModelOutput(
            last_hidden_state=hidden_states,
            hidden_states=all_hidden_states,
            attentions=all_self_attentions,
        )


class VitsTextEncoder(nn.Module):
    """
    Transformer encoder that uses relative positional representation instead of absolute positional encoding.
    """

    def __init__(self, config: VitsConfig):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, config.pad_token_id)
        self.encoder = VitsEncoder(config)
        self.project = nn.Conv1d(config.hidden_size, config.flow_size * 2, kernel_size=1)

    def get_input_embeddings(self):
        return self.embed_tokens

    def set_input_embeddings(self, value):
        self.embed_tokens = value

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.FloatTensor,
        attention_mask: Optional[torch.Tensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = True,
    ) -> Union[Tuple[torch.Tensor], VitsTextEncoderOutput]:
        hidden_states = self.embed_tokens(input_ids) * math.sqrt(self.config.hidden_size)

        encoder_outputs = self.encoder(
            hidden_states=hidden_states,
            padding_mask=padding_mask,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        last_hidden_state = encoder_outputs[0] if not return_dict else encoder_outputs.last_hidden_state

        stats = self.project(last_hidden_state.transpose(1, 2)).transpose(1, 2) * padding_mask
        prior_means, prior_log_variances = torch.split(stats, self.config.flow_size, dim=2)

        if not return_dict:
            outputs = (last_hidden_state, prior_means, prior_log_variances) + encoder_outputs[1:]
            return outputs

        return VitsTextEncoderOutput(
            last_hidden_state=last_hidden_state,
            prior_means=prior_means,
            prior_log_variances=prior_log_variances,
            hidden_states=encoder_outputs.hidden_states,
            attentions=encoder_outputs.attentions,
        )


class VitsPreTrainedModel(PreTrainedModel):
    """
    An abstract class to handle weights initialization and a simple interface for downloading and loading pretrained
    models.
    """

    config_class = VitsConfig
    base_model_prefix = "vits"
    main_input_name = "input_ids"
    supports_gradient_checkpointing = True

    def _init_weights(self, module):
        """Initialize the weights"""
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
        elif isinstance(module, nn.Conv1d):
            nn.init.kaiming_normal_(module.weight)
            if module.bias is not None:
                k = math.sqrt(module.groups / (module.in_channels * module.kernel_size[0]))
                nn.init.uniform_(module.bias, a=-k, b=k)
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()


VITS_START_DOCSTRING = r"""
    This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the
    library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads
    etc.)

    This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.
    Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage
    and behavior.

    Parameters:
        config ([`VitsConfig`]):
            Model configuration class with all the parameters of the model. Initializing with a config file does not
            load the weights associated with the model, only the configuration. Check out the
            [`~PreTrainedModel.from_pretrained`] method to load the model weights.
"""


VITS_INPUTS_DOCSTRING = r"""
    Args:
        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):
            Indices of input sequence tokens in the vocabulary. Padding will be ignored by default should you provide
            it.

            Indices can be obtained using [`AutoTokenizer`]. See [`PreTrainedTokenizer.encode`] and
            [`PreTrainedTokenizer.__call__`] for details.

            [What are input IDs?](../glossary#input-ids)
        attention_mask (`torch.Tensor` of shape `(batch_size, sequence_length)`, *optional*):
            Mask to avoid performing convolution and attention on padding token indices. Mask values selected in `[0,
            1]`:

            - 1 for tokens that are **not masked**,
            - 0 for tokens that are **masked**.

            [What are attention masks?](../glossary#attention-mask)
        speaker_id (`int` or `Tuple[int,..]` or `List[int]` or `np.ndarray`, *optional*):
            Which speaker embedding to use. Only used for multispeaker models. Only used for multi-speaker models. You
            can use a single `speaker_id` or a list/tuple/numpy array, provided it is as long as the batch size.
        output_attentions (`bool`, *optional*):
            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned
            tensors for more detail.
        output_hidden_states (`bool`, *optional*):
            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for
            more detail.
        return_dict (`bool`, *optional*):
            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.
"""


@add_start_docstrings(
    "The complete VITS model, for text-to-speech synthesis.",
    VITS_START_DOCSTRING,
)
class VitsModel(VitsPreTrainedModel):
    def __init__(self, config: VitsConfig):
        super().__init__(config)
        self.config = config
        self.text_encoder = VitsTextEncoder(config)
        self.flow = VitsResidualCouplingBlock(config)
        self.decoder = VitsHifiGan(config)

        if config.use_stochastic_duration_prediction:
            self.duration_predictor = VitsStochasticDurationPredictor(config)
        else:
            self.duration_predictor = VitsDurationPredictor(config)

        if config.num_speakers > 1:
            self.embed_speaker = nn.Embedding(config.num_speakers, config.speaker_embedding_size)

        # This is used only for training.
        self.posterior_encoder = VitsPosteriorEncoder(config)

        # These parameters control the synthesised speech properties
        self.speaking_rate = config.speaking_rate
        self.noise_scale = config.noise_scale
        self.noise_scale_duration = config.noise_scale_duration

        # Initialize weights and apply final processing
        self.post_init()

    def get_encoder(self):
        return self.text_encoder

    @add_start_docstrings_to_model_forward(VITS_INPUTS_DOCSTRING)
    @replace_return_docstrings(output_type=VitsModelOutput, config_class=_CONFIG_FOR_DOC)
    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        speaker_id: Optional[int] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        labels: Optional[torch.FloatTensor] = None,
    ) -> Union[Tuple[Any], VitsModelOutput]:
        r"""
        labels (`torch.FloatTensor` of shape `(batch_size, config.spectrogram_bins, sequence_length)`, *optional*):
            Float values of target spectrogram. Timesteps set to `-100.0` are ignored (masked) for the loss
            computation.

        Returns:

        Example:

        ```python
        >>> from transformers import VitsTokenizer, VitsModel, set_seed
        >>> import torch

        >>> tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-eng")
        >>> model = VitsModel.from_pretrained("facebook/mms-tts-eng")

        >>> inputs = tokenizer(text="Hello - my dog is cute", return_tensors="pt")

        >>> set_seed(555)  # make deterministic

        >>> with torch.no_grad():
        ...     outputs = model(inputs["input_ids"])
        >>> outputs.waveform.shape
        torch.Size([1, 45824])
        ```
        """
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if attention_mask is not None:
            input_padding_mask = attention_mask.unsqueeze(-1).float()
        else:
            input_padding_mask = torch.ones_like(input_ids).unsqueeze(-1).float()

        if self.config.num_speakers > 1 and speaker_id is not None:
            if isinstance(speaker_id, int):
                speaker_id = torch.full(size=(1,), fill_value=speaker_id, device=self.device)
            elif isinstance(speaker_id, (list, tuple, np.ndarray)):
                speaker_id = torch.tensor(speaker_id, device=self.device)

            if not ((0 <= speaker_id).all() and (speaker_id < self.config.num_speakers).all()).item():
                raise ValueError(f"Set `speaker_id` in the range 0-{self.config.num_speakers - 1}.")
            if not (len(speaker_id) == 1 or len(speaker_id == len(input_ids))):
                raise ValueError(
                    f"You passed {len(speaker_id)} `speaker_id` but you should either pass one speaker id or `batch_size` `speaker_id`."
                )

            speaker_embeddings = self.embed_speaker(speaker_id).unsqueeze(-1)
        else:
            speaker_embeddings = None

        if labels is not None:
            raise NotImplementedError(
                "Training of VITS is not supported by this model. See `VitsModelForPreTraining`."
            )

        text_encoder_output = self.text_encoder(
            input_ids=input_ids,
            padding_mask=input_padding_mask,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        hidden_states = text_encoder_output[0] if not return_dict else text_encoder_output.last_hidden_state
        hidden_states = hidden_states.transpose(1, 2)
        input_padding_mask = input_padding_mask.transpose(1, 2)
        prior_means = text_encoder_output[1] if not return_dict else text_encoder_output.prior_means
        prior_log_variances = text_encoder_output[2] if not return_dict else text_encoder_output.prior_log_variances

        if self.config.use_stochastic_duration_prediction:
            log_duration = self.duration_predictor(
                hidden_states,
                input_padding_mask,
                speaker_embeddings,
                reverse=True,
                noise_scale=self.noise_scale_duration,
            )
        else:
            log_duration = self.duration_predictor(hidden_states, input_padding_mask, speaker_embeddings)

        length_scale = 1.0 / self.speaking_rate
        duration = torch.ceil(torch.exp(log_duration) * input_padding_mask * length_scale)
        predicted_lengths = torch.clamp_min(torch.sum(duration, [1, 2]), 1).long()

        # Create a padding mask for the output lengths of shape (batch, 1, max_output_length)
        indices = torch.arange(predicted_lengths.max(), dtype=predicted_lengths.dtype, device=predicted_lengths.device)
        output_padding_mask = indices.unsqueeze(0) < predicted_lengths.unsqueeze(1)
        output_padding_mask = output_padding_mask.unsqueeze(1).to(input_padding_mask.dtype)

        # Reconstruct an attention tensor of shape (batch, 1, out_length, in_length)
        attn_mask = torch.unsqueeze(input_padding_mask, 2) * torch.unsqueeze(output_padding_mask, -1)
        batch_size, _, output_length, input_length = attn_mask.shape
        cum_duration = torch.cumsum(duration, -1).view(batch_size * input_length, 1)
        indices = torch.arange(output_length, dtype=duration.dtype, device=duration.device)
        valid_indices = indices.unsqueeze(0) < cum_duration
        valid_indices = valid_indices.to(attn_mask.dtype).view(batch_size, input_length, output_length)
        padded_indices = valid_indices - nn.functional.pad(valid_indices, [0, 0, 1, 0, 0, 0])[:, :-1]
        attn = padded_indices.unsqueeze(1).transpose(2, 3) * attn_mask

        # Expand prior distribution
        prior_means = torch.matmul(attn.squeeze(1), prior_means).transpose(1, 2)
        prior_log_variances = torch.matmul(attn.squeeze(1), prior_log_variances).transpose(1, 2)

        prior_latents = prior_means + torch.randn_like(prior_means) * torch.exp(prior_log_variances) * self.noise_scale
        latents = self.flow(prior_latents, output_padding_mask, speaker_embeddings, reverse=True)

        spectrogram = latents * output_padding_mask
        waveform = self.decoder(spectrogram, speaker_embeddings)
        waveform = waveform.squeeze(1)
        sequence_lengths = predicted_lengths * np.prod(self.config.upsample_rates)

        if not return_dict:
            outputs = (waveform, sequence_lengths, spectrogram) + text_encoder_output[3:]
            return outputs

        return VitsModelOutput(
            waveform=waveform,
            sequence_lengths=sequence_lengths,
            spectrogram=spectrogram,
            hidden_states=text_encoder_output.hidden_states,
            attentions=text_encoder_output.attentions,
        )


class VitsDiscriminator(VitsPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)

        if config.discriminator_scale_channels is not None:
            self.discriminators = nn.ModuleList(
                [VitsHifiGanDiscriminatorScaleResidualBlock(config.discriminator_scale_channels, config.leaky_relu_slope)]
            )
        else:
            self.discriminators = nn.ModuleList([])
        
        self.discriminators.extend(
            [
                VitsHifiGanDiscriminatorPeriodResidualBlock(
                    config.discriminator_period_channels,
                    period,
                    config.discriminator_kernel_size,
                    config.discriminator_stride,
                    config.leaky_relu_slope,
                )
                for period in config.discriminator_periods
            ]
        )

    def forward(self, hidden_states):
        fmaps = []
        discriminated_hidden_states_list = []

        for discriminator in self.discriminators:
            discriminated_hidden_states, fmap = discriminator(hidden_states)
            fmaps.append(fmap)
            discriminated_hidden_states_list.append(discriminated_hidden_states)

        return discriminated_hidden_states_list, fmaps

    def apply_weight_norm(self):
        for disc in self.discriminators:
            disc.apply_weight_norm()

    def remove_weight_norm(self):
        for disc in self.discriminators:
            disc.remove_weight_norm()


@add_start_docstrings(
    "VITS model with a discriminator, to train VITS.",
    VITS_START_DOCSTRING,
)
class VitsModelForPreTraining(VitsPreTrainedModel):
    def __init__(self, config: VitsConfig):
        super().__init__(config)
        self.config = config
        self.text_encoder = VitsTextEncoder(config)
        self.flow = VitsResidualCouplingBlock(config)
        self.decoder = VitsHifiGan(config)

        if config.use_stochastic_duration_prediction:
            self.duration_predictor = VitsStochasticDurationPredictor(config)
        else:
            self.duration_predictor = VitsDurationPredictor(config)

        if config.num_speakers > 1:
            self.embed_speaker = nn.Embedding(config.num_speakers, config.speaker_embedding_size)

        # This is used only for training.
        self.posterior_encoder = VitsPosteriorEncoder(config)
        self.discriminator = VitsDiscriminator(config)

        # These parameters control the synthesised speech properties
        self.speaking_rate = config.speaking_rate
        self.noise_scale = config.noise_scale
        self.noise_scale_duration = config.noise_scale_duration
        self.segment_size = self.config.segment_size // self.config.hop_length

        # Initialize weights and apply final processing
        self.post_init()

    def resize_speaker_embeddings(
        self,
        new_num_speakers: int,
        speaker_embedding_size: Optional[int] = None,
        pad_to_multiple_of: Optional[int] = 2,
    ):
        if pad_to_multiple_of is not None:
            new_num_speakers = ((new_num_speakers + pad_to_multiple_of - 1) // pad_to_multiple_of) * pad_to_multiple_of

        # first, take care of embed_speaker
        if self.config.num_speakers <= 1:
            if speaker_embedding_size is None:
                raise ValueError(
                    "The current model had no previous speaker embedding, but `speaker_embedding_size` is not specified. Pass `speaker_embedding_size` to this method."
                )
            # create new embedding layer
            new_embeddings = nn.Embedding(
                new_num_speakers,
                speaker_embedding_size,
                device=self.device,
            )
            # initialize all new embeddings
            self._init_weights(new_embeddings)
        else:
            new_embeddings = self._get_resized_embeddings(self.embed_speaker, new_num_speakers)

        self.embed_speaker = new_embeddings

        # then take care of sub-models
        self.flow.resize_speaker_embeddings(speaker_embedding_size)
        for flow in self.flow.flows:
            self._init_weights(flow.wavenet.cond_layer)

        self.decoder.resize_speaker_embedding(speaker_embedding_size)
        self._init_weights(self.decoder.cond)

        self.duration_predictor.resize_speaker_embeddings(speaker_embedding_size)
        self._init_weights(self.duration_predictor.cond)

        self.posterior_encoder.resize_speaker_embeddings(speaker_embedding_size)
        self._init_weights(self.posterior_encoder.wavenet.cond_layer)

        self.config.num_speakers = new_num_speakers
        self.config.speaker_embedding_size = speaker_embedding_size

    def get_input_embeddings(self):
        return self.text_encoder.get_input_embeddings()

    def set_input_embeddings(self, value):
        self.text_encoder.set_input_embeddings(value)

    def apply_weight_norm(self):
        self.decoder.apply_weight_norm()
        self.flow.apply_weight_norm()
        self.posterior_encoder.apply_weight_norm()

    def remove_weight_norm(self):
        self.decoder.remove_weight_norm()
        self.flow.remove_weight_norm()
        self.posterior_encoder.remove_weight_norm()

    def discriminate(self, hidden_states):
        return self.discriminator(hidden_states)

    def get_encoder(self):
        return self.text_encoder

    def _inference_forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        speaker_embeddings: Optional[torch.Tensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        padding_mask: Optional[torch.Tensor] = None,
    ):
        text_encoder_output = self.text_encoder(
            input_ids=input_ids,
            padding_mask=padding_mask,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        hidden_states = text_encoder_output[0] if not return_dict else text_encoder_output.last_hidden_state
        hidden_states = hidden_states.transpose(1, 2)
        input_padding_mask = padding_mask.transpose(1, 2)
        prior_means = text_encoder_output[1] if not return_dict else text_encoder_output.prior_means
        prior_log_variances = text_encoder_output[2] if not return_dict else text_encoder_output.prior_log_variances

        if self.config.use_stochastic_duration_prediction:
            log_duration = self.duration_predictor(
                hidden_states,
                input_padding_mask,
                speaker_embeddings,
                reverse=True,
                noise_scale=self.noise_scale_duration,
            )
        else:
            log_duration = self.duration_predictor(hidden_states, input_padding_mask, speaker_embeddings)

        length_scale = 1.0 / self.speaking_rate
        duration = torch.ceil(torch.exp(log_duration) * input_padding_mask * length_scale)
        predicted_lengths = torch.clamp_min(torch.sum(duration, [1, 2]), 1).long()

        # Create a padding mask for the output lengths of shape (batch, 1, max_output_length)
        indices = torch.arange(predicted_lengths.max(), dtype=predicted_lengths.dtype, device=predicted_lengths.device)
        output_padding_mask = indices.unsqueeze(0) < predicted_lengths.unsqueeze(1)
        output_padding_mask = output_padding_mask.unsqueeze(1).to(input_padding_mask.dtype)

        # Reconstruct an attention tensor of shape (batch, 1, out_length, in_length)
        attn_mask = torch.unsqueeze(input_padding_mask, 2) * torch.unsqueeze(output_padding_mask, -1)
        batch_size, _, output_length, input_length = attn_mask.shape
        cum_duration = torch.cumsum(duration, -1).view(batch_size * input_length, 1)
        indices = torch.arange(output_length, dtype=duration.dtype, device=duration.device)
        valid_indices = indices.unsqueeze(0) < cum_duration
        valid_indices = valid_indices.to(attn_mask.dtype).view(batch_size, input_length, output_length)
        padded_indices = valid_indices - nn.functional.pad(valid_indices, [0, 0, 1, 0, 0, 0])[:, :-1]
        attn = padded_indices.unsqueeze(1).transpose(2, 3) * attn_mask

        # Expand prior distribution
        prior_means = torch.matmul(attn.squeeze(1), prior_means).transpose(1, 2)
        prior_log_variances = torch.matmul(attn.squeeze(1), prior_log_variances).transpose(1, 2)

        prior_latents = prior_means + torch.randn_like(prior_means) * torch.exp(prior_log_variances) * self.noise_scale
        latents = self.flow(prior_latents, output_padding_mask, speaker_embeddings, reverse=True)

        spectrogram = latents * output_padding_mask
        waveform = self.decoder(spectrogram, speaker_embeddings)
        waveform = waveform.squeeze(1)
        sequence_lengths = predicted_lengths * np.prod(self.config.upsample_rates)

        if not return_dict:
            outputs = (waveform, sequence_lengths, spectrogram) + text_encoder_output[3:]
            return outputs

        return VitsModelOutput(
            waveform=waveform,
            sequence_lengths=sequence_lengths,
            spectrogram=spectrogram,
            hidden_states=text_encoder_output.hidden_states,
            attentions=text_encoder_output.attentions,
        )

    @add_start_docstrings_to_model_forward(VITS_INPUTS_DOCSTRING)
    @replace_return_docstrings(output_type=VitsModelOutput, config_class=_CONFIG_FOR_DOC)
    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        speaker_id: Optional[int] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        labels: Optional[torch.FloatTensor] = None,
        labels_attention_mask: Optional[torch.Tensor] = None,
        monotonic_alignment_function: Optional[Callable] = None,
    ) -> Union[Tuple[Any], VitsModelOutput]:
        r"""
        labels (`torch.FloatTensor` of shape `(batch_size, config.spectrogram_bins, sequence_length)`, *optional*):
            Float values of target spectrogram.
        labels_attention_mask (`torch.FloatTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Mask to avoid performing convolution and attention on labels. Mask values selected in `[0, 1]`:

            - 1 for tokens that are **not masked**,
            - 0 for tokens that are **masked**.

            [What are attention masks?](../glossary#attention-mask)
        monotonic_alignment_function (`Callable`, *optional*):
            Monotonic alignment function. Used for training, i.e when `labels` are provided. By default, it will use a
            Pytorch implementation of the monotonic alignment function which is awfully slow. An alternative relying on
            cython is proposed in examples/pytorch/text-to-speech/run_vits_finetuning.py
        Returns:

        Example:

        ```python
        >>> from transformers import VitsTokenizer, VitsModelForPreTraining, set_seed
        >>> import torch

        >>> tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-eng")
        >>> model = VitsModelForPreTraining.from_pretrained("facebook/mms-tts-eng")

        >>> inputs = tokenizer(text="Hello - my dog is cute", return_tensors="pt")

        >>> set_seed(555)  # make deterministic

        >>> with torch.no_grad():
        ...     outputs = model(inputs["input_ids"])
        >>> outputs.waveform.shape
        torch.Size([1, 45824])
        ```
        """
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        monotonic_alignment_function = (
            monotonic_align_max_path if monotonic_alignment_function is None else monotonic_alignment_function
        )

        if attention_mask is not None:
            input_padding_mask = attention_mask.unsqueeze(-1).float()
        else:
            input_padding_mask = torch.ones_like(input_ids).unsqueeze(-1).float()

        if self.config.num_speakers > 1 and speaker_id is not None:
            if isinstance(speaker_id, int):
                speaker_id = torch.full(size=(1,), fill_value=speaker_id, device=self.device)
            elif isinstance(speaker_id, (list, tuple, np.ndarray)):
                speaker_id = torch.tensor(speaker_id, device=self.device)

            if not ((0 <= speaker_id).all() and (speaker_id < self.config.num_speakers).all()).item():
                raise ValueError(f"Set `speaker_id` in the range 0-{self.config.num_speakers - 1}.")
            if not (len(speaker_id) == 1 or len(speaker_id == len(input_ids))):
                raise ValueError(
                    f"You passed {len(speaker_id)} `speaker_id` but you should either pass one speaker id or `batch_size` `speaker_id`."
                )

            speaker_embeddings = self.embed_speaker(speaker_id).unsqueeze(-1)
        else:
            speaker_embeddings = None

        # if inference, return inference forward of VitsModel
        if labels is None:
            return self._inference_forward(
                input_ids,
                attention_mask,
                speaker_embeddings,
                output_attentions,
                output_hidden_states,
                return_dict,
                input_padding_mask,
            )

        if labels_attention_mask is not None:
            labels_padding_mask = labels_attention_mask.unsqueeze(1).float()
        else:
            labels_attention_mask = torch.ones((labels.shape[0], labels.shape[2])).float().to(self.device)
            labels_padding_mask = labels_attention_mask.unsqueeze(1)

        text_encoder_output = self.text_encoder(
            input_ids=input_ids,
            padding_mask=input_padding_mask,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        hidden_states = text_encoder_output[0] if not return_dict else text_encoder_output.last_hidden_state
        hidden_states = hidden_states.transpose(1, 2)
        input_padding_mask = input_padding_mask.transpose(1, 2)
        prior_means = text_encoder_output[1] if not return_dict else text_encoder_output.prior_means
        prior_log_variances = text_encoder_output[2] if not return_dict else text_encoder_output.prior_log_variances

        latents, posterior_means, posterior_log_variances = self.posterior_encoder(
            labels, labels_padding_mask, speaker_embeddings
        )
        prior_latents = self.flow(latents, labels_padding_mask, speaker_embeddings, reverse=False)

        prior_means, prior_log_variances = prior_means.transpose(1, 2), prior_log_variances.transpose(1, 2)
        with torch.no_grad():
            # negative cross-entropy

            # [batch_size, d, latent_length]
            prior_variances = torch.exp(-2 * prior_log_variances)
            # [batch_size, 1, latent_length]
            neg_cent1 = torch.sum(-0.5 * math.log(2 * math.pi) - prior_log_variances, [1], keepdim=True)
            # [batch_size, text_length, d] x [batch_size, d, latent_length] = [batch_size, text_length, latent_length]
            neg_cent2 = torch.matmul(-0.5 * (prior_latents**2).transpose(1, 2), prior_variances)
            # [batch_size, text_length, d] x [batch_size, d, latent_length] = [batch_size, text_length, latent_length]
            neg_cent3 = torch.matmul(prior_latents.transpose(1, 2), (prior_means * prior_variances))
            # [batch_size, 1, latent_length]
            neg_cent4 = torch.sum(-0.5 * (prior_means**2) * prior_variances, [1], keepdim=True)

            # [batch_size, text_length, latent_length]
            neg_cent = neg_cent1 + neg_cent2 + neg_cent3 + neg_cent4

            attn_mask = torch.unsqueeze(input_padding_mask, 2) * torch.unsqueeze(labels_padding_mask, -1)

            attn = monotonic_alignment_function(neg_cent, attn_mask.squeeze(1)).unsqueeze(1).detach()

        durations = attn.sum(2)

        if self.config.use_stochastic_duration_prediction:
            log_duration = self.duration_predictor(
                hidden_states, input_padding_mask, speaker_embeddings, durations=durations, reverse=False
            )
            log_duration = log_duration / torch.sum(input_padding_mask)
        else:
            log_duration_padded = torch.log(durations + 1e-6) * input_padding_mask
            log_duration = self.duration_predictor(hidden_states, input_padding_mask, speaker_embeddings)
            log_duration = torch.sum((log_duration - log_duration_padded) ** 2, [1, 2]) / torch.sum(input_padding_mask)

        # expand priors
        prior_means = torch.matmul(attn.squeeze(1), prior_means.transpose(1, 2)).transpose(1, 2)
        prior_log_variances = torch.matmul(attn.squeeze(1), prior_log_variances.transpose(1, 2)).transpose(1, 2)

        label_lengths = labels_attention_mask.sum(dim=1)
        latents_slice, ids_slice = rand_slice_segments(latents, label_lengths, segment_size=self.segment_size)

        waveform = self.decoder(latents_slice, speaker_embeddings)

        if not return_dict:
            outputs = (
                waveform,
                log_duration,
                attn,
                ids_slice,
                input_padding_mask,
                labels_padding_mask,
                latents,
                prior_latents,
                prior_means,
                prior_log_variances,
                posterior_means,
                posterior_log_variances,
            )
            return outputs

        return VitsTrainingOutput(
            waveform=waveform,
            log_duration=log_duration,
            attn=attn,
            ids_slice=ids_slice,
            input_padding_mask=input_padding_mask,
            labels_padding_mask=labels_padding_mask,
            latents=latents,
            prior_latents=prior_latents,
            prior_means=prior_means,
            prior_log_variances=prior_log_variances,
            posterior_means=posterior_means,
            posterior_log_variances=posterior_log_variances,
        )

In [ ]:
%%writefile /tmp/btp/third_party/finetune_hf_vits/plot.py
import logging
import matplotlib


matplotlib.use("Agg")

MATPLOTLIB_FLAG = False


def plot_spectrogram_to_numpy(spectrogram):
    global MATPLOTLIB_FLAG
    if not MATPLOTLIB_FLAG:
        import matplotlib

        matplotlib.use("Agg")
        MATPLOTLIB_FLAG = True
        mpl_logger = logging.getLogger("matplotlib")
        mpl_logger.setLevel(logging.WARNING)
    import matplotlib.pylab as plt
    import numpy as np

    fig, ax = plt.subplots(figsize=(10, 2))
    im = ax.imshow(spectrogram, aspect="auto", origin="lower", interpolation="none")
    plt.colorbar(im, ax=ax)
    plt.xlabel("Frames")
    plt.ylabel("Channels")
    plt.tight_layout()

    fig.canvas.draw()
    data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close()
    return data


def plot_alignment_to_numpy(alignment, info=None):
    global MATPLOTLIB_FLAG
    if not MATPLOTLIB_FLAG:
        import matplotlib

        matplotlib.use("Agg")
        MATPLOTLIB_FLAG = True
        mpl_logger = logging.getLogger("matplotlib")
        mpl_logger.setLevel(logging.WARNING)
    import matplotlib.pylab as plt
    import numpy as np

    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(alignment.transpose(), aspect="auto", origin="lower", interpolation="none")
    fig.colorbar(im, ax=ax)
    xlabel = "Decoder timestep"
    if info is not None:
        xlabel += "\n\n" + info
    plt.xlabel(xlabel)
    plt.ylabel("Encoder timestep")
    plt.tight_layout()

    fig.canvas.draw()
    data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close()
    return data

In [ ]:
%%writefile /tmp/btp/third_party/finetune_hf_vits/LICENSE
MIT License

Copyright (c) 2023 Yoach Lacombe

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
%%writefile /tmp/btp/third_party/finetune_hf_vits/README.md
Vendored from https://github.com/ylacombe/finetune-hf-vits at commit 6f3f51f4d667f5c3eef89484d151ffd39d2c2b89 (MIT License, see LICENSE).
Only the model/feature-extraction modules and the discriminator converter are used;
the training loop lives in btp/tts.py and the Cython monotonic alignment is replaced by btp/mas.py.

In [ ]:
%%writefile /tmp/btp/requirements-colab.txt
# Installed on top of Colab's preinstalled torch/torchaudio/transformers/datasets/librosa/soundfile.
uroman
sacrebleu
jiwer
wandb
safetensors
python-docx
pypdf

In [ ]:
!touch /tmp/btp/btp/__init__.py && cd /tmp/btp && python -c "import btp.cli; print('btp imports OK')"

## Run the job

In [ ]:
FAILED = []
STAGE_SECONDS = {}
def btp(args, env=None):
    """Run one pipeline stage, streaming its output; failures are recorded, not fatal."""
    print(f"\n===== btp {args}  ({(time.time() - START) / 3600:.2f} h elapsed)", flush=True)
    t0 = time.time()
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen(f"cd /tmp/btp && python -m btp.cli {args}", shell=True, env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    name = args.split()[0]
    STAGE_SECONDS[name] = round(STAGE_SECONDS.get(name, 0) + time.time() - t0, 1)
    if rc:
        FAILED.append((name, rc))
    return rc

def time_left_h():
    return (float(os.environ['BTP_DEADLINE']) - time.time()) / 3600

In [ ]:
if JOB == 'smoke' and not SKIP:
    SMOKE = {'BTP_WORKSPACE': '/tmp/btp_smoke'}           # not saved: smoke outputs are throwaway
    sets = {'saq': 'SAQBTLN1DA', 'luo': 'LUOBIBN1DA', 'swh': 'SWABIBN1DA', 'kik': 'KIKKGVN1DA', 'dig': 'DIGTBLN1DA'}
    btp(f"check --langs {SMOKE_LANG}", SMOKE)
    btp(f"corpus --langs {SMOKE_LANG}", SMOKE)
    btp(f"import-openbible --langs {SMOKE_LANG} --max-chapters 1", SMOKE)   # 1 file (~0.8 GB) if available
    btp(f"align --langs {SMOKE_LANG} --sets {sets[SMOKE_LANG]} --books MRK COL PHP --max-chapters-per-book 2", SMOKE)
    btp(f"asr --langs {SMOKE_LANG} --max-steps 30 --set asr.eval_steps=15 --set asr.save_steps=15 --set asr.eval_samples=20", SMOKE)
    btp(f"mt --langs {SMOKE_LANG} --max-steps 30 --set mt.eval_steps=15 --set mt.save_steps=30 --set mt.eval_samples_per_lang=20 --set mt.warmup_steps=5", SMOKE)
    btp(f"tts --langs {SMOKE_LANG} --max-steps 30 --set tts.eval_steps=15 --set tts.save_steps=30 --set tts.per_device_batch=8", SMOKE)
    btp(f"evaluate --langs {SMOKE_LANG}", SMOKE)
    os.makedirs('/tmp/btp_smoke/books', exist_ok=True)
    PARAGRAPHS = ['The rain came early this year. Farmers planted maize and beans before the first storm.',
                  "In the evening, the children gathered around the fire and listened to their grandmother's stories."]
    open('/tmp/btp_smoke/books/sample.txt', 'w').write((chr(10) * 2).join(PARAGRAPHS))
    btp(f"translate-book --langs {SMOKE_LANG} --book /tmp/btp_smoke/books/sample.txt", SMOKE)
    btp(f"synthesize-book --langs {SMOKE_LANG} --review /tmp/btp_smoke/books/{SMOKE_LANG}/sample.review.tsv", SMOKE)
    for sub in ('reports', 'logs'):                       # keep the evidence, not the models
        shutil.copytree(f'/tmp/btp_smoke/{sub}', f'/kaggle/working/smoke/{sub}', dirs_exist_ok=True)
    for f in glob.glob(f'/tmp/btp_smoke/books/{SMOKE_LANG}/sample/*.wav'):
        shutil.copy(f, '/kaggle/working/smoke/')

In [ ]:
TRAINED_BEFORE = {os.path.basename(f)[len('train_'):-5] for f in glob.glob(WS + '/status/train_*.json')
                  if json.load(open(f)).get('finished')}
if SKIP:
    print('SKIPPED:', SKIP)
elif JOB == 'full':
    # storage pruning keeps only final/ of finished models (20 GB output limit)
    cap = (f"--set align.max_hours_per_lang={ALIGN_CAP_HOURS} --set tts.prune_checkpoint_when_done=true "
           f"--set mt.prune_checkpoint_when_done=true")
    for stage in ('check', 'corpus', 'import-openbible', 'align', 'pseudo', 'mt', 'asr', 'tts', 'evaluate'):
        if time_left_h() < 0.5:
            print(f'deadline reached -- {stage} and later stages run in the next session'); break
        env = None
        if stage in STAGE_HOURS:      # this stage's own (earlier) deadline; it checkpoints and resumes next run
            env = {'BTP_DEADLINE': str(min(float(os.environ['BTP_DEADLINE']), time.time() + STAGE_HOURS[stage] * 3600))}
        btp(f"{stage} --langs {LANGS} {cap}", env)
    btp(f"status --langs {LANGS}")
# (stages whose training already finished return immediately: btp.utils.training_done)
elif JOB == 'b':
    for stage in ('corpus', 'pseudo --max-chapters 60', 'queue --n 300', 'import-transcripts'):
        btp(f"{stage} --langs {B_LANG}")
elif JOB == 'book' and BOOK:
    btp(f"translate-book --langs {BOOK_LANG} --book '{BOOK}'")
    stem = os.path.splitext(os.path.basename(BOOK))[0]
    btp(f"synthesize-book --langs {BOOK_LANG} --review {WS}/books/{BOOK_LANG}/{stem}.review.tsv")

In [ ]:
# Summary + keep the saved output under Kaggle's 20 GB limit
shutil.rmtree('/tmp/btp_cache', ignore_errors=True)
for f in sorted(glob.glob('/kaggle/working/**/reports/stage_*.json', recursive=True)):
    d = json.load(open(f))
    bad = 'error' in d or any(isinstance(v, dict) and 'error' in v for v in (d.get('result') or {}).values())
    print(('ERROR ' if bad else 'ok    ') + f)
def tree_gb(top):
    return sum(os.path.getsize(os.path.join(dp, fn)) for dp, _, fns in os.walk(top) for fn in fns
               if not os.path.islink(os.path.join(dp, fn))) / 1e9

def drop(path):
    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
    elif os.path.exists(path):
        os.remove(path)

# W&B files are already uploaded when W&B was online -- no need to keep them in the output
if os.environ.get('WANDB_API_KEY'):
    drop(WS + '/wandb')
used = tree_gb('/kaggle/working')
TRIMMED = []
if used > OUTPUT_LIMIT_GB:
    # Over the limit Kaggle saves nothing at all (all of this run's progress lost). Trim, least valuable first.
    for pattern in (WS + '/wandb', WS + '/**/*.prev', WS + '/**/*.tmp', WS + '/models/tts/*/checkpoint.pt'):
        for p in glob.glob(pattern, recursive=True):
            if used <= OUTPUT_LIMIT_GB:
                break
            gb = tree_gb(p) if os.path.isdir(p) else os.path.getsize(p) / 1e9
            drop(p)
            used -= gb
            TRIMMED.append(os.path.relpath(p, WS))
    print('OUTPUT OVER THE LIMIT -- trimmed:', TRIMMED)
BREAKDOWN = {}
for top in sorted(glob.glob(WS + '/*/*')):
    if os.path.isdir(top):
        g = tree_gb(top)
        if g >= 0.05:
            BREAKDOWN[os.path.relpath(top, WS)] = round(g, 2)
print('output by folder (GB):', json.dumps(dict(sorted(BREAKDOWN.items(), key=lambda kv: -kv[1]))))
print(f'saved output: {used:.1f} GB of 20 GB | failed stages: {FAILED or "none"} | elapsed {(time.time() - START) / 3600:.2f} h')
# Progress snapshot: lets the scheduler verify that a resumed run continued (and when training is complete)
sys.path.insert(0, '/tmp/btp')
progress = {'status_done': {}, 'checkpoint_steps': {}, 'trained': []}
for f in glob.glob(WS + '/status/*.json'):
    name = os.path.basename(f)[:-5]
    d = json.load(open(f))
    if not isinstance(d, dict):            # e.g. kaggle_ledger.json is a list, not a status record
        continue
    if name.startswith('train_'):
        if d.get('finished'):
            progress['trained'].append(name[len('train_'):])
    elif isinstance(d.get('done'), list):
        progress['status_done'][name] = len(d['done'])
for f in glob.glob(WS + '/models/*/*/checkpoint.pt') + glob.glob(WS + '/models/mt/checkpoint/trainer_state.pt'):
    try:
        progress['checkpoint_steps'][os.path.relpath(f, WS + '/models')] = int(torch.load(f, map_location='cpu', weights_only=False)['step'])
    except Exception as e:
        progress['checkpoint_steps'][os.path.relpath(f, WS + '/models')] = f'unreadable: {e}'
all_done = False
if JOB == 'full':
    from btp.languages import resolve_langs
    codes = resolve_langs(LANGS)
    t = set(progress['trained'])
    all_done = any(x.startswith('mt_') for x in t) and all(f'asr_{c}' in t and f'tts_{c}' in t for c in codes)
if JOB == 'full' and not SKIP:
    ledger.append({'start': START, 'hours': round((time.time() - START) / 3600 + 0.1, 3)})
    json.dump(ledger, open(LEDGER, 'w'))
json.dump({'finished': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'skip': SKIP,
           'gpu_hours_7d_before': round(used_7d, 2),
           'job': JOB, 'langs': LANGS, 'deadline_hours': DEADLINE_HOURS, 'restored_from': RESTORED,
           'restored_version': BEST_VERSION, 'gpu_quota': QUOTA,
           'failed': FAILED, 'stage_seconds': STAGE_SECONDS, 'progress': progress, 'all_done': all_done,
           'output_gb': round(used, 2), 'output_breakdown_gb': BREAKDOWN, 'trimmed': TRIMMED,
           'hours': round((time.time() - START) / 3600, 2)},
          open('/kaggle/working/run_summary.json', 'w'), indent=1)
print('progress:', json.dumps(progress), '| all training done:', all_done)